In [ ]:
!pip install pandas pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 33.2 MB/s eta 0:00:00


##Muster check

In [ ]:
# ============================================================
# BDHS 2022 HYPERTENSION DATASET
# PR FILE ONLY
# FINAL PUBLICATION-QUALITY COMPLETE-CASE PIPELINE
# FIXED VERSION WITH CONSISTENT INCLUSION/EXCLUSION CRITERIA
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_spss(
    "/content/BDPR81FL.SAV",
    convert_categoricals=False
)

print("=" * 70)
print("BDHS 2022 HYPERTENSION ANALYTIC PIPELINE")
print("=" * 70)

print("\nInitial dataset shape:", df.shape)

# ============================================================
# 2. ELIGIBILITY CRITERIA
# ============================================================

print("\n" + "=" * 70)
print("APPLYING ELIGIBILITY CRITERIA")
print("=" * 70)

# ------------------------------------------------------------
# DE FACTO RESIDENTS
# ------------------------------------------------------------

before = len(df)

df = df[df["HV103"] == 1].copy()

print("\n1. De facto residents only")
print("Removed:", before - len(df))
print("Remaining:", len(df))

# ------------------------------------------------------------
# ADULTS >=18 YEARS
# ------------------------------------------------------------

before = len(df)

df = df[df["HV105"] >= 18].copy()

print("\n2. Adults aged >=18 years")
print("Removed:", before - len(df))
print("Remaining:", len(df))

# ------------------------------------------------------------
# EXCLUDE PREGNANT WOMEN
# HA54 = currently pregnant
# 1 = yes
# ------------------------------------------------------------

if "HA54" in df.columns:

    before = len(df)

    df = df[
        ~(
            (df["HV104"] == 2) &
            (df["HA54"] == 1)
        )
    ].copy()

    print("\n3. Pregnant women excluded")
    print("Removed:", before - len(df))
    print("Remaining:", len(df))

else:

    print("\n3. Pregnancy variable not available")

# ============================================================
# 3. BLOOD PRESSURE PROCESSING FUNCTION
# ============================================================

def get_bp_average(row, s_cols, d_cols):

    systolic = []
    diastolic = []

    # --------------------------------------------------------
    # CLEAN SBP
    # --------------------------------------------------------

    for c in s_cols:

        val = row.get(c, np.nan)

        if pd.notna(val) and val < 900:
            systolic.append(val)
        else:
            systolic.append(np.nan)

    # --------------------------------------------------------
    # CLEAN DBP
    # --------------------------------------------------------

    for c in d_cols:

        val = row.get(c, np.nan)

        if pd.notna(val) and val < 900:
            diastolic.append(val)
        else:
            diastolic.append(np.nan)

    # --------------------------------------------------------
    # DHS RECOMMENDED AVERAGING
    # --------------------------------------------------------

    if (
        not np.isnan(systolic[1]) and
        not np.isnan(systolic[2])
    ):

        sbp = (systolic[1] + systolic[2]) / 2
        dbp = (diastolic[1] + diastolic[2]) / 2

        reading_type = "2nd_and_3rd"

    elif not np.isnan(systolic[1]):

        sbp = systolic[1]
        dbp = diastolic[1]

        reading_type = "2nd_only"

    elif not np.isnan(systolic[0]):

        sbp = systolic[0]
        dbp = diastolic[0]

        reading_type = "1st_only"

    else:

        sbp = np.nan
        dbp = np.nan

        reading_type = "missing"

    return pd.Series([sbp, dbp, reading_type])

# ============================================================
# 4. BLOOD PRESSURE VARIABLES
# ============================================================

print("\n" + "=" * 70)
print("BLOOD PRESSURE PROCESSING")
print("=" * 70)

df["sbp_final"] = np.nan
df["dbp_final"] = np.nan
df["bp_reading_type"] = np.nan
df["is_on_meds"] = 0

# ------------------------------------------------------------
# MEN
# ------------------------------------------------------------

men = df["HV104"] == 1

bp_men = df[men].apply(
    lambda r: get_bp_average(
        r,
        ["MBP9", "MBP13", "MBP22"],
        ["MBP10", "MBP14", "MBP23"]
    ),
    axis=1
)

df.loc[
    men,
    ["sbp_final", "dbp_final", "bp_reading_type"]
] = bp_men.values

# antihypertensive medication

df.loc[
    men & (df["MBP19"] == 1),
    "is_on_meds"
] = 1

# ------------------------------------------------------------
# WOMEN
# ------------------------------------------------------------

women = df["HV104"] == 2

bp_women = df[women].apply(
    lambda r: get_bp_average(
        r,
        ["WBP9", "WBP13", "WBP22"],
        ["WBP10", "WBP14", "WBP23"]
    ),
    axis=1
)

df.loc[
    women,
    ["sbp_final", "dbp_final", "bp_reading_type"]
] = bp_women.values

# antihypertensive medication

df.loc[
    women & (df["WBP19"] == 1),
    "is_on_meds"
] = 1

# ------------------------------------------------------------
# PRINT BP SUMMARY
# ------------------------------------------------------------

print("\nBlood pressure reading type distribution:")
print(df["bp_reading_type"].value_counts(dropna=False))

# ============================================================
# 5. PHYSIOLOGICAL OUTLIER FILTERING
# ============================================================

print("\n" + "=" * 70)
print("OUTLIER FILTERING")
print("=" * 70)

# ------------------------------------------------------------
# SBP OUTLIERS
# ------------------------------------------------------------

sbp_outliers = (
    (df["sbp_final"] < 70) |
    (df["sbp_final"] > 250)
)

print("\nSBP outliers detected:",
      sbp_outliers.sum())

df.loc[sbp_outliers, "sbp_final"] = np.nan

# ------------------------------------------------------------
# DBP OUTLIERS
# ------------------------------------------------------------

dbp_outliers = (
    (df["dbp_final"] < 40) |
    (df["dbp_final"] > 150)
)

print("DBP outliers detected:",
      dbp_outliers.sum())

df.loc[dbp_outliers, "dbp_final"] = np.nan

# ============================================================
# 6. KEEP PARTICIPANTS WITH VALID BP
# ============================================================

before = len(df)

df = df.dropna(
    subset=["sbp_final", "dbp_final"]
).copy()

print("\nParticipants removed due to invalid BP:",
      before - len(df))

print("Participants with valid BP:",
      len(df))

# ============================================================
# 7. HYPERTENSION OUTCOME
# ============================================================

print("\n" + "=" * 70)
print("HYPERTENSION CLASSIFICATION")
print("=" * 70)

df["hypertension"] = np.where(
    (
        (df["sbp_final"] >= 140) |
        (df["dbp_final"] >= 90) |
        (df["is_on_meds"] == 1)
    ),
    1,
    0
)

print("\nHypertension distribution:")
print(df["hypertension"].value_counts())

print("\nHypertension prevalence (%):")
print(
    round(
        df["hypertension"].mean() * 100,
        2
    )
)

# ============================================================
# 8. BMI PROCESSING
# ============================================================

print("\n" + "=" * 70)
print("BMI PROCESSING")
print("=" * 70)

df["bmi_raw"] = np.where(
    df["HV104"] == 1,
    df["HB40"],
    df["HA40"]
)

df["BMI"] = np.where(
    df["bmi_raw"] < 9000,
    df["bmi_raw"] / 100,
    np.nan
)

# ------------------------------------------------------------
# BMI OUTLIERS
# ------------------------------------------------------------

bmi_outliers = (
    (df["BMI"] < 12) |
    (df["BMI"] > 60)
)

print("\nBMI outliers detected:",
      bmi_outliers.sum())

df.loc[bmi_outliers, "BMI"] = np.nan

# ------------------------------------------------------------
# ASIAN BMI CATEGORIES
# ------------------------------------------------------------

df["BMI_category"] = pd.cut(
    df["BMI"],
    bins=[0, 18.5, 23, 27.5, 100],
    labels=[
        "Underweight",
        "Normal",
        "Overweight",
        "Obese"
    ]
)

print("\nBMI category distribution:")
print(df["BMI_category"].value_counts())

# ============================================================
# 9. GLUCOSE / DIABETES
# ============================================================

print("\n" + "=" * 70)
print("GLUCOSE / DIABETES PROCESSING")
print("=" * 70)

df["glucose"] = df["SB367"].fillna(
    df["SB267"]
)

# invalid glucose

df.loc[
    df["glucose"] >= 900,
    "glucose"
] = np.nan

print("\nMissing glucose values:",
      df["glucose"].isna().sum())

# diabetes

df["diabetes"] = np.where(
    df["glucose"] >= 126,
    1,
    0
)

# prediabetes

df["prediabetes"] = np.where(
    (
        (df["glucose"] >= 100) &
        (df["glucose"] < 126)
    ),
    1,
    0
)

print("\nDiabetes distribution:")
print(df["diabetes"].value_counts())

# ============================================================
# 10. AGE GROUPS
# ============================================================

df["age_group"] = pd.cut(
    df["HV105"],
    bins=[18, 29, 39, 49, 59, 69, 120],
    labels=[
        "18-29",
        "30-39",
        "40-49",
        "50-59",
        "60-69",
        "70+"
    ],
    include_lowest=True
)

# ============================================================
# 11. HOUSEHOLD VARIABLES
# ============================================================

print("\n" + "=" * 70)
print("HOUSEHOLD VARIABLES")
print("=" * 70)

# household crowding

df["crowding_index"] = (
    df["HV009"] / df["HV216"]
)

df["crowding_index"] = df[
    "crowding_index"
].replace(
    [np.inf, -np.inf],
    np.nan
)

# ============================================================
# 12. IMPROVED WATER
# ============================================================

improved_water_codes = [
    11, 12, 13, 14,
    21,
    31,
    41,
    51,
    61
]

df["improved_water"] = np.where(
    df["HV201"].isin(improved_water_codes),
    1,
    0
)

# ============================================================
# 13. IMPROVED TOILET
# ============================================================

improved_toilet_codes = [
    11, 12, 13, 14,
    21, 22, 23
]

df["improved_toilet"] = np.where(
    df["HV205"].isin(improved_toilet_codes),
    1,
    0
)

# ============================================================
# 14. ELECTRICITY
# ============================================================

df["electricity"] = df["HV206"]

# ============================================================
# 15. MOBILE PHONE
# ============================================================

if "HV243A" in df.columns:

    df["mobile_phone"] = df["HV243A"]

else:

    df["mobile_phone"] = np.nan

# ============================================================
# 16. EDUCATION CLEANING
# ============================================================

df["HV106"] = df["HV106"].replace(
    8,
    np.nan
)

# ============================================================
# 17. FINAL VARIABLE SELECTION
# ============================================================

print("\n" + "=" * 70)
print("FINAL VARIABLE SELECTION")
print("=" * 70)

final_variables = {

    # outcome
    "hypertension": "hypertension",

    # blood pressure
    "sbp_final": "sbp_final",
    "dbp_final": "dbp_final",

    # demographic
    "HV105": "age",
    "age_group": "age_group",
    "HV104": "sex",

    # socioeconomic
    "HV106": "education_level",
    "HV115": "marital_status",
    "HV025": "residence_type",
    "HV024": "division",
    "HV270": "wealth_index",

    # anthropometric
    "BMI": "BMI",
    "BMI_category": "BMI_category",

    # metabolic
    "glucose": "glucose",
    "diabetes": "diabetes",
    "prediabetes": "prediabetes",

    # household
    "HV009": "household_size",
    "HV216": "sleeping_rooms",
    "crowding_index": "crowding_index",

    # infrastructure
    "electricity": "electricity",
    "improved_water": "improved_water",
    "improved_toilet": "improved_toilet",
    "mobile_phone": "mobile_phone",

    # survey design
    "HV005": "sample_weight",
    "HV021": "cluster_id",
    "HV023": "strata_id"
}

df_final = df[
    list(final_variables.keys())
].rename(
    columns=final_variables
)

# ============================================================
# 18. SURVEY WEIGHT
# ============================================================

df_final["sample_weight"] = (
    df_final["sample_weight"] / 1000000
)

# ============================================================
# 19. REMOVE DUPLICATES
# ============================================================

before = len(df_final)

df_final = df_final.drop_duplicates()

print("\nDuplicate rows removed:",
      before - len(df_final))

# ============================================================
# 20. COMPLETE-CASE ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("COMPLETE-CASE CLEANING")
print("=" * 70)

print("\nMissing values BEFORE cleaning:")
print(df_final.isna().sum())

complete_case_vars = [

    "hypertension",

    "age",
    "age_group",
    "sex",

    "education_level",
    "marital_status",
    "residence_type",
    "division",
    "wealth_index",

    "BMI",
    "BMI_category",

    "glucose",
    "diabetes",

    "household_size",
    "crowding_index",

    "improved_water",
    "improved_toilet",

    "sample_weight",
    "cluster_id",
    "strata_id"
]

before = len(df_final)

df_final = df_final.dropna(
    subset=complete_case_vars
)

print("\nRows removed during complete-case analysis:",
      before - len(df_final))

print("Final analytic sample size:",
      len(df_final))

# ============================================================
# 21. FINAL MISSING CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL MISSING VALUE CHECK")
print("=" * 70)

print(df_final.isna().sum())

print("\nTotal missing values:",
      df_final.isna().sum().sum())

# ============================================================
# 22. FINAL DATASET SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET SUMMARY")
print("=" * 70)

print("\nFinal dataset shape:")
print(df_final.shape)

print("\nHypertension prevalence:")
print(
    round(
        df_final["hypertension"].mean() * 100,
        2
    ),
    "%"
)

print("\nSex distribution:")
print(df_final["sex"].value_counts())

print("\nAge group distribution:")
print(df_final["age_group"].value_counts())

print("\nEducation distribution:")
print(df_final["education_level"].value_counts())

print("\nResidence distribution:")
print(df_final["residence_type"].value_counts())

print("\nWealth index distribution:")
print(df_final["wealth_index"].value_counts())

print("\nDiabetes distribution:")
print(df_final["diabetes"].value_counts())

print("\nBMI category distribution:")
print(df_final["BMI_category"].value_counts())

# ============================================================
# 23. EXPORT FINAL DATASET
# ============================================================

output_file = (
    "BDHS_2022_HYPERTENSION_COMPLETE_CASE_FINAL.csv"
)

df_final.to_csv(
    output_file,
    index=False
)

print("\nDataset exported successfully!")
print("File name:", output_file)

# ============================================================
# 24. VARIABLE DESCRIPTION TABLE
# ============================================================

print("\n" + "=" * 70)
print("CREATING VARIABLE DESCRIPTION TABLE")
print("=" * 70)

description_map = {

    "hypertension":
    "Hypertension status (SBP>=140 or DBP>=90 or medication use)",

    "sbp_final":
    "Average systolic blood pressure (mmHg)",

    "dbp_final":
    "Average diastolic blood pressure (mmHg)",

    "age":
    "Age in years",

    "age_group":
    "Age category",

    "sex":
    "Sex of respondent",

    "education_level":
    "Educational attainment",

    "marital_status":
    "Current marital status",

    "residence_type":
    "Urban/rural residence",

    "division":
    "Administrative division",

    "wealth_index":
    "Household wealth quintile",

    "BMI":
    "Body Mass Index (kg/m²)",

    "BMI_category":
    "Asian BMI classification",

    "glucose":
    "Blood glucose level (mg/dL)",

    "diabetes":
    "Diabetes status",

    "prediabetes":
    "Prediabetes status",

    "household_size":
    "Total household members",

    "sleeping_rooms":
    "Number of sleeping rooms",

    "crowding_index":
    "Persons per sleeping room",

    "electricity":
    "Household electricity access",

    "improved_water":
    "Improved drinking water source",

    "improved_toilet":
    "Improved sanitation facility",

    "mobile_phone":
    "Household mobile phone ownership",

    "sample_weight":
    "Survey sampling weight",

    "cluster_id":
    "Primary sampling unit",

    "strata_id":
    "Sampling strata"
}

variable_description = pd.DataFrame({

    "Variable":
    df_final.columns,

    "Description":
    [
        description_map.get(col, "N/A")
        for col in df_final.columns
    ]
})

variable_description.to_csv(
    "BDHS_2022_VARIABLE_DESCRIPTIONS.csv",
    index=False
)

print("\nVariable description table exported!")

print("\n" + "=" * 70)
print("PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 70)

BDHS 2022 HYPERTENSION ANALYTIC PIPELINE

Initial dataset shape: (132463, 549)

APPLYING ELIGIBILITY CRITERIA

1. De facto residents only
Removed: 5866
Remaining: 126597

2. Adults aged >=18 years
Removed: 44716
Remaining: 81881

3. Pregnant women excluded
Removed: 548
Remaining: 81333

BLOOD PRESSURE PROCESSING


/tmp/ipykernel_3759/3711182450.py:183: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd', '2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd',


Blood pressure reading type distribution:
bp_reading_type
missing        67771
2nd_and_3rd    13055
1st_only         288
2nd_only         219
Name: count, dtype: int64

OUTLIER FILTERING

SBP outliers detected: 0
DBP outliers detected: 0

Participants removed due to invalid BP: 67771
Participants with valid BP: 13562

HYPERTENSION CLASSIFICATION

Hypertension distribution:
hypertension
0    10696
1     2866
Name: count, dtype: int64

Hypertension prevalence (%):
21.13

BMI PROCESSING

BMI outliers detected: 0

BMI category distribution:
BMI_category
Normal         5386
Overweight     4302
Underweight    2089
Obese          1669
Name: count, dtype: int64

GLUCOSE / DIABETES PROCESSING

Missing glucose values: 312

Diabetes distribution:
diabetes
0    12511
1     1051
Name: count, dtype: int64

HOUSEHOLD VARIABLES

FINAL VARIABLE SELECTION

Duplicate rows removed: 0

COMPLETE-CASE CLEANING

Missing values BEFORE cleaning:
hypertension         0
sbp_final            0
dbp_final          

In [ ]:
# ============================================================
# BDHS 2022 HYPERTENSION DATASET
# FINAL PUBLICATION-QUALITY COMPLETE-CASE & ML PIPELINE
# COMBINED MEN + WOMEN
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_spss(
    "/content/BDPR81FL.SAV",
    convert_categoricals=False
)

print("=" * 70)
print("INITIAL DATA")
print("=" * 70)
print("Initial shape:", df.shape)

# ============================================================
# 2. ELIGIBILITY CRITERIA
# ============================================================

# 1. De facto residents
df = df[df["HV103"] == 1].copy()
print("\nAfter de facto filter:", len(df))

# 2. Adults ≥18 years
df = df[df["HV105"] >= 18].copy()
print("After adult filter:", len(df))

# 3. Exclude pregnant women (Handling NaN safely)
if "HA54" in df.columns:
    df = df[~((df["HV104"] == 2) & (df["HA54"] == 1))].copy()
    print("After pregnancy exclusion:", len(df))

# ============================================================
# 3. BLOOD PRESSURE PROCESSING FUNCTION
# ============================================================

def get_bp_average(row, s_cols, d_cols):
    systolic = [row.get(c, np.nan) if pd.notna(row.get(c, np.nan)) and row.get(c, np.nan) < 900 else np.nan for c in s_cols]
    diastolic = [row.get(c, np.nan) if pd.notna(row.get(c, np.nan)) and row.get(c, np.nan) < 900 else np.nan for c in d_cols]

    if not np.isnan(systolic[1]) and not np.isnan(systolic[2]):
        sbp = (systolic[1] + systolic[2]) / 2
        dbp = (diastolic[1] + diastolic[2]) / 2
        r_type = "2nd_and_3rd"
    elif not np.isnan(systolic[1]):
        sbp = systolic[1]
        dbp = diastolic[1]
        r_type = "2nd_only"
    elif not np.isnan(systolic[0]):
        sbp = systolic[0]
        dbp = diastolic[0]
        r_type = "1st_only"
    else:
        sbp, dbp, r_type = np.nan, np.nan, "missing"

    return pd.Series([sbp, dbp, r_type])

# ============================================================
# 4. APPLY BLOOD PRESSURE & OUTLIER FILTERING
# ============================================================

df["sbp_final"] = np.nan
df["dbp_final"] = np.nan
df["bp_reading_type"] = np.nan
df["is_on_meds"] = 0

# Men
men = df["HV104"] == 1
df.loc[men, ["sbp_final", "dbp_final", "bp_reading_type"]] = df[men].apply(
    lambda r: get_bp_average(r, ["MBP9", "MBP13", "MBP22"], ["MBP10", "MBP14", "MBP23"]), axis=1).values
df.loc[men & (df["MBP19"] == 1), "is_on_meds"] = 1

# Women
women = df["HV104"] == 2
df.loc[women, ["sbp_final", "dbp_final", "bp_reading_type"]] = df[women].apply(
    lambda r: get_bp_average(r, ["WBP9", "WBP13", "WBP22"], ["WBP10", "WBP14", "WBP23"]), axis=1).values
df.loc[women & (df["WBP19"] == 1), "is_on_meds"] = 1

# Physiological Outliers Filter
sbp_outliers = (df["sbp_final"] < 70) | (df["sbp_final"] > 250)
dbp_outliers = (df["dbp_final"] < 40) | (df["dbp_final"] > 150)
df.loc[sbp_outliers, "sbp_final"] = np.nan
df.loc[dbp_outliers, "dbp_final"] = np.nan

# Drop Missing BP AND those with < 2 readings
df = df.dropna(subset=["sbp_final", "dbp_final"]).copy()
df = df[df["bp_reading_type"] != "1st_only"].copy()
print("\nAfter removing missing/<2 BP readings and outliers:", len(df))

# ============================================================
# 5. HYPERTENSION OUTCOME
# ============================================================

df["hypertension"] = np.where(
    (df["sbp_final"] >= 140) | (df["dbp_final"] >= 90) | (df["is_on_meds"] == 1), 1, 0
)

# ============================================================
# 6. BMI & GLUCOSE/DIABETES
# ============================================================

# BMI
df["bmi_raw"] = np.where(df["HV104"] == 1, df["HB40"], df["HA40"])
df["BMI"] = np.where(df["bmi_raw"] < 9000, df["bmi_raw"] / 100, np.nan)
df.loc[(df["BMI"] < 12) | (df["BMI"] > 60), "BMI"] = np.nan # Outlier filter

df["BMI_category"] = pd.cut(
    df["BMI"], bins=[0, 18.5, 23, 27.5, 100],
    labels=["Underweight", "Normal", "Overweight", "Obese"]
)

# Glucose & Diabetes
df["glucose"] = df["SB367"].fillna(df["SB267"])
df.loc[df["glucose"] >= 900, "glucose"] = np.nan

# Check for diabetes medication variable (commonly SB269 in BDHS)
if "SB269" in df.columns:
    meds_condition = (df["SB269"] == 1)
else:
    meds_condition = False

df["diabetes"] = np.where((df["glucose"] >= 126) | meds_condition, 1, 0)
df["prediabetes"] = np.where((df["glucose"] >= 100) & (df["glucose"] < 126), 1, 0)

# ============================================================
# 7. SOCIODEMOGRAPHIC & HOUSEHOLD
# ============================================================

df["age_group"] = pd.cut(df["HV105"], bins=[18, 29, 39, 49, 59, 69, 120],
                         labels=["18-29", "30-39", "40-49", "50-59", "60-69", "70+"], include_lowest=True)
df["HV106"] = df["HV106"].replace(8, np.nan) # Remove don't know for education
df["crowding_index"] = (df["HV009"] / df["HV216"]).replace([np.inf, -np.inf], np.nan)
df["improved_water"] = np.where(df["HV201"].isin([11, 12, 13, 14, 21, 31, 41, 51, 61]), 1, 0)
df["improved_toilet"] = np.where(df["HV205"].isin([11, 12, 13, 14, 21, 22, 23]), 1, 0)
df["electricity"] = df["HV206"]
df["mobile_phone"] = df["HV243A"] if "HV243A" in df.columns else np.nan

# ============================================================
# 8. COMPLETE-CASE SELECTION
# ============================================================

final_variables = {
    "hypertension": "hypertension", "sbp_final": "sbp_final", "dbp_final": "dbp_final",
    "HV105": "age", "age_group": "age_group", "HV104": "sex", "HV106": "education_level",
    "HV115": "marital_status", "HV025": "residence_type", "HV024": "division", "HV270": "wealth_index",
    "BMI": "BMI", "BMI_category": "BMI_category", "glucose": "glucose", "diabetes": "diabetes",
    "HV009": "household_size", "crowding_index": "crowding_index", "electricity": "electricity",
    "improved_water": "improved_water", "improved_toilet": "improved_toilet",
    "HV005": "sample_weight", "HV021": "cluster_id", "HV023": "strata_id"
}

df_final = df[list(final_variables.keys())].rename(columns=final_variables)
df_final["sample_weight"] = df_final["sample_weight"] / 1000000
df_final = df_final.drop_duplicates()

complete_case_vars = [
    "hypertension", "age", "age_group", "sex", "education_level", "marital_status",
    "residence_type", "division", "wealth_index", "BMI", "BMI_category", "diabetes",
    "household_size", "crowding_index", "cluster_id", "strata_id"
]

df_final = df_final.dropna(subset=complete_case_vars).copy()
print("\nFinal analytic sample size:", len(df_final))

# ============================================================
# 9. MACHINE LEARNING PREPROCESSING
# ============================================================

print("\n" + "=" * 70)
print("MACHINE LEARNING PREPROCESSING")
print("=" * 70)

# 9.1 One-Hot Encoding Categorical Variables
categorical_cols = ["age_group", "sex", "education_level", "marital_status",
                    "residence_type", "division", "wealth_index", "BMI_category"]

df_ml = pd.get_dummies(df_final, columns=categorical_cols, drop_first=True)

# 9.2 Z-Score Normalization for Continuous Variables
continuous_cols = ["age", "BMI", "household_size", "crowding_index"]
scaler = StandardScaler()
df_ml[continuous_cols] = scaler.fit_transform(df_ml[continuous_cols])

# 9.3 Cluster-Aware Train/Val/Test Split (60/20/20)
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_val_idx, test_idx = next(gss_test.split(df_ml, groups=df_ml['cluster_id']))

df_train_val = df_ml.iloc[train_val_idx].copy()
df_test = df_ml.iloc[test_idx].copy()

# Split the remaining 80% into 60% Train / 20% Val (which is 25% of the 80%)
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss_val.split(df_train_val, groups=df_train_val['cluster_id']))

df_train = df_train_val.iloc[train_idx].copy()
df_val = df_train_val.iloc[val_idx].copy()

print(f"Training Set: {len(df_train)} records")
print(f"Validation Set: {len(df_val)} records")
print(f"Testing Set: {len(df_test)} records")

# ============================================================
# 10. EXPORTS
# ============================================================

df_final.to_csv("BDHS_2022_HYPERTENSION_COMPLETE_CASE.csv", index=False)
df_train.to_csv("BDHS_2022_TRAIN.csv", index=False)
df_val.to_csv("BDHS_2022_VAL.csv", index=False)
df_test.to_csv("BDHS_2022_TEST.csv", index=False)

print("\nPreprocessed ML datasets exported successfully!")

INITIAL DATA
Initial shape: (132463, 549)

After de facto filter: 126597
After adult filter: 81881
After pregnancy exclusion: 81333


/tmp/ipykernel_3759/3679319455.py:79: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd', '2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd', 'missing', 'missing', 'missing', 'missing', 'missing', '2nd_and_3rd', 


After removing missing/<2 BP readings and outliers: 13274

Final analytic sample size: 13147

MACHINE LEARNING PREPROCESSING
Training Set: 7900 records
Validation Set: 2608 records
Testing Set: 2639 records

Preprocessed ML datasets exported successfully!


In [ ]:
# ============================================================
# BDHS 2022 HYPERTENSION DATASET
# FINAL PUBLICATION-QUALITY COMPLETE-CASE & ML PIPELINE
# COMBINED MEN + WOMEN — UNIFIED VERSION
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_spss(
    "/content/BDPR81FL.SAV",
    convert_categoricals=False
)

print("=" * 70)
print("BDHS 2022 HYPERTENSION ANALYTIC PIPELINE")
print("=" * 70)
print(f"Initial dataset shape: {df.shape}")

# ============================================================
# 2. ELIGIBILITY CRITERIA
# ============================================================

print("\n" + "=" * 70)
print("APPLYING ELIGIBILITY CRITERIA")
print("=" * 70)

# 2.1 De facto residents only
before = len(df)
df = df[df["HV103"] == 1].copy()
print(f"1. De facto residents filter | Removed: {before - len(df)} | Remaining: {len(df)}")

# 2.2 Adults aged >=18 years
before = len(df)
df = df[df["HV105"] >= 18].copy()
print(f"2. Adults aged >=18 years    | Removed: {before - len(df)} | Remaining: {len(df)}")

# 2.3 Exclude currently pregnant women
if "HA54" in df.columns:
    before = len(df)
    df = df[~((df["HV104"] == 2) & (df["HA54"] == 1))].copy()
    print(f"3. Pregnant women excluded   | Removed: {before - len(df)} | Remaining: {len(df)}")
else:
    print("3. Pregnancy variable (HA54) not found in dataset.")

# ============================================================
# 3. BLOOD PRESSURE PROCESSING FUNCTION
# ============================================================

def get_bp_average(row, s_cols, d_cols):
    systolic = []
    diastolic = []

    # Clean Systolic readings (<900 filters missing/flagged values)
    for c in s_cols:
        val = row.get(c, np.nan)
        systolic.append(val if pd.notna(val) and val < 900 else np.nan)

    # Clean Diastolic readings
    for c in d_cols:
        val = row.get(c, np.nan)
        diastolic.append(val if pd.notna(val) and val < 900 else np.nan)

    # DHS Recommended Averaging Rule
    if not np.isnan(systolic[1]) and not np.isnan(systolic[2]):
        sbp = (systolic[1] + systolic[2]) / 2
        dbp = (diastolic[1] + diastolic[2]) / 2
        reading_type = "2nd_and_3rd"
    elif not np.isnan(systolic[1]):
        sbp = systolic[1]
        dbp = diastolic[1]
        reading_type = "2nd_only"
    elif not np.isnan(systolic[0]):
        sbp = systolic[0]
        dbp = diastolic[0]
        reading_type = "1st_only"
    else:
        sbp, dbp = np.nan, np.nan
        reading_type = "missing"

    return pd.Series([sbp, dbp, reading_type])

# ============================================================
# 4. BLOOD PRESSURE PROCESSING & COHORT EXTRACTION
# ============================================================
# ============================================================
# 4. BLOOD PRESSURE PROCESSING & COHORT EXTRACTION (FIXED)
# ============================================================

print("\n" + "=" * 70)
print("BLOOD PRESSURE PROCESSING & PHYSIOLOGICAL FILTERING")
print("=" * 70)

# FIX: Explicitly initialize blood pressure columns with correct dtypes
df["sbp_final"] = np.nan                        # Float
df["dbp_final"] = np.nan                        # Float
df["bp_reading_type"] = np.nan                  # Will hold strings
df["bp_reading_type"] = df["bp_reading_type"].astype(object) # <-- THIS FIXES THE WARNING
df["is_on_meds"] = 0                            # Integer

# 4.1 Process Male Subpopulation
men = df["HV104"] == 1
if men.any():
    bp_men = df[men].apply(
        lambda r: get_bp_average(r, ["MBP9", "MBP13", "MBP22"], ["MBP10", "MBP14", "MBP23"]),
        axis=1
    )
    df.loc[men, ["sbp_final", "dbp_final", "bp_reading_type"]] = bp_men.values
    df.loc[men & (df["MBP19"] == 1), "is_on_meds"] = 1

# 4.2 Process Female Subpopulation
women = df["HV104"] == 2
if women.any():
    bp_women = df[women].apply(
        lambda r: get_bp_average(r, ["WBP9", "WBP13", "WBP22"], ["WBP10", "WBP14", "WBP23"]),
        axis=1
    )
    df.loc[women, ["sbp_final", "dbp_final", "bp_reading_type"]] = bp_women.values
    df.loc[women & (df["WBP19"] == 1), "is_on_meds"] = 1

# 4.3 Apply Physiological Outlier Cutoffs
sbp_outliers = (df["sbp_final"] < 70) | (df["sbp_final"] > 250)
dbp_outliers = (df["dbp_final"] < 40) | (df["dbp_final"] > 150)

print(f"SBP outliers masked (outside 70-250 mmHg): {sbp_outliers.sum()}")
print(f"DBP outliers masked (outside 40-150 mmHg): {dbp_outliers.sum()}")

df.loc[sbp_outliers, "sbp_final"] = np.nan
df.loc[dbp_outliers, "dbp_final"] = np.nan

# 4.4 Strict Measurement Enforcement: Exclude missing AND incomplete (<2 readings) records
before_bp = len(df)
df = df.dropna(subset=["sbp_final", "dbp_final"]).copy()
df = df[df["bp_reading_type"] != "1st_only"].copy()
print(f"BP Measurement Exclusion | Removed: {before_bp - len(df)} | Remaining: {len(df)}")

# ============================================================
# 5. HYPERTENSION CLASSIFICATION (JNC-7 OUTCOME)
# ============================================================

df["hypertension"] = np.where(
    (df["sbp_final"] >= 140) | (df["dbp_final"] >= 90) | (df["is_on_meds"] == 1),
    1,
    0
)

# ============================================================
# 6. COVARIATE ENGINEERING (BMI & CLINICAL DIABETES)
# ============================================================

print("\n" + "=" * 70)
print("ANTHROPOMETRIC & METABOLIC COVARIATE PROCESSING")
print("=" * 70)

# 6.1 BMI Extraction and Clean Outliers
df["bmi_raw"] = np.where(df["HV104"] == 1, df["HB40"], df["HA40"])
df["BMI"] = np.where(df["bmi_raw"] < 9000, df["bmi_raw"] / 100, np.nan)
bmi_outliers = (df["BMI"] < 12) | (df["BMI"] > 60)
df.loc[bmi_outliers, "BMI"] = np.nan
print(f"BMI physiological outliers masked (outside 12-60 kg/m²): {bmi_outliers.sum()}")

# Asian-Specific Cutoffs
df["BMI_category"] = pd.cut(
    df["BMI"],
    bins=[0, 18.5, 23, 27.5, 100],
    labels=["Underweight", "Normal", "Overweight", "Obese"]
)

# 6.2 Fasting Blood Glucose & Diabetes Protocol
df["glucose"] = df["SB367"].fillna(df["SB267"])
df.loc[df["glucose"] >= 900, "glucose"] = np.nan

# Evaluate medication status if documented (SB269)
diabetes_meds = (df["SB269"] == 1) if "SB269" in df.columns else False

df["diabetes"] = np.where((df["glucose"] >= 126) | diabetes_meds, 1, 0)
df["prediabetes"] = np.where((df["glucose"] >= 100) & (df["glucose"] < 126), 1, 0)

# ============================================================
# 7. SOCIODEMOGRAPHIC & ENVIRONMENT VARIABLES
# ============================================================

# Categorize Age groups (10-year brackets)
df["age_group"] = pd.cut(
    df["HV105"],
    bins=[18, 29, 39, 49, 59, 69, 120],
    labels=["18-29", "30-39", "40-49", "50-59", "60-69", "70+"],
    include_lowest=True
)

df["HV106"] = df["HV106"].replace(8, np.nan)  # Structural cleanup: 'don't know' -> nan
df["crowding_index"] = (df["HV009"] / df["HV216"]).replace([np.inf, -np.inf], np.nan)
df["improved_water"] = np.where(df["HV201"].isin([11, 12, 13, 14, 21, 31, 41, 51, 61]), 1, 0)
df["improved_toilet"] = np.where(df["HV205"].isin([11, 12, 13, 14, 21, 22, 23]), 1, 0)
df["electricity"] = df["HV206"]
df["mobile_phone"] = df["HV243A"] if "HV243A" in df.columns else np.nan

# ============================================================
# 8. STRUCTURAL FILTERING & COMPLETE-CASE ISOLATION
# ============================================================

print("\n" + "=" * 70)
print("STRUCTURAL RECONSTRUCTION & COMPLETE-CASE SELECTION")
print("=" * 70)

final_variables = {
    "hypertension": "hypertension", "sbp_final": "sbp_final", "dbp_final": "dbp_final",
    "HV105": "age", "age_group": "age_group", "HV104": "sex", "HV106": "education_level",
    "HV115": "marital_status", "HV025": "residence_type", "HV024": "division", "HV270": "wealth_index",
    "BMI": "BMI", "BMI_category": "BMI_category", "glucose": "glucose", "diabetes": "diabetes",
    "prediabetes": "prediabetes", "HV009": "household_size", "crowding_index": "crowding_index",
    "electricity": "electricity", "improved_water": "improved_water", "improved_toilet": "improved_toilet",
    "mobile_phone": "mobile_phone", "HV005": "sample_weight", "HV021": "cluster_id", "HV023": "strata_id"
}

df_final = df[list(final_variables.keys())].rename(columns=final_variables)

# Apply standard normalization matrix to DHS survey weight values
df_final["sample_weight"] = df_final["sample_weight"] / 1000000

# Primary structural de-duplication
before_dup = len(df_final)
df_final = df_final.drop_duplicates()
print(f"Duplicate records removed: {before_dup - len(df_final)}")

# Covariates list requiring complete records for multi-variable analyses
complete_case_vars = [
    "hypertension", "age", "age_group", "sex", "education_level", "marital_status",
    "residence_type", "division", "wealth_index", "BMI", "BMI_category", "diabetes",
    "household_size", "crowding_index", "improved_water", "improved_toilet"
]

before_cc = len(df_final)
df_final = df_final.dropna(subset=complete_case_vars).copy()
print(f"Missing covariate exclusion | Removed: {before_cc - len(df_final)}")
print(f"Final analytical sample size (N): {len(df_final)}")

# Save descriptive base study dataset
df_final.to_csv("BDHS_2022_HYPERTENSION_COMPLETE_CASE.csv", index=False)

# ============================================================
# 9. MACHINE LEARNING PREPROCESSING ENGINE
# ============================================================

print("\n" + "=" * 70)
print("MACHINE LEARNING PIPELINE PREPROCESSING")
print("=" * 70)

# 9.1 Matrix Nominal Transformation via One-Hot Encoding
categorical_cols = ["age_group", "sex", "education_level", "marital_status",
                    "residence_type", "division", "wealth_index", "BMI_category"]

df_ml = pd.get_dummies(df_final, columns=categorical_cols, drop_first=True)

# 9.2 Feature Scaling via Z-Score Standardization
continuous_cols = ["age", "BMI", "household_size", "crowding_index", "glucose"]
scaler = StandardScaler()
df_ml[continuous_cols] = scaler.fit_transform(df_ml[continuous_cols])

# 9.3 Cluster-Aware Train/Validation/Test Partitioning Matrix (60% / 20% / 20%)
# Splits are executed explicitly to avoid cluster-level and information leakages
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_val_idx, test_idx = next(gss_test.split(df_ml, groups=df_ml['cluster_id']))

df_train_val = df_ml.iloc[train_val_idx].copy()
df_test = df_ml.iloc[test_idx].copy()

# Partitioning the internal subset to establish evaluation matrices (0.25 * 0.80 = 0.20)
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss_val.split(df_train_val, groups=df_train_val['cluster_id']))

df_train = df_train_val.iloc[train_idx].copy()
df_val = df_train_val.iloc[val_idx].copy()

print(f"→ Training Split Size   (60%): {len(df_train)} rows")
print(f"→ Validation Split Size (20%): {len(df_val)} rows")
print(f"→ Testing Split Size    (20%): {len(df_test)} rows")

# ============================================================
# 10. MATRIX SYSTEM EXPORTATIONS
# ============================================================

df_train.to_csv("BDHS_2022_TRAIN.csv", index=False)
df_val.to_csv("BDHS_2022_VAL.csv", index=False)
df_test.to_csv("BDHS_2022_TEST.csv", index=False)

print("\n" + "=" * 70)
print("PIPELINE PROCESSING EXPORTS TERMINATED SUCCESSFULLY")
print("=" * 70)

BDHS 2022 HYPERTENSION ANALYTIC PIPELINE
Initial dataset shape: (132463, 549)

APPLYING ELIGIBILITY CRITERIA
1. De facto residents filter | Removed: 5866 | Remaining: 126597
2. Adults aged >=18 years    | Removed: 44716 | Remaining: 81881
3. Pregnant women excluded   | Removed: 548 | Remaining: 81333

BLOOD PRESSURE PROCESSING & PHYSIOLOGICAL FILTERING
SBP outliers masked (outside 70-250 mmHg): 0
DBP outliers masked (outside 40-150 mmHg): 0
BP Measurement Exclusion | Removed: 68059 | Remaining: 13274

ANTHROPOMETRIC & METABOLIC COVARIATE PROCESSING
BMI physiological outliers masked (outside 12-60 kg/m²): 0

STRUCTURAL RECONSTRUCTION & COMPLETE-CASE SELECTION
Duplicate records removed: 0
Missing covariate exclusion | Removed: 127
Final analytical sample size (N): 13147

MACHINE LEARNING PIPELINE PREPROCESSING
→ Training Split Size   (60%): 7900 rows
→ Validation Split Size (20%): 2608 rows
→ Testing Split Size    (20%): 2639 rows

PIPELINE PROCESSING EXPORTS TERMINATED SUCCESSFULLY


In [ ]:
# ============================================================
# EXTRACTING EXACT GENDER STRATA COUNTS FOR PUBLICATION
# ============================================================
print("\n" + "=" * 70)
print("MANUSCRIPT REPORTING: MALE VS FEMALE CASE DETAILS")
print("=" * 70)

# 1. Total adults entering the BP processing stage (by Sex)
print("Adults entering BP analysis phase by Sex (1=Male, 2=Female):")
print(df.groupby("HV104").size())

# 2. Breakdown of Blood Pressure measurement types completed
print("\nBlood Pressure completion types by Sex:")
print(pd.crosstab(df["HV104"], df["bp_reading_type"]))

# 3. Final complete-case analytical sample split
print("\nFinal Analytical Sample Size (N = 13,147) Split:")
male_count = (df_final["sex"] == 1).sum()
female_count = (df_final["sex"] == 2).sum()
print(f"→ Final Valid Males   (sex=1): {male_count} rows ({male_count/len(df_final)*100:.2f}%)")
print(f"→ Final Valid Females (sex=2): {female_count} rows ({female_count/len(df_final)*100:.2f}%)")

# 4. Final Hypertension Outcome Distribution by Sex
print("\nHypertension Prevalence within Final Sample Split:")
print(pd.crosstab(df_final["sex"], df_final["hypertension"], margins=True))


MANUSCRIPT REPORTING: MALE VS FEMALE CASE DETAILS
Adults entering BP analysis phase by Sex (1=Male, 2=Female):
HV104
1.0    6027
2.0    7247
dtype: int64

Blood Pressure completion types by Sex:
bp_reading_type  2nd_and_3rd  2nd_only
HV104                                 
1.0                     5921       106
2.0                     7134       113

Final Analytical Sample Size (N = 13,147) Split:
→ Final Valid Males   (sex=1): 5986 rows (45.53%)
→ Final Valid Females (sex=2): 7161 rows (54.47%)

Hypertension Prevalence within Final Sample Split:
hypertension      0     1    All
sex                             
1.0            4956  1030   5986
2.0            5427  1734   7161
All           10383  2764  13147


In [ ]:
# ==============================================================================
# BDHS 2022 HYPERTENSION ANALYTIC PIPELINE (UNIFIED MEN + WOMEN VERSION)
# ==============================================================================
# Fully Optimized to Eliminate FutureWarnings & Prevent ML Data Leakage
# ==============================================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

# ------------------------------------------------------------------------------
# 1. DATA ACQUISITION
# ------------------------------------------------------------------------------
print("=" * 80)
print("LOADING DATASET")
print("=" * 80)

df = pd.read_spss("/content/BDPR81FL.SAV", convert_categoricals=False)

print(f"Initial raw dataset shape: {df.shape}")

# ------------------------------------------------------------------------------
# 2. HOUSEHOLD ELIGIBILITY CRITERIA
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("APPLYING ELIGIBILITY CRITERIA")
print("=" * 80)

# 2.1 Filter by De Facto Residential Status
n_before = len(df)
df = df[df["HV103"] == 1].copy()
print(f"1. De facto residents filter | Removed: {n_before - len(df):<6} | Remaining: {len(df)}")

# 2.2 Filter by Target Adult Age (>= 18 Years)
n_before = len(df)
df = df[df["HV105"] >= 18].copy()
print(f"2. Adults aged >=18 years    | Removed: {n_before - len(df):<6} | Remaining: {len(df)}")

# 2.3 Exclude Currently Pregnant Women (Gestational Cardiovascular Variations)
if "HA54" in df.columns:
    n_before = len(df)
    df = df[~((df["HV104"] == 2) & (df["HA54"] == 1))].copy()
    print(f"3. Pregnant women excluded   | Removed: {n_before - len(df):<6} | Remaining: {len(df)}")
else:
    print("3. Pregnancy variable (HA54) not found. Skipping filter.")

# ------------------------------------------------------------------------------
# 3. BLOOD PRESSURE STRUCTURAL EXTRACTION FUNCTION
# ------------------------------------------------------------------------------
def get_bp_average(row, s_cols, d_cols):
    systolic = []
    diastolic = []

    # Clean extreme flags/system missing values (coded as >= 900 in DHS)
    for c in s_cols:
        val = row.get(c, np.nan)
        systolic.append(val if pd.notna(val) and val < 900 else np.nan)

    for c in d_cols:
        val = row.get(c, np.nan)
        diastolic.append(val if pd.notna(val) and val < 900 else np.nan)

    # Standard Epidemiological Rule (Average of 2nd and 3rd readings)
    if not np.isnan(systolic[1]) and not np.isnan(systolic[2]):
        sbp = (systolic[1] + systolic[2]) / 2
        dbp = (diastolic[1] + diastolic[2]) / 2
        reading_type = "2nd_and_3rd"
    elif not np.isnan(systolic[1]):
        sbp = systolic[1]
        dbp = diastolic[1]
        reading_type = "2nd_only"
    elif not np.isnan(systolic[0]):
        sbp = systolic[0]
        dbp = diastolic[0]
        reading_type = "1st_only"
    else:
        sbp, dbp = np.nan, np.nan
        reading_type = "missing"

    return pd.Series([sbp, dbp, reading_type])

# ------------------------------------------------------------------------------
# 4. BLOOD PRESSURE SELECTION & PHYSIOLOGICAL MASKING
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("BLOOD PRESSURE PROCESSING & PHYSIOLOGICAL FILTERING")
print("=" * 80)

# Initialize columns with precise dtypes to resolve multi-type FutureWarnings
df["sbp_final"] = np.nan
df["dbp_final"] = np.nan
df["bp_reading_type"] = np.nan
df["bp_reading_type"] = df["bp_reading_type"].astype(object)  # Explicit Object Type
df["is_on_meds"] = 0

# 4.1 Process Male Demographics (Using Male Biomarker Modules)
men_mask = df["HV104"] == 1
if men_mask.any():
    bp_men = df[men_mask].apply(
        lambda r: get_bp_average(r, ["MBP9", "MBP13", "MBP22"], ["MBP10", "MBP14", "MBP23"]),
        axis=1
    )
    df.loc[men_mask, ["sbp_final", "dbp_final", "bp_reading_type"]] = bp_men.values
    if "MBP19" in df.columns:
        df.loc[men_mask & (df["MBP19"] == 1), "is_on_meds"] = 1

# 4.2 Process Female Demographics (Using Female Biomarker Modules)
women_mask = df["HV104"] == 2
if women_mask.any():
    bp_women = df[women_mask].apply(
        lambda r: get_bp_average(r, ["WBP9", "WBP13", "WBP22"], ["WBP10", "WBP14", "WBP23"]),
        axis=1
    )
    df.loc[women_mask, ["sbp_final", "dbp_final", "bp_reading_type"]] = bp_women.values
    if "WBP19" in df.columns:
        df.loc[women_mask & (df["WBP19"] == 1), "is_on_meds"] = 1

# 4.3 Apply Outlier Mitigation Protocol (Clinical Threshold Limits)
sbp_outliers = (df["sbp_final"] < 70) | (df["sbp_final"] > 250)
dbp_outliers = (df["dbp_final"] < 40) | (df["dbp_final"] > 150)

print(f"SBP outliers masked (outside 70-250 mmHg): {sbp_outliers.sum()}")
print(f"DBP outliers masked (outside 40-150 mmHg): {dbp_outliers.sum()}")

df.loc[sbp_outliers, "sbp_final"] = np.nan
df.loc[dbp_outliers, "dbp_final"] = np.nan

# 4.4 Dropping Single-Reading and Missing Cases
n_before = len(df)
df = df.dropna(subset=["sbp_final", "dbp_final"]).copy()
df = df[df["bp_reading_type"] != "1st_only"].copy()
print(f"BP Subsample Filter          | Removed: {n_before - len(df):<6} | Remaining: {len(df)}")

# 4.5 Standard Hypertension Categorization (JNC-7 Diagnostic Standard)
df["hypertension"] = np.where(
    (df["sbp_final"] >= 140) | (df["dbp_final"] >= 90) | (df["is_on_meds"] == 1),
    1,
    0
)

# ------------------------------------------------------------------------------
# 5. COVARIATE CLINICAL PREPARATION
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ANTHROPOMETRIC & METABOLIC COVARIATE PROCESSING")
print("=" * 80)

# 5.1 Extract and Clean BMI Values
df["bmi_raw"] = np.where(df["HV104"] == 1, df["HB40"], df["HA40"])
df["BMI"] = np.where(df["bmi_raw"] < 9000, df["bmi_raw"] / 100, np.nan)
bmi_outliers = (df["BMI"] < 12) | (df["BMI"] > 60)
df.loc[bmi_outliers, "BMI"] = np.nan
print(f"BMI biological outliers masked (outside 12-60 kg/m²): {bmi_outliers.sum()}")

# Establish Asian-Specific Classification Matrices
df["BMI_category"] = pd.cut(
    df["BMI"],
    bins=[0, 18.5, 23, 27.5, 100],
    labels=["Underweight", "Normal", "Overweight", "Obese"]
)

# 5.2 Fasting Blood Glucose (FBG) Assessment
glucose_col = "SB367" if "SB367" in df.columns else "SB267"
df["glucose"] = df[glucose_col]
df.loc[df["glucose"] >= 900, "glucose"] = np.nan

diabetes_meds = (df["SB269"] == 1) if "SB269" in df.columns else False
df["diabetes"] = np.where((df["glucose"] >= 126) | diabetes_meds, 1, 0)
df["prediabetes"] = np.where((df["glucose"] >= 100) & (df["glucose"] < 126), 1, 0)

# 5.3 Sociodemographic Transformations
df["age_group"] = pd.cut(
    df["HV105"],
    bins=[18, 29, 39, 49, 59, 69, 120],
    labels=["18-29", "30-39", "40-49", "50-59", "60-69", "70+"],
    include_lowest=True
)
df["HV106"] = df["HV106"].replace(8, np.nan)  # Convert 'Don't know' to Missing
df["crowding_index"] = (df["HV009"] / df["HV216"]).replace([np.inf, -np.inf], np.nan)
df["improved_water"] = np.where(df["HV201"].isin([11, 12, 13, 14, 21, 31, 41, 51, 61]), 1, 0)
df["improved_toilet"] = np.where(df["HV205"].isin([11, 12, 13, 14, 21, 22, 23]), 1, 0)

# ------------------------------------------------------------------------------
# 6. STRUCTURAL FILTERING & COMPLETE-CASE ISOLATION
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("STRUCTURAL RECONSTRUCTION & COMPLETE-CASE SELECTION")
print("=" * 80)

variable_mapping = {
    "hypertension": "hypertension", "sbp_final": "sbp_final", "dbp_final": "dbp_final",
    "HV105": "age", "age_group": "age_group", "HV104": "sex", "HV106": "education_level",
    "HV115": "marital_status", "HV025": "residence_type", "HV024": "division", "HV270": "wealth_index",
    "BMI": "BMI", "BMI_category": "BMI_category", "glucose": "glucose", "diabetes": "diabetes",
    "prediabetes": "prediabetes", "HV009": "household_size", "crowding_index": "crowding_index",
    "improved_water": "improved_water", "improved_toilet": "improved_toilet", "HV005": "sample_weight",
    "HV021": "cluster_id", "HV023": "strata_id"
}

df_final = df[list(variable_mapping.keys())].rename(columns=variable_mapping)
df_final["sample_weight"] = df_final["sample_weight"] / 1000000

# Deduplication Block
n_before = len(df_final)
df_final = df_final.drop_duplicates()
print(f"Duplicate records removed: {n_before - len(df_final)}")

# Isolate Rows with Missing Target Covariates
covariates_for_analysis = [
    "hypertension", "age", "age_group", "sex", "education_level", "marital_status",
    "residence_type", "division", "wealth_index", "BMI", "BMI_category", "diabetes",
    "household_size", "crowding_index", "improved_water", "improved_toilet"
]

n_before = len(df_final)
df_final = df_final.dropna(subset=covariates_for_analysis).copy()
print(f"Missing covariate exclusion  | Removed: {n_before - len(df_final):<6} | Remaining: {len(df_final)}")
print(f"Final analytical sample size (N): {len(df_final)}")

# Save Clean Core File
df_final.to_csv("BDHS_2022_HYPERTENSION_COMPLETE_CASE.csv", index=False)

# ------------------------------------------------------------------------------
# 7. MACHINE LEARNING ENGINE: ENCODING, SCALING, AND CLUSTER SPLITS
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("MACHINE LEARNING PIPELINE PREPROCESSING")
print("=" * 80)

# 7.1 Nominal Category Conversions via One-Hot Matrix Transformation
nominal_features = ["age_group", "sex", "education_level", "marital_status",
                    "residence_type", "division", "wealth_index", "BMI_category"]
df_ml = pd.get_dummies(df_final, columns=nominal_features, drop_first=True)

# 7.2 Standard Scaling Matrix Implementation
continuous_features = ["age", "BMI", "household_size", "crowding_index", "glucose"]
scaler = StandardScaler()
df_ml[continuous_features] = scaler.fit_transform(df_ml[continuous_features])

# 7.3 Leakage Proof Partitioning via GroupShuffleSplit (Cluster ID Control)
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_val_idx, test_idx = next(gss_test.split(df_ml, groups=df_ml['cluster_id']))

df_train_val = df_ml.iloc[train_val_idx].copy()
df_test = df_ml.iloc[test_idx].copy()

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss_val.split(df_train_val, groups=df_train_val['cluster_id']))

df_train = df_train_val.iloc[train_idx].copy()
df_val = df_train_val.iloc[val_idx].copy()

print(f"→ Training Split Size   (60%): {len(df_train)} rows")
print(f"→ Validation Split Size (20%): {len(df_val)} rows")
print(f"→ Testing Split Size    (20%): {len(df_test)} rows")

df_train.to_csv("BDHS_2022_TRAIN.csv", index=False)
df_val.to_csv("BDHS_2022_VAL.csv", index=False)
df_test.to_csv("BDHS_2022_TEST.csv", index=False)

# ------------------------------------------------------------------------------
# 8. DETAILED MANUSCRIPT GENDER SUMMARY BREAKDOWN
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("MANUSCRIPT SUMMARY: SEX STRATA STRATIFICATION REPORT")
print("=" * 80)

male_final = (df_final["sex"] == 1).sum()
female_final = (df_final["sex"] == 2).sum()

print(f"Final Study Participants Matrix:")
print(f"  → Total Evaluated Males   (sex=1): {male_final} records ({male_final/len(df_final)*100:.2f}%)")
print(f"  → Total Evaluated Females (sex=2): {female_final} records ({female_final/len(df_final)*100:.2f}%)")

print("\nCross-Tabulation Matrix: Hypertension Distribution by Sex Profile:")
cross_tab = pd.crosstab(
    df_final["sex"].map({1: "Male", 2: "Female"}),
    df_final["hypertension"].map({0: "Normal", 1: "Hypertensive"}),
    margins=True
)
print(cross_tab)

print("\n" + "=" * 80)
print("PIPELINE EXPORT COMPLETED SUCCESSFULLY")
print("=" * 80)

LOADING DATASET
Initial raw dataset shape: (132463, 549)

APPLYING ELIGIBILITY CRITERIA
1. De facto residents filter | Removed: 5866   | Remaining: 126597
2. Adults aged >=18 years    | Removed: 44716  | Remaining: 81881
3. Pregnant women excluded   | Removed: 548    | Remaining: 81333

BLOOD PRESSURE PROCESSING & PHYSIOLOGICAL FILTERING
SBP outliers masked (outside 70-250 mmHg): 0
DBP outliers masked (outside 40-150 mmHg): 0
BP Subsample Filter          | Removed: 68059  | Remaining: 13274

ANTHROPOMETRIC & METABOLIC COVARIATE PROCESSING
BMI biological outliers masked (outside 12-60 kg/m²): 0

STRUCTURAL RECONSTRUCTION & COMPLETE-CASE SELECTION
Duplicate records removed: 0
Missing covariate exclusion  | Removed: 127    | Remaining: 13147
Final analytical sample size (N): 13147

MACHINE LEARNING PIPELINE PREPROCESSING
→ Training Split Size   (60%): 7900 rows
→ Validation Split Size (20%): 2608 rows
→ Testing Split Size    (20%): 2639 rows

MANUSCRIPT SUMMARY: SEX STRATA STRATIFICATION 

## code

In [ ]:
# ==============================================================================
# BDHS 2022 HYPERTENSION ANALYTIC & SURVEY-AWARE MACHINE LEARNING PIPELINE
# ==============================================================================
# Core Target: JNC-7 Hypertension Outcome Mapping (N = 13,147 Complete Cases)
# Sampling Design: Two-Stage Stratified Cluster Random Sampling (675 EAs)
# Field Response Rates: 97% Female BP Response | 92% Male BP Response
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    accuracy_score,
    cohen_kappa_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    brier_score_loss,
    average_precision_score,
    precision_recall_curve
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# PUBLICATION-QUALITY GRAPHICS REFINEMENT
plt.style.use("default")
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "figure.dpi": 300
})

# ==============================================================================
# PHASE A: MULTI-STAGE SAMPLING FRAME & POPULATION FILTERS
# ==============================================================================
print("=" * 80)
print("BDHS 2022 DESIGN FRAMEWORK INFRASTRUCTURE ACTIVATION")
print("=" * 80)
print("1st-Stage Sampling Frame: 2011 Bangladesh Population & Housing Census")
print("Total Selected Clusters: 675 Enumeration Areas (237 Urban / 438 Rural)")
print("2nd-Stage Sampling Frame: Systematic Selection of ~45 Households/EA")
print("Total Survey Footprint: 30,330 Households Selected for Interview")
print("Biomarker Questionnaire Block: Random 1-in-6 Household Subsample")
print("Documented Field Compliance: 97% Female BP Response | 92% Male BP Response")
print("-" * 80)

# Load the comprehensive household roster file
df = pd.read_spss("/content/BDPR81FL.SAV", convert_categoricals=False)
print(f"Initial raw dataset footprint: {df.shape}")

print("\n" + "=" * 80)
print("APPLYING MANDATORY EPIDEMIOLOGICAL INCLUSION/EXCLUSION CRITERIA")
print("=" * 80)

# A.1 Isolate De Facto Residents (Slept in the house the night before the survey)
n_before = len(df)
df = df[df["HV103"] == 1].copy()
print(f"1. De facto residents filter | Removed: {n_before - len(df):<6} | Remaining: {len(df)}")

# A.2 Target Adult Subpopulation (Aged >= 18 Years old)
n_before = len(df)
df = df[df["HV105"] >= 18].copy()
print(f"2. Adults aged >=18 years    | Removed: {n_before - len(df):<6} | Remaining: {len(df)}")

# A.3 Exclude Gestational Physiology (Drop actively pregnant women)
if "HA54" in df.columns:
    n_before = len(df)
    df = df[~((df["HV104"] == 2) & (df["HA54"] == 1))].copy()
    print(f"3. Pregnant women excluded   | Removed: {n_before - len(df):<6} | Remaining: {len(df)}")

# ==============================================================================
# PHASE B: BIOMARKER INTERVIEW LOOP PROCESSING & OUTLIER MITIGATION
# ==============================================================================
def get_bp_average(row, s_cols, d_cols):
    """Calculates chronological BP means based on strict 10-min interval survey protocols."""
    systolic, diastolic = [], []
    for c in s_cols:
        val = row.get(c, np.nan)
        systolic.append(val if pd.notna(val) and val < 900 else np.nan)
    for c in d_cols:
        val = row.get(c, np.nan)
        diastolic.append(val if pd.notna(val) and val < 900 else np.nan)

    # Prioritize 2nd and 3rd reading average to eliminate white-coat artifacts
    if not np.isnan(systolic[1]) and not np.isnan(systolic[2]):
        sbp = (systolic[1] + systolic[2]) / 2
        dbp = (diastolic[1] + diastolic[2]) / 2
        reading_type = "2nd_and_3rd"
    elif not np.isnan(systolic[1]):
        sbp = systolic[1]
        dbp = diastolic[1]
        reading_type = "2nd_only"
    elif not np.isnan(systolic[0]):
        sbp = systolic[0]
        dbp = diastolic[0]
        reading_type = "1st_only"
    else:
        sbp, dbp = np.nan, np.nan
        reading_type = "missing"
    return pd.Series([sbp, dbp, reading_type])

# Initialize destination vectors with explicit data types
df["sbp_final"] = np.nan
df["dbp_final"] = np.nan
df["bp_reading_type"] = np.nan
df["bp_reading_type"] = df["bp_reading_type"].astype(object)
df["is_on_meds"] = 0

# B.1 Map and Process Male Biomarker Cohort Modules (Aged 15-54 Module Pool)
men = df["HV104"] == 1
if men.any():
    bp_men = df[men].apply(
        lambda r: get_bp_average(r, ["MBP9", "MBP13", "MBP22"], ["MBP10", "MBP14", "MBP23"]), axis=1
    )
    df.loc[men, ["sbp_final", "dbp_final", "bp_reading_type"]] = bp_men.values
    if "MBP19" in df.columns:
        df.loc[men & (df["MBP19"] == 1), "is_on_meds"] = 1

# B.2 Map and Process Female Biomarker Cohort Modules (Aged 15-49 Module Pool)
women = df["HV104"] == 2
if women.any():
    bp_women = df[women].apply(
        lambda r: get_bp_average(r, ["WBP9", "WBP13", "WBP22"], ["WBP10", "WBP14", "WBP23"]), axis=1
    )
    df.loc[women, ["sbp_final", "dbp_final", "bp_reading_type"]] = bp_women.values
    if "WBP19" in df.columns:
        df.loc[women & (df["WBP19"] == 1), "is_on_meds"] = 1

# B.3 Clinical Outlier Masking
sbp_outliers = (df["sbp_final"] < 70) | (df["sbp_final"] > 250)
dbp_outliers = (df["dbp_final"] < 40) | (df["dbp_final"] > 150)
df.loc[sbp_outliers, "sbp_final"] = np.nan
df.loc[dbp_outliers, "dbp_final"] = np.nan

# B.4 Drop Non-Biomarker Subsample Blocks and Single-Reading Artifacts
n_before = len(df)
df = df.dropna(subset=["sbp_final", "dbp_final"]).copy()
df = df[df["bp_reading_type"] != "1st_only"].copy()
print(f"4. BP Subsample Filter       | Removed: {n_before - len(df):<6} | Remaining: {len(df)}")

# B.5 Apply JNC-7 Multi-Condition Rule (SBP >= 140 OR DBP >= 90 OR Prescribed Meds)
df["hypertension"] = np.where(
    (df["sbp_final"] >= 140) | (df["dbp_final"] >= 90) | (df["is_on_meds"] == 1), 1, 0
)

# ==============================================================================
# PHASE C: ANTHROPOMETRIC & SOCIO-ECOLOGICAL FEATURE ENGINEERING
# ==============================================================================
# C.1 Extract and Standardize BMI to Asian-Specific Ranges
df["bmi_raw"] = np.where(df["HV104"] == 1, df["HB40"], df["HA40"])
df["BMI"] = np.where(df["bmi_raw"] < 9000, df["bmi_raw"] / 100, np.nan)
bmi_outliers = (df["BMI"] < 12) | (df["BMI"] > 60)
df.loc[bmi_outliers, "BMI"] = np.nan

df["BMI_category"] = pd.cut(
    df["BMI"], bins=[0, 18.5, 23.0, 27.5, 100],
    labels=["Underweight", "Normal", "Overweight", "Obese"]
)

# C.2 Metabolic Diabetes Feature Engineering
glucose_col = "SB367" if "SB367" in df.columns else "SB267"
df["glucose"] = df[glucose_col]
df.loc[df["glucose"] >= 900, "glucose"] = np.nan
diabetes_meds = (df["SB269"] == 1) if "SB269" in df.columns else False
df["diabetes"] = np.where((df["glucose"] >= 126) | diabetes_meds, 1, 0)

# C.3 Sociodemographic Stratification Features
df["age_group"] = pd.cut(
    df["HV105"], bins=[18, 29, 39, 49, 59, 69, 120],
    labels=["18-29", "30-39", "40-49", "50-59", "60-69", "70+"], include_lowest=True
)
df["HV106"] = df["HV106"].replace(8, np.nan)
df["crowding_index"] = (df["HV009"] / df["HV216"]).replace([np.inf, -np.inf], np.nan)
df["improved_water"] = np.where(df["HV201"].isin([11, 12, 13, 14, 21, 31, 41, 51, 61]), "1", "0")
df["improved_toilet"] = np.where(df["HV205"].isin([11, 12, 13, 14, 21, 22, 23]), "1", "0")

df["electricity"] = np.where(df["HV244"] == 1, "1", "0")
df["mobile_phone"] = np.where(df["HV243A"] == 1, "1", "0")
df["household_size_cat"] = pd.cut(df["HV009"], bins=[0, 4, 7, 100], labels=["Small", "Medium", "Large"])
df["crowding_category"] = pd.cut(df["crowding_index"], bins=[0, 1.5, 3.0, 100], labels=["Low", "Moderate", "High"])

# ==============================================================================
# PHASE D: MASTER MATRIX STANDARDIZATION & COMPLETE-CASE ISOLATION
# ==============================================================================
variable_mapping = {
    "hypertension": "hypertension", "HV105": "age", "age_group": "age_group", "HV104": "sex",
    "HV106": "education_level", "HV115": "marital_status", "HV025": "residence_type",
    "HV024": "division", "HV270": "wealth_index", "BMI": "BMI", "BMI_category": "BMI_category",
    "diabetes": "diabetes", "household_size_cat": "household_size_cat", "crowding_category": "crowding_category",
    "electricity": "electricity", "improved_water": "improved_water", "improved_toilet": "improved_toilet",
    "mobile_phone": "mobile_phone", "glucose": "glucose", "HV005": "sample_weight",
    "HV021": "cluster_id", "HV023": "strata_id"
}

df_final = df[list(variable_mapping.keys())].rename(columns=variable_mapping)
df_final["sample_weight"] = df_final["sample_weight"] / 1000000

# Deduplication and Null Extraction Across Predictors
df_final = df_final.drop_duplicates()
categorical_vars = [
    'age_group', 'sex', 'education_level', 'marital_status', 'residence_type',
    'division', 'wealth_index', 'BMI_category', 'diabetes', 'household_size_cat',
    'crowding_category', 'electricity', 'improved_water', 'improved_toilet', 'mobile_phone'
]
n_before = len(df_final)
df_final = df_final.dropna(subset=categorical_vars + ["age", "BMI", "glucose"]).copy()
print(f"5. Missing covariate filter   | Removed: {n_before - len(df_final):<6} | Remaining: {len(df_final)}")
print("-" * 80)
print(f"FINAL ANALYTICAL COMPLETE-CASE MATRIX SIZE (N): {len(df_final)}")
print(f"Unweighted Gender Split: Male n = {len(df_final[df_final['sex']=='1'])} | Female n = {len(df_final[df_final['sex']=='2'])}")
print("=" * 80)

# Save verified matrix to storage
df_final.to_csv("/content/BDHS_2022_HYPERTENSION_SELECTED_VARIABLES.csv", index=False)

# ==============================================================================
# PHASE E: MACHINE LEARNING PIPELINE PREPROCESSING (CLUSTER-AWARE SPLITS)
# ==============================================================================
target = "hypertension"
design_cols = ["cluster_id", "strata_id", "sample_weight"]
df_final[target] = df_final[target].astype(int)

for col in categorical_vars:
    df_final[col] = df_final[col].astype(str)

# Perform One-Hot Matrix Transformation (avoiding collinear dummy traps)
X = pd.get_dummies(df_final.drop(columns=[target] + design_cols), drop_first=True)
y = df_final[target]
w = df_final["sample_weight"]
clusters = df_final["cluster_id"]

# Leakage-Proof Partitioning via GroupShuffleSplit using Cluster IDs (Primary Sampling Units)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups=clusters))

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss2.split(X.iloc[train_val_idx], y.iloc[train_val_idx], groups=clusters.iloc[train_val_idx]))

X_train, X_val, X_test = X.iloc[train_idx], X.iloc[val_idx], X.iloc[test_idx]
y_train, y_val, y_test = y.iloc[train_idx], y.iloc[val_idx], y.iloc[test_idx]
w_train, w_val, w_test = w.iloc[train_idx], w.iloc[val_idx], w.iloc[test_idx]

# Apply Z-score Normalization to Continuous Vectors
continuous_features = ["age", "BMI", "glucose"]
scaler = StandardScaler()
X_train[continuous_features] = scaler.fit_transform(X_train[continuous_features])
X_val[continuous_features] = scaler.transform(X_val[continuous_features])
X_test[continuous_features] = scaler.transform(X_test[continuous_features])

print(f"→ Leakage-Proof Train Split Size (60%): {len(X_train)} rows")
print(f"→ Leakage-Proof Val Split Size   (20%): {len(X_val)} rows")
print(f"→ Leakage-Proof Test Split Size  (20%): {len(X_test)} rows")

# ==============================================================================
# PHASE F: METRIC FUNCTIONS & MODEL DICTIONARY CONFIGURATION
# ==============================================================================
def evaluate_metrics(y_true, p, w, threshold=0.5):
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, sample_weight=w).ravel()
    sensitivity = tp / (tp + fn + 1e-9)
    specificity = tn / (tn + fp + 1e-9)
    return {
        "Accuracy": accuracy_score(y_true, pred, sample_weight=w),
        "AUC": roc_auc_score(y_true, p, sample_weight=w),
        "PR_AUC": average_precision_score(y_true, p, sample_weight=w),
        "Kappa": cohen_kappa_score(y_true, pred, sample_weight=w),
        "Precision": precision_score(y_true, pred, sample_weight=w),
        "Recall": recall_score(y_true, pred, sample_weight=w),
        "F1": f1_score(y_true, pred, sample_weight=w),
        "Brier": brier_score_loss(y_true, p, sample_weight=w),
        "Sensitivity": sensitivity,
        "Specificity": specificity
    }

imb_ratio = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "Logistic Regression": LogisticRegression(penalty="l2", C=1, max_iter=2000, class_weight="balanced", solver="liblinear", random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, min_samples_leaf=20, min_samples_split=40, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=800, min_samples_leaf=10, min_samples_split=20, max_features="sqrt", class_weight="balanced", n_jobs=-1, random_state=42),
    "SVM": SVC(C=1, kernel="rbf", gamma="scale", probability=True, class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(n_estimators=1000, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, min_child_weight=5, scale_pos_weight=imb_ratio, eval_metric="auc", tree_method="hist", random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu", solver="adam", alpha=0.001, early_stopping=True, max_iter=500, random_state=42)
}

# ==============================================================================
# PHASE G: MODEL TRAINING & 5-FOLD NESTED GROUP CROSS-VALIDATION
# ==============================================================================
master_results = []
roc_plot_data = {}

for name, model in models.items():
    print(f"Executing Model Training Node: {name}")
    if name == "MLP":
        model.fit(X_train, y_train)
    else:
        model.fit(X_train, y_train, sample_weight=w_train)

    p_test = model.predict_proba(X_test)[:, 1]
    master_results.append([f"{name} (Train)"] + list(evaluate_metrics(y_train, model.predict_proba(X_train)[:, 1], w_train).values()))
    master_results.append([f"{name} (Val)"] + list(evaluate_metrics(y_val, model.predict_proba(X_val)[:, 1], w_val).values()))
    master_results.append([f"{name} (Test)"] + list(evaluate_metrics(y_test, p_test, w_test).values()))
    roc_plot_data[name] = p_test

# 5-Fold Group Cross-Validation Engine (Isolated by cluster_id)
gkf = GroupKFold(n_splits=5)
cv_results = []
for name, model in models.items():
    print(f"Cross-Validating Stability Framework: {name}")
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=clusters)):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]
        w_tr, w_te = w.iloc[tr], w.iloc[te]

        # Scale locally within the loop folds to block cross-leakage
        X_tr_s, X_te_s = X_tr.copy(), X_te.copy()
        X_tr_s[continuous_features] = scaler.fit_transform(X_tr[continuous_features])
        X_te_s[continuous_features] = scaler.transform(X_te[continuous_features])

        if name == "MLP":
            model.fit(X_tr_s, y_tr)
        else:
            model.fit(X_tr_s, y_tr, sample_weight=w_tr)

        metrics = evaluate_metrics(y_te, model.predict_proba(X_te_s)[:, 1], w_te)
        metrics["Model"] = name
        cv_results.append(metrics)

cv_df = pd.DataFrame(cv_results)

# ==============================================================================
# PHASE H: MANUSCRIPT PLOT GENERATION EXPORTS
# ==============================================================================
print("\n" + "="*80)
print("SAVING HIGH-RESOLUTION VISUALIZATION EXPORTS (300 DPI)")
print("="*80)

# Export 1: ROC and Precision-Recall Analytics Panel
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
cv_lookup = cv_df.groupby("Model")["AUC"].agg(["mean", "std"]).to_dict('index')

for name, probs in roc_plot_data.items():
    fpr, tpr, _ = roc_curve(y_test, probs, sample_weight=w_test)
    holdout_auc = roc_auc_score(y_test, probs, sample_weight=w_test)
    cv_mean, cv_std = cv_lookup[name]['mean'], cv_lookup[name]['std']

    ax1.plot(fpr, tpr, linewidth=2, label=f"{name}: {holdout_auc:.3f} (CV: {cv_mean:.3f} ± {cv_std:.3f})")
    precision, recall, _ = precision_recall_curve(y_test, probs, sample_weight=w_test)
    ax2.plot(recall, precision, linewidth=2, label=name)

ax1.plot([0, 1], [0, 1], linestyle='--', color='black', alpha=0.7, label="Chance (0.50)")
ax1.set_xlabel("False Positive Rate (Weighted)", fontweight='bold')
ax1.set_ylabel("True Positive Rate (Weighted)", fontweight='bold')
ax1.set_title("A. Weighted ROC Curves\nHoldout AUC vs 5-Fold CV Status", fontweight='bold', pad=15)
ax1.legend(loc="lower right", frameon=True, shadow=True)
ax1.grid(alpha=0.3)

ax2.set_xlabel("Recall (Weighted)", fontweight='bold')
ax2.set_ylabel("Precision (Weighted)", fontweight='bold')
ax2.set_title("B. Weighted Precision-Recall Curves", fontweight='bold', pad=15)
ax2.legend(loc="lower left", frameon=True, shadow=True)
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("ROC_PR_Comparison.png", dpi=300)
plt.close()

# Export 2: Comparative Metric Panel across Holdout Partitions
cols = ["Model", "Accuracy", "AUC", "PR_AUC", "Kappa", "Precision", "Recall", "F1", "Brier", "Sensitivity", "Specificity"]
results_df = pd.DataFrame(master_results, columns=cols)
plot_df = results_df[results_df['Model'].str.contains("(Test)")].copy()
name_mapping = {"Logistic Regression (Test)": "LR", "Decision Tree (Test)": "DT", "Random Forest (Test)": "RF", "SVM (Test)": "SVC", "XGBoost (Test)": "XGB", "MLP (Test)": "MLP"}
plot_df['Model_Short'] = plot_df['Model'].map(name_mapping)

metrics_to_plot = ["Accuracy", "Recall", "Specificity", "AUC"]
display_labels = ['Accuracy', 'Recall', 'Specificity', 'ROC AUC']
colors = ['#1f77b4', '#2ca02c', '#8c564b', '#7f7f7f']

x = np.arange(len(plot_df))
width = 0.18
fig, ax = plt.subplots(figsize=(14, 7), facecolor='#f8f9fa')
ax.set_facecolor('#f8f9fa')

for i, col in enumerate(metrics_to_plot):
    bars = ax.bar(x + i*width, plot_df[col], width, label=display_labels[i], color=colors[i])
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01, f'{h:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(plot_df['Model_Short'], fontweight='bold')
ax.set_title("Model Comparison Based on Performance Metrics (Holdout Set)", fontweight='bold', fontsize=16, pad=40)
ax.set_ylabel("Score", fontweight='bold')
ax.set_ylim(0, 1.1)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.1), ncol=4, frameon=False, fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("Model_Metrics_Comparison.png", dpi=300)
plt.close()

print("\n" + "=" * 80)
print("PIPELINE SYSTEM TERMINATED SUCCESSFULLY | LOGS AND PLOTS SECURED")
print("=" * 80)

BDHS 2022 DESIGN FRAMEWORK INFRASTRUCTURE ACTIVATION
1st-Stage Sampling Frame: 2011 Bangladesh Population & Housing Census
Total Selected Clusters: 675 Enumeration Areas (237 Urban / 438 Rural)
2nd-Stage Sampling Frame: Systematic Selection of ~45 Households/EA
Total Survey Footprint: 30,330 Households Selected for Interview
Biomarker Questionnaire Block: Random 1-in-6 Household Subsample
Documented Field Compliance: 97% Female BP Response | 92% Male BP Response
--------------------------------------------------------------------------------
Initial raw dataset footprint: (132463, 549)

APPLYING MANDATORY EPIDEMIOLOGICAL INCLUSION/EXCLUSION CRITERIA
1. De facto residents filter | Removed: 5866   | Remaining: 126597
2. Adults aged >=18 years    | Removed: 44716  | Remaining: 81881
3. Pregnant women excluded   | Removed: 548    | Remaining: 81333
4. BP Subsample Filter       | Removed: 68059  | Remaining: 13274
5. Missing covariate filter   | Removed: 7420   | Remaining: 5854
-----------

## Another difinition

In [ ]:
# ============================================================
# BDHS 2022 HYPERTENSION DATASET
# PR FILE ONLY
# COMPLETE-CASE PUBLICATION-QUALITY PIPELINE
# COMBINED MEN + WOMEN
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_spss(
    "/content/BDPR81FL.SAV",
    convert_categoricals=False
)

print("=" * 70)
print("INITIAL DATA")
print("=" * 70)

print("Initial shape:", df.shape)

# ============================================================
# 2. ELIGIBILITY CRITERIA
# ============================================================

# de facto residents
df = df[df["HV103"] == 1].copy()

print("\nAfter de facto filter:", len(df))

# adults ≥18 years
df = df[df["HV105"] >= 18].copy()

print("After adult filter:", len(df))

# ============================================================
# 3. BLOOD PRESSURE FUNCTION
# ============================================================

def get_bp_average(row, s_cols, d_cols):

    systolic = []
    diastolic = []

    for c in s_cols:

        val = row.get(c, np.nan)

        if pd.notna(val) and val < 900:
            systolic.append(val)
        else:
            systolic.append(np.nan)

    for c in d_cols:

        val = row.get(c, np.nan)

        if pd.notna(val) and val < 900:
            diastolic.append(val)
        else:
            diastolic.append(np.nan)

    # DHS recommendation:
    # average 2nd and 3rd readings

    if not np.isnan(systolic[1]) and not np.isnan(systolic[2]):

        sbp = (systolic[1] + systolic[2]) / 2
        dbp = (diastolic[1] + diastolic[2]) / 2

    elif not np.isnan(systolic[1]):

        sbp = systolic[1]
        dbp = diastolic[1]

    elif not np.isnan(systolic[0]):

        sbp = systolic[0]
        dbp = diastolic[0]

    else:

        sbp = np.nan
        dbp = np.nan

    return pd.Series([sbp, dbp])

# ============================================================
# 4. BLOOD PRESSURE VARIABLES
# ============================================================

print("\n" + "=" * 70)
print("BLOOD PRESSURE PROCESSING")
print("=" * 70)

df["sbp_final"] = np.nan
df["dbp_final"] = np.nan
df["is_on_meds"] = 0

# ---------------- MEN ----------------

men = df["HV104"] == 1

bp_men = df[men].apply(
    lambda r: get_bp_average(
        r,
        ["MBP9", "MBP13", "MBP22"],
        ["MBP10", "MBP14", "MBP23"]
    ),
    axis=1
)

df.loc[
    men,
    ["sbp_final", "dbp_final"]
] = bp_men.values

df.loc[
    men & (df["MBP19"] == 1),
    "is_on_meds"
] = 1

# ---------------- WOMEN ----------------

women = df["HV104"] == 2

bp_women = df[women].apply(
    lambda r: get_bp_average(
        r,
        ["WBP9", "WBP13", "WBP22"],
        ["WBP10", "WBP14", "WBP23"]
    ),
    axis=1
)

df.loc[
    women,
    ["sbp_final", "dbp_final"]
] = bp_women.values

df.loc[
    women & (df["WBP19"] == 1),
    "is_on_meds"
] = 1

print("Men with valid BP:",
      df.loc[men, "sbp_final"].notna().sum())

print("Women with valid BP:",
      df.loc[women, "sbp_final"].notna().sum())

# ============================================================
# 5. KEEP VALID BP PARTICIPANTS
# ============================================================

before_bp = len(df)

df = df.dropna(
    subset=["sbp_final", "dbp_final"]
).copy()

after_bp = len(df)

print("\nAfter BP eligibility:", after_bp)
print("Removed:", before_bp - after_bp)

# ============================================================
# 6. HYPERTENSION OUTCOME
# ============================================================

df["hypertension"] = np.where(
    (df["sbp_final"] >= 140) |
    (df["dbp_final"] >= 90) |
    (df["is_on_meds"] == 1),
    1,
    0
)

print("\nHypertension distribution:")
print(df["hypertension"].value_counts())

# ============================================================
# 7. BMI
# ============================================================

print("\n" + "=" * 70)
print("BMI PROCESSING")
print("=" * 70)

df["bmi_raw"] = np.where(
    df["HV104"] == 1,
    df["HB40"],
    df["HA40"]
)

df["BMI"] = np.where(
    df["bmi_raw"] < 9000,
    df["bmi_raw"] / 100,
    np.nan
)

print("BMI missing:",
      df["BMI"].isna().sum())

# Asian BMI cutoff

df["BMI_category"] = pd.cut(
    df["BMI"],
    bins=[0, 18.5, 23, 27.5, 100],
    labels=[
        "Underweight",
        "Normal",
        "Overweight",
        "Obese"
    ]
)

# ============================================================
# 8. GLUCOSE / DIABETES
# ============================================================

print("\n" + "=" * 70)
print("GLUCOSE PROCESSING")
print("=" * 70)

df["glucose"] = df["SB367"].fillna(
    df["SB267"]
)

# invalid glucose

df.loc[
    df["glucose"] >= 900,
    "glucose"
] = np.nan

print("Glucose missing:",
      df["glucose"].isna().sum())

# diabetes

df["diabetes"] = np.where(
    df["glucose"] >= 126,
    1,
    0
)

# prediabetes

df["prediabetes"] = np.where(
    (df["glucose"] >= 100) &
    (df["glucose"] < 126),
    1,
    0
)

# ============================================================
# 9. AGE GROUPS
# ============================================================

df["age_group"] = pd.cut(
    df["HV105"],
    bins=[18, 29, 39, 49, 59, 69, 120],
    labels=[
        "18-29",
        "30-39",
        "40-49",
        "50-59",
        "60-69",
        "70+"
    ],
    include_lowest=True
)

# ============================================================
# 10. HOUSEHOLD VARIABLES
# ============================================================

df["crowding_index"] = (
    df["HV009"] / df["HV216"]
)

df["crowding_index"] = df[
    "crowding_index"
].replace(
    [np.inf, -np.inf],
    np.nan
)

# ============================================================
# 11. WATER SOURCE
# ============================================================

improved_water_codes = [
    11, 12, 13, 14,
    21,
    31,
    41,
    51,
    61
]

df["improved_water"] = np.where(
    df["HV201"].isin(improved_water_codes),
    1,
    0
)

# ============================================================
# 12. TOILET FACILITY
# ============================================================

improved_toilet_codes = [
    11, 12, 13, 14,
    21, 22, 23
]

df["improved_toilet"] = np.where(
    df["HV205"].isin(improved_toilet_codes),
    1,
    0
)

# # ============================================================
# # 13. CLEAN COOKING FUEL
# # ============================================================

# clean_fuel_codes = [
#     1, 2, 3, 4, 5
# ]

# df["clean_cooking_fuel"] = np.where(
#     df["HV226"].isin(clean_fuel_codes),
#     1,
#     0
# )

# ============================================================
# 14. ELECTRICITY
# ============================================================

df["electricity"] = df["HV206"]

# ============================================================
# 15. MOBILE PHONE
# ============================================================

if "HV243A" in df.columns:

    df["mobile_phone"] = df["HV243A"]

else:

    df["mobile_phone"] = np.nan

# # ============================================================
# # 16. INTERNET USE
# # ============================================================

# if "HV243B" in df.columns:

#     df["internet_use"] = df["HV243B"]

# else:

#     df["internet_use"] = np.nan

# # ============================================================
# # 17. SMOKING
# # ============================================================

# smoking_vars = [
#     "SM108", "SM109",
#     "S108", "S109"
# ]

# for c in smoking_vars:

#     if c not in df.columns:
#         df[c] = 0

# df["smoker"] = np.where(
#     (df["SM108"] == 1) |
#     (df["SM109"] == 1) |
#     (df["S108"] == 1) |
#     (df["S109"] == 1),
#     1,
#     0
# )

# ============================================================
# 18. WORKING STATUS
# ============================================================

# if "HV714" in df.columns:

#     df["currently_working"] = df["HV714"]

# else:

#     df["currently_working"] = np.nan

# ============================================================
# 19. EDUCATION CLEANING
# ============================================================

print("\n" + "=" * 70)
print("CLEANING SOCIODEMOGRAPHIC VARIABLES")
print("=" * 70)

# remove don't know

df["HV106"] = df["HV106"].replace(
    8,
    np.nan
)

# ============================================================
# 20. FINAL VARIABLE SELECTION
# ============================================================

final_variables = {

    # outcome
    "hypertension": "hypertension",

    # BP
    "sbp_final": "sbp_final",
    "dbp_final": "dbp_final",

    # demographic
    "HV105": "age",
    "age_group": "age_group",
    "HV104": "sex",
    "HV106": "education_level",
    "HV115": "marital_status",
    "HV025": "residence_type",
    "HV024": "division",

    # socioeconomic
    "HV270": "wealth_index",
    # "currently_working": "currently_working",

    # anthropometric
    "BMI": "BMI",
    "BMI_category": "BMI_category",

    # metabolic
    "glucose": "glucose",
    "diabetes": "diabetes",
    "prediabetes": "prediabetes",

    # behavioral
    # "smoker": "smoker",

    # household/environment
    "HV009": "household_size",
    "HV216": "sleeping_rooms",
    "crowding_index": "crowding_index",
    "electricity": "electricity",
    "improved_water": "improved_water",
    "improved_toilet": "improved_toilet",
    # "clean_cooking_fuel": "clean_cooking_fuel",

    # technology
    "mobile_phone": "mobile_phone",
    # "internet_use": "internet_use",

    # survey design
    "HV005": "sample_weight",
    "HV021": "cluster_id",
    "HV023": "strata_id"
}

# ============================================================
# 21. CREATE FINAL DATASET
# ============================================================

df_final = df[
    list(final_variables.keys())
].rename(
    columns=final_variables
)

# ============================================================
# 22. SURVEY WEIGHT
# ============================================================

df_final["sample_weight"] = (
    df_final["sample_weight"] / 1000000
)

# ============================================================
# 23. REMOVE DUPLICATES
# ============================================================

before_dup = len(df_final)

df_final = df_final.drop_duplicates()

after_dup = len(df_final)

print("\nDuplicates removed:",
      before_dup - after_dup)

# ============================================================
# 24. COMPLETE-CASE CLEANING
# ============================================================

print("\n" + "=" * 70)
print("COMPLETE-CASE CLEANING")
print("=" * 70)

print("\nMissing values BEFORE cleaning:")
print(df_final.isna().sum())

# variables required for complete-case analysis

complete_case_vars = [

    "hypertension",

    "age",
    "age_group",
    "sex",

    "education_level",
    "marital_status",
    "residence_type",
    "division",
    "wealth_index",

    "BMI",
    "BMI_category",

    "glucose",
    "diabetes",

    "household_size",
    "crowding_index",

    "improved_water",
    "improved_toilet",

    "sample_weight",
    "cluster_id",
    "strata_id"
]

before_complete = len(df_final)

df_final = df_final.dropna(
    subset=complete_case_vars
)

after_complete = len(df_final)

print("\nRows removed:",
      before_complete - after_complete)

print("Final analytic sample:",
      after_complete)

# ============================================================
# 25. FINAL MISSING CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL MISSING VALUES")
print("=" * 70)

print(df_final.isna().sum())

print("\nTotal missing values:",
      df_final.isna().sum().sum())

# ============================================================
# 26. EXPORT FINAL DATASET
# ============================================================

df_final.to_csv(
    "BDHS_2022_HYPERTENSION_COMPLETE_CASE.csv",
    index=False
)

# ============================================================
# 27. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET SUMMARY")
print("=" * 70)

print("Final shape:",
      df_final.shape)

print("\nHypertension prevalence:")
print(df_final["hypertension"].value_counts())

print("\nSex distribution:")
print(df_final["sex"].value_counts())

print("\nEducation distribution:")
print(df_final["education_level"].value_counts())

print("\nResidence distribution:")
print(df_final["residence_type"].value_counts())

print("\nWealth distribution:")
print(df_final["wealth_index"].value_counts())

print("\nDiabetes distribution:")
print(df_final["diabetes"].value_counts())

print("\nBMI category distribution:")
print(df_final["BMI_category"].value_counts())

# print("\nSmoking distribution:")
# print(df_final["smoker"].value_counts())

print("\nDataset successfully exported!")

# ============================================================
# 28. VARIABLE DESCRIPTION TABLE
# ============================================================
# ============================================================
# 28. VARIABLE DESCRIPTION TABLE (AUTOMATED)
# ============================================================
# ============================================================
# 28. VARIABLE DESCRIPTION TABLE
# ============================================================

description_map = {
    "htn": "Hypertension (SBP>=140, DBP>=90, or on medication)",
    "sbp": "Systolic Blood Pressure (mmHg)",
    "dbp": "Diastolic Blood Pressure (mmHg)",
    "age": "Age in years",
    "age_cat": "Age category (10-year intervals)",
    "sex": "Sex of respondent",
    "education": "Education level attained",
    "marital_status": "Current marital status",
    "urban_rural": "Type of residence (Urban/Rural)",
    "division": "Administrative division",
    "wealth_idx": "Wealth quintile",
    "bmi": "Body Mass Index (kg/m^2)",
    "bmi_cat": "Asian BMI classification",
    "glucose": "Blood glucose level (mg/dL)",
    "diabetes": "Diabetes status",
    "prediabetes": "Prediabetes status",
    "hh_size": "Number of household members",
    "rooms": "Number of sleeping rooms",
    "crowding": "Crowding index (Members/Room)",
    "electricity": "Household has electricity",
    "water_type": "Improved drinking water access",
    "toilet_type": "Improved sanitation access",
    "mobile": "Mobile phone ownership",
    "weight": "Survey sampling weight",
    "cluster": "Primary Sampling Unit (PSU)",
    "strata": "Sampling strata"
}

# Generate and Export
current_descriptions = [description_map.get(col, "N/A") for col in df_final.columns]
variable_description = pd.DataFrame({
    "Variable": df_final.columns,
    "Description": current_descriptions
})

variable_description.to_csv("samBDHS_2022_DESCRIPTIONS_SHORT.csv", index=False)

INITIAL DATA
Initial shape: (132463, 549)

After de facto filter: 126597
After adult filter: 81881

BLOOD PRESSURE PROCESSING
Men with valid BP: 6192
Women with valid BP: 7656

After BP eligibility: 13848
Removed: 68033

Hypertension distribution:
hypertension
0    10971
1     2877
Name: count, dtype: int64

BMI PROCESSING
BMI missing: 116

GLUCOSE PROCESSING
Glucose missing: 322

CLEANING SOCIODEMOGRAPHIC VARIABLES

Duplicates removed: 0

COMPLETE-CASE CLEANING

Missing values BEFORE cleaning:
hypertension         0
sbp_final            0
dbp_final            0
age                  0
age_group            0
sex                  0
education_level     13
marital_status       0
residence_type       0
division             0
wealth_index         0
BMI                116
BMI_category       116
glucose            322
diabetes             0
prediabetes          0
household_size       0
sleeping_rooms       0
crowding_index       0
electricity          0
improved_water       0
improved_toilet  

In [ ]:
df = pd.read_csv("/content/BDHS_2022_HYPERTENSION_COMPLETE_CASE.csv")
df.columns

Index(['hypertension', 'sbp_final', 'dbp_final', 'age', 'age_group', 'sex',
       'education_level', 'marital_status', 'residence_type', 'division',
       'wealth_index', 'BMI', 'BMI_category', 'glucose', 'diabetes',
       'prediabetes', 'household_size', 'sleeping_rooms', 'crowding_index',
       'electricity', 'improved_water', 'improved_toilet', 'mobile_phone',
       'sample_weight', 'cluster_id', 'strata_id'],
      dtype='object')

In [ ]:
df['crowding_index'].value_counts()

,count
crowding_index,
2.000000,2656
3.000000,1358
2.500000,1344
1.500000,1200
1.000000,1098
...,...
0.375000,4
0.250000,4
0.285714,4


In [ ]:
# ============================================================
# HOUSEHOLD SIZE CATEGORY
# ============================================================

df['household_size_cat'] = pd.cut(
    df['household_size'],
    bins=[0, 3, 6, np.inf],
    labels=[
        'Small',#(1-3)
        'Medium',# (4-6)
        'Large'# (7+)
    ]
)

# check
print(df['household_size_cat'].value_counts())

household_size_cat
Medium    7666
Small     3082
Large     2651
Name: count, dtype: int64


In [ ]:
# ============================================================
# CROWDING INDEX CATEGORY
# ============================================================

df['crowding_category'] = pd.cut(
    df['crowding_index'],
    bins=[0, 2, 3, np.inf],
    labels=[
        'Low',# (<2)
        'Moderate',# (2-3)
        'High'# (>3)
    ],
    include_lowest=True
)

# check
print(df['crowding_category'].value_counts())

crowding_category
Low         7871
Moderate    3482
High        2046
Name: count, dtype: int64


In [ ]:
# ============================================================
# VALUE COUNTS FOR ALL PREDICTORS
# ============================================================

predictors = [
    'age_group',
    'sex',
    'education_level',
    'marital_status',
    'residence_type',
    'division',
    'wealth_index',
    'BMI_category',
    'diabetes',
    'household_size_cat',
    'crowding_category',
    'electricity',
    'improved_water',
    'improved_toilet',
    'mobile_phone'
]

# ============================================================
# LOOP THROUGH VARIABLES
# ============================================================

for var in predictors:

    print("\n" + "=" * 70)
    print(f"VARIABLE: {var}")
    print("=" * 70)

    # frequency
    freq = df[var].value_counts(dropna=False).sort_index()

    # percentage
    percent = round(df[var].value_counts(normalize=True, dropna=False)
                    .sort_index() * 100, 2)

    # combine
    summary = pd.DataFrame({
        "Count": freq,
        "Percent": percent
    })

    print(summary)


VARIABLE: age_group
           Count  Percent
age_group                
18-29       4023    30.02
30-39       3047    22.74
40-49       2401    17.92
50-59       1774    13.24
60-69       1385    10.34
70+          769     5.74

VARIABLE: sex
     Count  Percent
sex                
1.0   6004    44.81
2.0   7395    55.19

VARIABLE: education_level
                 Count  Percent
education_level                
0.0               3374    25.18
1.0               3394    25.33
2.0               4467    33.34
3.0               2164    16.15

VARIABLE: marital_status
                Count  Percent
marital_status                
0.0              1519    11.34
1.0             10643    79.43
3.0              1025     7.65
4.0               212     1.58

VARIABLE: residence_type
                Count  Percent
residence_type                
1.0              4605    34.37
2.0              8794    65.63

VARIABLE: division
          Count  Percent
division                
1.0        1435    10.71


In [ ]:
# ============================================================
# PUBLICATION-STYLE DESCRIPTIVE TABLE
# ============================================================

all_tables = []

for var in predictors:

    freq = df[var].value_counts(dropna=False)
    percent = round(df[var].value_counts(normalize=True, dropna=False) * 100, 2)

    temp = pd.DataFrame({
        "Variable": var,
        "Category": freq.index.astype(str),
        "Count": freq.values,
        "Percent": percent.values
    })

    all_tables.append(temp)

# combine all
final_table = pd.concat(all_tables, ignore_index=True)

# display
print(final_table)

# save
final_table.to_csv("predictor_value_counts.csv", index=False)

print("\nSaved as predictor_value_counts.csv")

              Variable     Category  Count  Percent
0            age_group        18-29   4023    30.02
1            age_group        30-39   3047    22.74
2            age_group        40-49   2401    17.92
3            age_group        50-59   1774    13.24
4            age_group        60-69   1385    10.34
5            age_group          70+    769     5.74
6                  sex          2.0   7395    55.19
7                  sex          1.0   6004    44.81
8      education_level          2.0   4467    33.34
9      education_level          1.0   3394    25.33
10     education_level          0.0   3374    25.18
11     education_level          3.0   2164    16.15
12      marital_status          1.0  10643    79.43
13      marital_status          0.0   1519    11.34
14      marital_status          3.0   1025     7.65
15      marital_status          4.0    212     1.58
16      residence_type          2.0   8794    65.63
17      residence_type          1.0   4605    34.37
18          

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_spss("/content/BDPR81FL.SAV", convert_categoricals=False)

# Define the columns provided in your Variables.docx
bp_columns = {
    'Men': ['MBP9', 'MBP10', 'MBP13', 'MBP14', 'MBP22', 'MBP23'],
    'Women': ['WBP9', 'WBP10', 'WBP13', 'WBP14', 'WBP22', 'WBP23']
}

print("--- DATA INTEGRITY PRE-CHECK ---")

for gender, cols in bp_columns.items():
    print(f"\nChecking {gender} variables...")
    for col in cols:
        if col in df.columns:
            # Count actual NaNs
            nulls = df[col].isna().sum()
            # Count DHS 'Missing' codes (994, 995, 996, 998, 999)
            invalid_codes = df[(df[col] >= 900) & (df[col] <= 999)][col].count()
            # Range check
            valid_min = df[df[col] < 900][col].min()
            valid_max = df[df[col] < 900][col].max()

            print(f"{col:6} | Missing(NaN): {nulls:<5} | Invalid(900+): {invalid_codes:<5} | Range: {valid_min}-{valid_max}")
        else:
            print(f"ALERT: {col} NOT FOUND IN DATASET")

# Population Check
eligible = df[(df["HV103"] == 1) & (df["HV105"] >= 18)]
print(f"\nEligible De Facto Adults (18+): {len(eligible)}")

--- DATA INTEGRITY PRE-CHECK ---

Checking Men variables...
MBP9   | Missing(NaN): 126064 | Invalid(900+): 2     | Range: 65.0-249.0
MBP10  | Missing(NaN): 126064 | Invalid(900+): 2     | Range: 41.0-144.0
MBP13  | Missing(NaN): 126236 | Invalid(900+): 1     | Range: 62.0-234.0
MBP14  | Missing(NaN): 126236 | Invalid(900+): 1     | Range: 38.0-173.0
MBP22  | Missing(NaN): 126347 | Invalid(900+): 0     | Range: 69.0-228.0
MBP23  | Missing(NaN): 126347 | Invalid(900+): 0     | Range: 36.0-144.0

Checking Women variables...
WBP9   | Missing(NaN): 124563 | Invalid(900+): 1     | Range: 65.0-241.0
WBP10  | Missing(NaN): 124563 | Invalid(900+): 1     | Range: 41.0-166.0
WBP13  | Missing(NaN): 124696 | Invalid(900+): 0     | Range: 60.0-243.0
WBP14  | Missing(NaN): 124696 | Invalid(900+): 0     | Range: 37.0-145.0
WBP22  | Missing(NaN): 124818 | Invalid(900+): 1     | Range: 68.0-238.0
WBP23  | Missing(NaN): 124818 | Invalid(900+): 1     | Range: 44.0-136.0

Eligible De Facto Adults (18+): 81

In [ ]:
import pandas as pd
import numpy as np

# 1. LOAD DATA
df = pd.read_spss("/content/BDPR81FL.SAV", convert_categoricals=False)

# 2. APPLY POPULATION FILTERS
# Inclusion: De Facto residents (HV103=1) and Adults (HV105>=18)
df_study = df[(df["HV103"] == 1) & (df["HV105"] >= 18)].copy()

# 3. BP AVERAGING LOGIC
def get_bdhs_bp_average(row, s_cols, d_cols):
    # Mask invalid codes (900+)
    s_vals = [row[c] if row[c] < 900 else np.nan for c in s_cols]
    d_vals = [row[c] if row[c] < 900 else np.nan for c in d_cols]

    # Hierarchical Logic: Avg(2,3) -> Else 2nd -> Else 1st
    if not np.isnan(s_vals[1]) and not np.isnan(s_vals[2]):
        s_avg, d_avg = (s_vals[1] + s_vals[2]) / 2, (d_vals[1] + d_vals[2]) / 2
    elif not np.isnan(s_vals[1]):
        s_avg, d_avg = s_vals[1], d_vals[1]
    else:
        s_avg, d_avg = s_vals[0], d_vals[0]
    return s_avg, d_avg

# 4. PROCESS BIOMARKERS
df_study["sbp_final"] = np.nan
df_study["dbp_final"] = np.nan
df_study["is_on_bp_meds"] = 0

# Men (MBP Variables)
m_mask = df_study["HV104"] == 1
if m_mask.any():
    m_bp = df_study[m_mask].apply(lambda r: get_bdhs_bp_average(r, ["MBP12", "MBP17", "MBP26"], ["MBP13", "MBP18", "MBP27"]), axis=1, result_type='expand')
    df_study.loc[m_mask, ["sbp_final", "dbp_final"]] = m_bp.values
    df_study.loc[m_mask & (df_study["MBP25"] == 1), "is_on_bp_meds"] = 1

# Women (WBP Variables)
w_mask = df_study["HV104"] == 2
if w_mask.any():
    w_bp = df_study[w_mask].apply(lambda r: get_bdhs_bp_average(r, ["WBP12", "WBP17", "WBP26"], ["WBP13", "WBP18", "WBP27"]), axis=1, result_type='expand')
    df_study.loc[w_mask, ["sbp_final", "dbp_final"]] = w_bp.values
    df_study.loc[w_mask & (df_study["WBP25"] == 1), "is_on_bp_meds"] = 1

# 5. MEASUREMENT-LEVEL EXCLUSION (This results in N=13,848)
df_study = df_study.dropna(subset=["sbp_final", "dbp_final"])

# 6. DEFINE ADDITIONAL OUTCOMES (Diabetes & BMI)
# Diabetes logic
df_study['glucose_raw'] = df_study['SB267'].fillna(df_study['SB367'])
df_study['glucose'] = np.where(df_study['glucose_raw'] >= 900, np.nan, df_study['glucose_raw'])
df_study['is_on_diab_meds'] = np.where((df_study['SB240'] == 1) | (df_study['SB340'] == 1), 1, 0)
df_study['diabetes'] = np.where((df_study['glucose'] >= 126) | (df_study['is_on_diab_meds'] == 1), 1, 0)

# BMI logic
df_study['bmi'] = df_study['HA40'].fillna(df_study['HB40']) / 100

# Hypertension Outcome
df_study["hypertension"] = np.where(
    (df_study["sbp_final"] >= 140) | (df_study["dbp_final"] >= 90) | (df_study["is_on_bp_meds"] == 1), 1, 0
)

# 7. RENAME & SELECT ALL RESEARCH VARIABLES
rename_map = {
    'HV105': 'age',
    'HV104': 'sex',
    'HV106': 'education_level',
    'HV115': 'marital_status',
    'HV025': 'residence_type',
    'HV024': 'division',
    'HV270': 'wealth_index',
    'HV009': 'household_size',
    'HV219': 'head_sex',
    'HV206': 'has_electricity',
    'HV201': 'water_source',
    'HV205': 'toilet_type',
    'HV216': 'sleeping_rooms',
    'HV005': 'sample_weight_raw',
    'HV021': 'cluster_id',
    'HV023': 'strata_id'
}

# Apply renames
df_final = df_study.rename(columns=rename_map)

# Add derived clinical columns
final_columns = list(rename_map.values()) + [
    'hypertension', 'sbp_final', 'dbp_final', 'is_on_bp_meds',
    'diabetes', 'glucose', 'is_on_diab_meds', 'bmi'
]

# Create final weights
df_final['sample_weight'] = df_final['sample_weight_raw'] / 1000000

# 8. SAVE
df_final[final_columns].to_csv("BDHS_2022_Full_Research_Data.csv", index=False)

print(f"Final Study Sample Size: {len(df_final)}")

Final Study Sample Size: 13567


In [ ]:
len(df_final)

13411

In [ ]:
import pandas as pd
import numpy as np

# 1. LOAD DATA
# convert_categoricals=False ensures we get raw numeric codes (e.g., 1, 2) rather than strings
df = pd.read_spss("/content/BDPR81FL.SAV", convert_categoricals=False)

# -------------------------------------------------------------------------
# 2. POPULATION INCLUSION
# -------------------------------------------------------------------------
# HV103 == 1: Stayed in household last night (De Facto)
# HV105 >= 18: Adults only
df_study = df[(df["HV103"] == 1) & (df["HV105"] >= 18)].copy()

# -------------------------------------------------------------------------
# 3. CONSTRUCTING THE TARGET (Hypertension Status)
# -------------------------------------------------------------------------
def get_bdhs_bp_average(row, s_cols, d_cols):
    """Calculates average BP. Codes >= 900 are treated as missing."""
    try:
        s = [row[c] if row[c] < 900 else np.nan for c in s_cols]
        d = [row[c] if row[c] < 900 else np.nan for c in d_cols]
    except KeyError:
        return np.nan, np.nan

    # Priority: Average of 2nd/3rd readings, else 2nd, else 1st.
    if not np.isnan(s[1]) and not np.isnan(s[2]):
        return (s[1] + s[2]) / 2, (d[1] + d[2]) / 2
    elif not np.isnan(s[1]):
        return s[1], d[1]
    elif not np.isnan(s[0]):
        return s[0], d[0]
    return np.nan, np.nan

df_study["sbp_final"] = np.nan
df_study["dbp_final"] = np.nan
df_study["is_on_meds"] = 0

# Process both genders into the same columns
for gender, prefix in [(1, "M"), (2, "W")]:
    mask = df_study["HV104"] == gender
    if mask.any():
        bp_results = df_study[mask].apply(
            lambda r: get_bdhs_bp_average(r, [f"{prefix}BP9", f"{prefix}BP13", f"{prefix}BP22"],
                                             [f"{prefix}BP10", f"{prefix}BP14", f"{prefix}BP23"]),
            axis=1, result_type='expand'
        )
        df_study.loc[mask, ["sbp_final", "dbp_final"]] = bp_results.values
        df_study.loc[mask & (df_study[f"{prefix}BP19"] == 1), "is_on_meds"] = 1

# Define Target: SBP >= 140 OR DBP >= 90 OR taking medication
df_study = df_study.dropna(subset=["sbp_final", "dbp_final"])
df_study["hypertension"] = np.where(
    (df_study["sbp_final"] >= 140) | (df_study["dbp_final"] >= 90) | (df_study["is_on_meds"] == 1), 1, 0
)

# -------------------------------------------------------------------------
# 4. PREDICTOR CLEANING (Harmonizing Men & Women)
# -------------------------------------------------------------------------
# BMI: Combining HB40 and HA40. 9998/9999 are missing/out-of-range.
df_study['BMI'] = np.where(df_study['HV104'] == 1, df_study['HB40'], df_study['HA40'])
df_study['BMI'] = np.where(df_study['BMI'] < 9000, df_study['BMI'] / 100, np.nan)

# Glucose: Combining SB367 and SB267. 998/999 are missing.
df_study['glucose'] = df_study['SB367'].fillna(df_study['SB267'])
df_study['glucose'] = np.where(df_study['glucose'] < 900, df_study['glucose'], np.nan)

# -------------------------------------------------------------------------
# 5. RENAMING & FINAL EXPORT
# -------------------------------------------------------------------------
rename_map = {
    'HV105': 'age',              # Universal age for both genders
    'HV104': 'sex',              # 1=Male, 2=Female
    'HV106': 'education_level',
    'HV115': 'marital_status',
    'HV025': 'residence_type',   # Urban/Rural
    'HV024': 'division',         # Geographic Division
    'HV270': 'wealth_index',     # Quintile 1-5
    'HV009': 'household_size',
    'HV219': 'head_sex',
    'HV206': 'has_electricity',
    'HV201': 'water_source',
    'HV205': 'toilet_type',
    'HV216': 'sleeping_rooms',
    'HV005': 'sample_weight_raw',
    'HV021': 'cluster_id',
    'HV023': 'strata_id'
}

# Final feature selection
final_cols = ['hypertension', 'BMI', 'glucose'] + list(rename_map.values())
df_final = df_study.rename(columns=rename_map)[final_cols].dropna()

# Save
df_final.to_csv("BDHS_2022_Hypertension_Study_Final.csv", index=False)

print(f"Dataset generated. Final N = {len(df_final)}")

Dataset generated. Final N = 13411


In [ ]:
import pandas as pd
import numpy as np

def build_final_bdhs_dataset(df):
    df_clean = df.copy()

    # --- Step 1: Handle Missing Values (994-999) ---
    biomarker_cols = [
        'WBP7', 'WBP8', 'WBP18', 'WBP19', 'WBP26', 'WBP27',
        'MBP7', 'MBP8', 'MBP18', 'MBP19', 'MBP26', 'MBP27',
        'SB267', 'SB367', 'HA40', 'HB40'
    ]
    for col in biomarker_cols:
        if col in df_clean.columns:
            df_clean[col] = np.where(df_clean[col] >= 900, np.nan, df_clean[col])

    # --- Step 2: Blood Pressure Averaging ---
    def get_avg_bp(row, prefix):
        s1, d1 = row[f'{prefix}7'], row[f'{prefix}8']
        s2, d2 = row[f'{prefix}18'], row[f'{prefix}19']
        s3, d3 = row[f'{prefix}26'], row[f'{prefix}27']

        # DHS Rule: Avg of 2nd & 3rd. If 3rd missing, use 2nd. If both missing, use 1st.
        if pd.notnull(s2) and pd.notnull(s3):
            sbp, dbp = (s2 + s3)/2, (d2 + d3)/2
        elif pd.notnull(s2):
            sbp, dbp = s2, d2
        else:
            sbp, dbp = s1, d1
        return sbp, dbp

    # Apply to Women (HV104=2) and Men (HV104=1)
    df_clean[['sbp', 'dbp']] = np.nan
    mask_w = df_clean['HV104'] == 2
    mask_m = df_clean['HV104'] == 1

    df_clean.loc[mask_w, ['sbp', 'dbp']] = df_clean[mask_w].apply(lambda r: get_avg_bp(r, 'WBP'), axis=1, result_type='expand').values
    df_clean.loc[mask_m, ['sbp', 'dbp']] = df_clean[mask_m].apply(lambda r: get_avg_bp(r, 'MBP'), axis=1, result_type='expand').values

    # --- Step 3: Target Definitions ---
    # Hypertension Target
    df_clean['is_on_bp_meds'] = np.where((df_clean['WBP25'] == 1) | (df_clean['MBP25'] == 1), 1, 0)
    df_clean['hypertension'] = np.where(
        (df_clean['sbp'] >= 140) | (df_clean['dbp'] >= 90) | (df_clean['is_on_bp_meds'] == 1), 1, 0
    )

    # Diabetes Target
    df_clean['glucose'] = df_clean['SB267'].fillna(df_clean['SB367'])
    df_clean['is_on_diab_meds'] = np.where((df_clean['SB240'] == 1) | (df_clean['SB340'] == 1), 1, 0)
    df_clean['diabetes'] = np.where(
        (df_clean['glucose'] >= 126) | (df_clean['is_on_diab_meds'] == 1), 1, 0
    )

    # BMI calculation (scaling from DHS format)
    df_clean['bmi'] = df_clean['HA40'].fillna(df_clean['HB40']) / 100

    # --- Step 4: Map Final Feature Set ---
    rename_map = {
        'HV105': 'age',
        'HV104': 'sex',
        'HV106': 'education_level',
        'HV115': 'marital_status',
        'HV025': 'residence_type',
        'HV024': 'division',
        'HV270': 'wealth_index',
        'HV009': 'household_size',
        'HV219': 'head_sex',
        'HV206': 'has_electricity',
        'HV201': 'water_source',
        'HV205': 'toilet_type',
        'HV216': 'sleeping_rooms',
        'HV005': 'sample_weight',
        'HV021': 'cluster_id',
        'HV023': 'strata_id'
    }

    final_features = list(rename_map.values()) + ['bmi', 'sbp', 'dbp', 'glucose', 'hypertension', 'diabetes']
    df_final = df_clean.rename(columns=rename_map)[final_features]

    # Scale Sample Weight
    df_final['sample_weight'] = df_final['sample_weight'] / 1000000

    return df_final.dropna(subset=['hypertension'])

# --- Step 5: Save the Data ---
# processed_df = build_final_bdhs_dataset(raw_df)
# processed_df.to_csv("BDHS_2022_ML_Ready.csv", index=False)

In [ ]:
len(df_final)

13411

In [ ]:
import pandas as pd
import numpy as np

# 1. LOAD DATA
df = pd.read_spss("/content/BDPR81FL.SAV", convert_categoricals=False)

# -------------------------------------------------------------------------
# 2. APPLY POPULATION FILTERS (INCLUSION/EXCLUSION)
# -------------------------------------------------------------------------
# Inclusion: De Facto residents (HV103=1) and Adults (HV105>=18)
df_study = df[(df["HV103"] == 1) & (df["HV105"] >= 18)].copy()

# -------------------------------------------------------------------------
# 3. DEFINE MEASUREMENT LOGIC FUNCTION
# -------------------------------------------------------------------------
def get_bdhs_bp_average(row, s_cols, d_cols):
    """
    Exclusion: Treat codes 994-999 (Refused/Missing) as NaN.
    Hierarchical Logic:
    - Average of 2nd and 3rd readings.
    - Fallback to 2nd if 3rd is missing.
    - Fallback to 1st if 2nd/3rd are missing.
    """
    # Filter out invalid measurement codes (900+)
    s_vals = [row[c] if row[c] < 900 else np.nan for c in s_cols]
    d_vals = [row[c] if row[c] < 900 else np.nan for c in d_cols]

    # Systolic Logic
    if not np.isnan(s_vals[1]) and not np.isnan(s_vals[2]):
        s_avg = (s_vals[1] + s_vals[2]) / 2
    elif not np.isnan(s_vals[1]):
        s_avg = s_vals[1]
    else:
        s_avg = s_vals[0]

    # Diastolic Logic
    if not np.isnan(d_vals[1]) and not np.isnan(d_vals[2]):
        d_avg = (d_vals[1] + d_vals[2]) / 2
    elif not np.isnan(d_vals[1]):
        d_avg = d_vals[1]
    else:
        d_avg = d_vals[0]

    return s_avg, d_avg

# -------------------------------------------------------------------------
# 4. DATA PROCESSING
# -------------------------------------------------------------------------
df_study["sbp_final"] = np.nan
df_study["dbp_final"] = np.nan
df_study["is_on_meds"] = 0

# Process Men (MBP variables)
m_mask = df_study["HV104"] == 1
if m_mask.any():
    m_bp = df_study[m_mask].apply(lambda r: get_bdhs_bp_average(r, ["MBP9", "MBP13", "MBP22"], ["MBP10", "MBP14", "MBP23"]), axis=1, result_type='expand')
    df_study.loc[m_mask, ["sbp_final", "dbp_final"]] = m_bp.values
    df_study.loc[m_mask & (df_study["MBP19"] == 1), "is_on_meds"] = 1

# Process Women (WBP variables)
w_mask = df_study["HV104"] == 2
if w_mask.any():
    w_bp = df_study[w_mask].apply(lambda r: get_bdhs_bp_average(r, ["WBP9", "WBP13", "WBP22"], ["WBP10", "WBP14", "WBP23"]), axis=1, result_type='expand')
    df_study.loc[w_mask, ["sbp_final", "dbp_final"]] = w_bp.values
    df_study.loc[w_mask & (df_study["WBP19"] == 1), "is_on_meds"] = 1

# -------------------------------------------------------------------------
# 5. MEASUREMENT-LEVEL EXCLUSION
# -------------------------------------------------------------------------
# Remove respondents who did not have at least one valid BP reading
df_study = df_study.dropna(subset=["sbp_final", "dbp_final"])

# -------------------------------------------------------------------------
# 6. DEFINE OUTCOMES & WEIGHTS
# -------------------------------------------------------------------------
# Hypertension Definition
df_study["hypertension"] = np.where(
    (df_study["sbp_final"] >= 140) | (df_study["dbp_final"] >= 90) | (df_study["is_on_meds"] == 1), 1, 0
)

# Sample Weight
df_study["sample_weight"] = df_study["HV005"] / 1000000

# -------------------------------------------------------------------------
# 7. EXPORT CLEAN DATASET
# -------------------------------------------------------------------------
final_cols = [
    'hypertension', 'sbp_final', 'dbp_final', 'is_on_meds', 'sample_weight',
    'HV021', 'HV023', 'HV024', 'HV025', 'HV104', 'HV105', 'HV106', 'HV270', 'SB367', 'SB267'
]
df_study[final_cols].to_csv("BDHS_Final_Hypertension_Data.csv", index=False)

print(f"Final Study Sample Size: {len(df_study)}")
print("CSV File Saved: BDHS_Final_Hypertension_Data.csv")

Final Study Sample Size: 13848
CSV File Saved: BDHS_Final_Hypertension_Data.csv


In [ ]:
import pandas as pd
import numpy as np

# 1. LOAD DATA
df = pd.read_spss("/content/BDPR81FL.SAV", convert_categoricals=False)

# -------------------------------------------------------------------------
# 2. POPULATION FILTERS (Inclusion & Exclusion Stage 1)
# -------------------------------------------------------------------------
# Inclusion: De Facto residents (HV103=1) and Adults (HV105>=18)
df_study = df[(df["HV103"] == 1) & (df["HV105"] >= 18)].copy()

# -------------------------------------------------------------------------
# 3. MEASUREMENT LOGIC (Inclusion & Exclusion Stage 2)
# -------------------------------------------------------------------------
def get_bdhs_bp_average(row, s_cols, d_cols):
    # Fix the NameError: use 'c' consistently as the iterator variable
    try:
        s = [row[c] if row[c] < 900 else np.nan for c in s_cols]
        d = [row[c] if row[c] < 900 else np.nan for c in d_cols] # Fixed: changed 'for d' to 'for c'
    except KeyError:
        return np.nan, np.nan

    # Hierarchical Inclusion: 2nd+3rd > 2nd > 1st
    if not np.isnan(s[1]) and not np.isnan(s[2]):
        return (s[1] + s[2]) / 2, (d[1] + d[2]) / 2
    elif not np.isnan(s[1]):
        return s[1], d[1]
    elif not np.isnan(s[0]):
        return s[0], d[0]
    else:
        return np.nan, np.nan

# --- Apply BP Calculations ---
df_study["sbp_final"] = np.nan
df_study["dbp_final"] = np.nan
df_study["is_on_meds"] = 0

# Men (MBP)
m_mask = df_study["HV104"] == 1
if m_mask.any():
    m_bp = df_study[m_mask].apply(lambda r: get_bdhs_bp_average(r, ["MBP9", "MBP13", "MBP22"], ["MBP10", "MBP14", "MBP23"]), axis=1, result_type='expand')
    df_study.loc[m_mask, ["sbp_final", "dbp_final"]] = m_bp.values
    df_study.loc[m_mask & (df_study["MBP19"] == 1), "is_on_meds"] = 1

# Women (WBP)
w_mask = df_study["HV104"] == 2
if w_mask.any():
    w_bp = df_study[w_mask].apply(lambda r: get_bdhs_bp_average(r, ["WBP9", "WBP13", "WBP22"], ["WBP10", "WBP14", "WBP23"]), axis=1, result_type='expand')
    df_study.loc[w_mask, ["sbp_final", "dbp_final"]] = w_bp.values
    df_study.loc[w_mask & (df_study["WBP19"] == 1), "is_on_meds"] = 1

# -------------------------------------------------------------------------
# 4. DERIVED VARIABLES (Predictor Harmonization)
# -------------------------------------------------------------------------
# BMI Calculation & Outlier Exclusion (DHS uses implied 2 decimal places)
df_study['bmi_val'] = np.where(df_study['HV104'] == 1, df_study['HB40'], df_study['HA40'])
df_study['BMI'] = np.where(df_study['bmi_val'] < 9000, df_study['bmi_val'] / 100, np.nan)

# Unified Tobacco Use (1=Yes, 0=No)
# Checks for smoking or smokeless tobacco use across all gender-specific columns
tobacco_cols = ['SM36A', 'SM36B', 'SH36A', 'SH36B']
for col in tobacco_cols:
    if col not in df_study.columns: df_study[col] = 0 # Safety check

df_study['smoker'] = np.where(
    (df_study['SM36A'] == 1) | (df_study['SM36B'] == 1) |
    (df_study['SH36A'] == 1) | (df_study['SH36B'] == 1), 1, 0
)

# Glucose Harmonization
df_study['glucose'] = df_study['SB367'].fillna(df_study['SB267'])
df_study.loc[df_study['glucose'] >= 900, 'glucose'] = np.nan

# -------------------------------------------------------------------------
# 5. FINAL EXCLUSION & HYPERTENSION DEF
# -------------------------------------------------------------------------
# Exclusion: Final measurement check (Must have SBP/DBP)
df_study = df_study.dropna(subset=["sbp_final", "dbp_final"])

# Outcome Definition
df_study["hypertension"] = np.where(
    (df_study["sbp_final"] >= 140) | (df_study["dbp_final"] >= 90) | (df_study["is_on_meds"] == 1), 1, 0
)

# Sample Weighting
df_study["sample_weight"] = df_study["HV005"] / 1000000

# --------------------------------------------------
# 6. EXPORT
# --------------------------------------------------
final_predictors = [
    'hypertension', 'HV104', 'HV105', 'HV106', 'HV115', 'HV025', 'HV024',
    'HV270', 'HV009', 'HV219',  'HV201', 'HV205', 'HV206',
    'HV226', 'HV216', 'HV271', 'HV208', 'HV243A', 'HV243C', 'HV243E',
    'BMI', 'glucose', 'smoker', 'sample_weight', 'HV021', 'HV023'
]

df_study[final_predictors].to_csv("BDHS_Hypertension_ML_Final.csv", index=False)

print(f"Success! Final sample size for ML: {len(df_study)}")

Success! Final sample size for ML: 13848


In [ ]:
df = pd.read_csv("/content/BDHS_Hypertension_ML_Final.csv")
df.columns

Index(['hypertension', 'HV104', 'HV105', 'HV106', 'HV115', 'HV025', 'HV024',
       'HV270', 'HV009', 'HV219', 'HV201', 'HV205', 'HV206', 'HV226', 'HV216',
       'HV271', 'HV208', 'HV243A', 'HV243C', 'HV243E', 'BMI', 'glucose',
       'smoker', 'sample_weight', 'HV021', 'HV023'],
      dtype='object')

In [ ]:
df.isnull().sum()

,0
hypertension,0
HV104,0
HV105,0
HV106,0
HV115,0
HV025,0
HV024,0
HV270,0
HV009,0
HV219,0


In [ ]:
import pandas as pd
import numpy as np

# 1. LOAD DATA
df = pd.read_spss("/content/BDPR81FL.SAV", convert_categoricals=False)

# -------------------------------------------------------------------------
# 2. POPULATION FILTERS (Inclusion & Exclusion Stage 1)
# -------------------------------------------------------------------------
# Inclusion: De Facto residents (HV103=1) and Adults (HV105>=18)
df_study = df[(df["HV103"] == 1) & (df["HV105"] >= 18)].copy()

# -------------------------------------------------------------------------
# 3. MEASUREMENT LOGIC (Inclusion & Exclusion Stage 2)
# -------------------------------------------------------------------------
def get_bdhs_bp_average(row, s_cols, d_cols):
    try:
        s = [row[c] if row[c] < 900 else np.nan for c in s_cols]
        d = [row[c] if row[c] < 900 else np.nan for c in d_cols]
    except KeyError:
        return np.nan, np.nan

    # Hierarchical Inclusion: 2nd+3rd > 2nd > 1st
    if not np.isnan(s[1]) and not np.isnan(s[2]):
        return (s[1] + s[2]) / 2, (d[1] + d[2]) / 2
    elif not np.isnan(s[1]):
        return s[1], d[1]
    elif not np.isnan(s[0]):
        return s[0], d[0]
    else:
        return np.nan, np.nan

# --- Apply BP Calculations ---
df_study["sbp_final"] = np.nan
df_study["dbp_final"] = np.nan
df_study["is_on_meds"] = 0

# Men (MBP)
m_mask = df_study["HV104"] == 1
if m_mask.any():
    m_bp = df_study[m_mask].apply(lambda r: get_bdhs_bp_average(r, ["MBP9", "MBP13", "MBP22"], ["MBP10", "MBP14", "MBP23"]), axis=1, result_type='expand')
    df_study.loc[m_mask, ["sbp_final", "dbp_final"]] = m_bp.values
    df_study.loc[m_mask & (df_study["MBP19"] == 1), "is_on_meds"] = 1

# Women (WBP)
w_mask = df_study["HV104"] == 2
if w_mask.any():
    w_bp = df_study[w_mask].apply(lambda r: get_bdhs_bp_average(r, ["WBP9", "WBP13", "WBP22"], ["WBP10", "WBP14", "WBP23"]), axis=1, result_type='expand')
    df_study.loc[w_mask, ["sbp_final", "dbp_final"]] = w_bp.values
    df_study.loc[w_mask & (df_study["WBP19"] == 1), "is_on_meds"] = 1

# -------------------------------------------------------------------------
# 4. DERIVED VARIABLES
# -------------------------------------------------------------------------
# BMI & Glucose (keeping NaNs to preserve the 13,848 count)
df_study['bmi_val'] = np.where(df_study['HV104'] == 1, df_study['HB40'], df_study['HA40'])
df_study['BMI'] = np.where(df_study['bmi_val'] < 9000, df_study['bmi_val'] / 100, np.nan)

df_study['glucose'] = df_study['SB367'].fillna(df_study['SB267'])
df_study.loc[df_study['glucose'] >= 900, 'glucose'] = np.nan

# Tobacco Use
tobacco_cols = ['SM36A', 'SM36B', 'SH36A', 'SH36B']
for col in tobacco_cols:
    if col not in df_study.columns: df_study[col] = 0
df_study['smoker'] = np.where(
    (df_study['SM36A'] == 1) | (df_study['SM36B'] == 1) |
    (df_study['SH36A'] == 1) | (df_study['SH36B'] == 1), 1, 0
)

# -------------------------------------------------------------------------
# 5. FINAL EXCLUSION & HYPERTENSION DEF
# -------------------------------------------------------------------------
# Filter for those with valid BP readings (Results in N=13,848)
df_study = df_study.dropna(subset=["sbp_final", "dbp_final"])

df_study["hypertension"] = np.where(
    (df_study["sbp_final"] >= 140) | (df_study["dbp_final"] >= 90) | (df_study["is_on_meds"] == 1), 1, 0
)

# --------------------------------------------------
# 6. RENAME AND EXPORT
# --------------------------------------------------
rename_map = {
    'HV105': 'age',
    'HV104': 'sex',
    'HV106': 'education_level',
    'HV115': 'marital_status',
    'HV025': 'residence_type',
    'HV024': 'division',
    'HV270': 'wealth_index',
    'HV009': 'household_size',
    'HV219': 'head_sex',
    'HV206': 'has_electricity',
    'HV201': 'water_source',
    'HV205': 'toilet_type',
    'HV216': 'sleeping_rooms',
    'HV005': 'sample_weight',
    'HV021': 'cluster_id',
    'HV023': 'strata_id'
}

# Select original columns + derived columns
derived_cols = ['hypertension', 'sbp_final', 'dbp_final', 'BMI', 'glucose', 'smoker']
cols_to_keep = list(rename_map.keys()) + derived_cols

# Create final dataframe with renamed columns
df_final = df_study[cols_to_keep].rename(columns=rename_map)

# Adjust sample weight
df_final['sample_weight'] = df_final['sample_weight'] / 1000000

# Save to CSV
df_final.to_csv("BDHS_Hypertension_Final_13848.csv", index=False)

print(f"Success! Final sample size: {len(df_final)}")

Success! Final sample size: 13848


In [ ]:
df=pd.read_csv("/content/BDHS_Hypertension_Final_13848.csv")
df.columns

Index(['age', 'sex', 'education_level', 'marital_status', 'residence_type',
       'division', 'wealth_index', 'household_size', 'head_sex',
       'has_electricity', 'water_source', 'toilet_type', 'sleeping_rooms',
       'sample_weight', 'cluster_id', 'strata_id', 'hypertension', 'sbp_final',
       'dbp_final', 'BMI', 'glucose', 'smoker'],
      dtype='object')

In [ ]:
df['hypertension'].value_counts()

,count
hypertension,
0,10971
1,2877


In [ ]:
# Loop through all columns and print the value counts
for col in df.columns:
    print(f"--- Value Counts for: {col} ---")
    print(df[col].value_counts(dropna=False))
    print("\n")

--- Value Counts for: age ---
age
35.0    553
40.0    529
45.0    463
18.0    461
30.0    449
       ... 
84.0      3
88.0      2
94.0      2
91.0      2
92.0      2
Name: count, Length: 76, dtype: int64


--- Value Counts for: sex ---
sex
2.0    7656
1.0    6192
Name: count, dtype: int64


--- Value Counts for: education_level ---
education_level
2.0    4580
0.0    3522
1.0    3493
3.0    2240
8.0      13
Name: count, dtype: int64


--- Value Counts for: marital_status ---
marital_status
1.0    10955
0.0     1563
3.0     1111
4.0      219
Name: count, dtype: int64


--- Value Counts for: residence_type ---
residence_type
2.0    9042
1.0    4806
Name: count, dtype: int64


--- Value Counts for: division ---
division
2.0    1938
3.0    1883
6.0    1797
4.0    1787
7.0    1716
8.0    1680
5.0    1577
1.0    1470
Name: count, dtype: int64


--- Value Counts for: wealth_index ---
wealth_index
5.0    3127
4.0    2903
2.0    2683
3.0    2651
1.0    2484
Name: count, dtype: int64


--- Value 

In [ ]:
# ============================================================
# BDHS 2022 HYPERTENSION DATASET CREATION
# USING ONLY PR FILE (BDPR81FL)
# Publication-Quality Clean Pipeline
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD PR FILE
# ============================================================

df = pd.read_spss(
    "/content/BDPR81FL.SAV",
    convert_categoricals=False
)

print("Initial shape:", df.shape)

# ============================================================
# 2. ELIGIBLE POPULATION
# ============================================================

# HV103 = de facto resident
# HV105 = age

df = df[
    (df["HV103"] == 1) &
    (df["HV105"] >= 18)
].copy()

print("Eligible adults:", len(df))

# ============================================================
# 3. BLOOD PRESSURE FUNCTION
# ============================================================

def get_bp_average(row, s_cols, d_cols):

    s = []
    d = []

    # systolic
    for c in s_cols:
        val = row.get(c, np.nan)

        if pd.notna(val) and val < 900:
            s.append(val)
        else:
            s.append(np.nan)

    # diastolic
    for c in d_cols:
        val = row.get(c, np.nan)

        if pd.notna(val) and val < 900:
            d.append(val)
        else:
            d.append(np.nan)

    # DHS recommendation:
    # average 2nd and 3rd reading

    if not np.isnan(s[1]) and not np.isnan(s[2]):

        sbp = (s[1] + s[2]) / 2
        dbp = (d[1] + d[2]) / 2

    elif not np.isnan(s[1]):

        sbp = s[1]
        dbp = d[1]

    elif not np.isnan(s[0]):

        sbp = s[0]
        dbp = d[0]

    else:

        sbp = np.nan
        dbp = np.nan

    return pd.Series([sbp, dbp])

# ============================================================
# 4. CREATE BP VARIABLES
# ============================================================

df["sbp_final"] = np.nan
df["dbp_final"] = np.nan
df["is_on_meds"] = 0

# ------------------------------------------------------------
# MEN
# ------------------------------------------------------------

men = df["HV104"] == 1

if men.any():

    bp_men = df[men].apply(
        lambda r: get_bp_average(
            r,
            ["MBP9", "MBP13", "MBP22"],
            ["MBP10", "MBP14", "MBP23"]
        ),
        axis=1
    )

    df.loc[men, ["sbp_final", "dbp_final"]] = bp_men.values

    # medication
    df.loc[
        men & (df["MBP19"] == 1),
        "is_on_meds"
    ] = 1

# ------------------------------------------------------------
# WOMEN
# ------------------------------------------------------------

women = df["HV104"] == 2

if women.any():

    bp_women = df[women].apply(
        lambda r: get_bp_average(
            r,
            ["WBP9", "WBP13", "WBP22"],
            ["WBP10", "WBP14", "WBP23"]
        ),
        axis=1
    )

    df.loc[women, ["sbp_final", "dbp_final"]] = bp_women.values

    # medication
    df.loc[
        women & (df["WBP19"] == 1),
        "is_on_meds"
    ] = 1

# ============================================================
# 5. KEEP ONLY VALID BP PARTICIPANTS
# ============================================================

df = df.dropna(
    subset=["sbp_final", "dbp_final"]
).copy()

print("Final BP sample:", len(df))

# ============================================================
# 6. HYPERTENSION DEFINITION
# ============================================================

df["hypertension"] = np.where(
    (df["sbp_final"] >= 140) |
    (df["dbp_final"] >= 90) |
    (df["is_on_meds"] == 1),
    1,
    0
)

# ============================================================
# 7. BMI
# ============================================================

# Men = HB40
# Women = HA40

df["bmi_raw"] = np.where(
    df["HV104"] == 1,
    df["HB40"],
    df["HA40"]
)

df["BMI"] = np.where(
    df["bmi_raw"] < 9000,
    df["bmi_raw"] / 100,
    np.nan
)

# BMI category

df["BMI_category"] = pd.cut(
    df["BMI"],
    bins=[0, 18.5, 25, 30, 100],
    labels=[
        "Underweight",
        "Normal",
        "Overweight",
        "Obese"
    ]
)

# ============================================================
# 8. GLUCOSE / DIABETES
# ============================================================

# Women = SB367
# Men = SB267

df["glucose"] = df["SB367"].fillna(df["SB267"])

# remove invalid values
df.loc[
    df["glucose"] >= 900,
    "glucose"
] = np.nan

# diabetes
df["diabetes"] = np.where(
    df["glucose"] >= 126,
    1,
    0
)

# prediabetes
df["prediabetes"] = np.where(
    (df["glucose"] >= 100) &
    (df["glucose"] < 126),
    1,
    0
)

# ============================================================
# 9. AGE GROUPS
# ============================================================

df["age_group"] = pd.cut(
    df["HV105"],
    bins=[18, 29, 39, 49, 59, 69, 120],
    labels=[
        "18-29",
        "30-39",
        "40-49",
        "50-59",
        "60-69",
        "70+"
    ]
)

# ============================================================
# 10. HOUSEHOLD CROWDING
# ============================================================

df["crowding_index"] = (
    df["HV009"] / df["HV216"]
)

# ============================================================
# 11. IMPROVED WATER
# ============================================================

improved_water_codes = [
    11, 12, 13, 14, 21, 31, 41
]

df["improved_water"] = np.where(
    df["HV201"].isin(improved_water_codes),
    1,
    0
)

# ============================================================
# 12. IMPROVED TOILET
# ============================================================

improved_toilet_codes = [
    11, 12, 13, 21, 22
]

df["improved_toilet"] = np.where(
    df["HV205"].isin(improved_toilet_codes),
    1,
    0
)

# ============================================================
# 13. CLEAN COOKING FUEL
# ============================================================

clean_fuel_codes = [
    1, 2, 3, 4, 5
]

df["clean_cooking_fuel"] = np.where(
    df["HV226"].isin(clean_fuel_codes),
    1,
    0
)

# ============================================================
# 14. ELECTRICITY
# ============================================================

df["electricity"] = df["HV206"]

# ============================================================
# 15. MOBILE PHONE
# ============================================================

if "HV243A" in df.columns:

    df["mobile_phone"] = df["HV243A"]

else:

    df["mobile_phone"] = np.nan

# ============================================================
# 16. INTERNET USE
# ============================================================

if "HV243B" in df.columns:

    df["internet_use"] = df["HV243B"]

else:

    df["internet_use"] = np.nan

# ============================================================
# 17. FLOOR MATERIAL
# ============================================================

df["floor_material"] = df["HV213"]

# ============================================================
# 18. WALL MATERIAL
# ============================================================

df["wall_material"] = df["HV214"]

# ============================================================
# 19. ROOF MATERIAL
# ============================================================

df["roof_material"] = df["HV215"]

# ============================================================
# 20. EDUCATION CLEANING
# ============================================================

# remove "don't know"

df["HV106"] = df["HV106"].replace(8, np.nan)

# ============================================================
# 21. FINAL VARIABLE LIST
# ============================================================

rename_dict = {

    # outcome
    "hypertension": "hypertension",

    # BP
    "sbp_final": "sbp_final",
    "dbp_final": "dbp_final",

    # demographic
    "HV105": "age",
    "age_group": "age_group",
    "HV104": "sex",
    "HV106": "education_level",
    "HV115": "marital_status",
    "HV025": "residence_type",
    "HV024": "division",
    "HV270": "wealth_index",


    # anthropometric
    "BMI": "BMI",
    "BMI_category": "BMI_category",

    # metabolic
    "glucose": "glucose",
    "diabetes": "diabetes",
    "prediabetes": "prediabetes",

    # household
    "HV009": "household_size",
    "HV216": "sleeping_rooms",
    "crowding_index": "crowding_index",
    "electricity": "electricity",
    "improved_water": "improved_water",
    "improved_toilet": "improved_toilet",
    "clean_cooking_fuel": "clean_cooking_fuel",

    # housing quality
    "floor_material": "floor_material",
    "wall_material": "wall_material",
    "roof_material": "roof_material",

    # technology
    "mobile_phone": "mobile_phone",
    "internet_use": "internet_use",

    # survey design
    "HV005": "sample_weight",
    "HV021": "cluster_id",
    "HV023": "strata_id"
}

# ============================================================
# 22. CREATE FINAL DATASET
# ============================================================

df_final = df[
    list(rename_dict.keys())
].rename(columns=rename_dict)

# ============================================================
# 23. SURVEY WEIGHT ADJUSTMENT
# ============================================================

df_final["sample_weight"] = (
    df_final["sample_weight"] / 1000000
)

# ============================================================
# 24. REMOVE DUPLICATES
# ============================================================

df_final = df_final.drop_duplicates()

# ============================================================
# 25. EXPORT FINAL DATASET
# ============================================================

df_final.to_csv(
    "BDHS_2022_Hypertension_PR_Only_Final.csv",
    index=False
)

# ============================================================
# 26. FINAL CHECKS
# ============================================================

print("\n================ FINAL DATASET =================")

print("Final shape:", df_final.shape)

print("\nHypertension prevalence:")
print(df_final["hypertension"].value_counts())

print("\nSex distribution:")
print(df_final["sex"].value_counts())

print("\nEducation:")
print(df_final["education_level"].value_counts())

print("\nResidence:")
print(df_final["residence_type"].value_counts())

print("\nWealth:")
print(df_final["wealth_index"].value_counts())

print("\nDiabetes:")
print(df_final["diabetes"].value_counts(dropna=False))

print("\nBMI missing:")
print(df_final["BMI"].isna().sum())

print("\nGlucose missing:")
print(df_final["glucose"].isna().sum())

print("\nDataset successfully exported!")

# ============================================================
# 27. OPTIONAL: VARIABLE LABEL TABLE
# ============================================================

variable_description = pd.DataFrame({

    "Variable": df_final.columns,

    "Description": [

        "Hypertension outcome",
        "Final systolic blood pressure",
        "Final diastolic blood pressure",

        "Age in years",
        "Age category",
        "Sex",
        "Education level",
        "Marital status",
        "Urban/rural residence",
        "Administrative division",
        "Wealth index",
        "Religion",

        "Body mass index",
        "BMI category",

        "Blood glucose",
        "Diabetes status",
        "Prediabetes status",

        "Household size",
        "Sleeping rooms",
        "Crowding index",
        "Electricity availability",
        "Improved drinking water",
        "Improved sanitation",
        "Clean cooking fuel",

        "Floor material",
        "Wall material",
        "Roof material",

        "Mobile phone ownership",
        "Internet use",

        "Survey sample weight",
        "Cluster/PSU",
        "Sampling strata"
    ]
})

variable_description.to_csv(
    "BDHS_2022_Variable_Descriptions.csv",
    index=False
)

print("\nVariable description file exported!")

Initial shape: (132463, 549)
Eligible adults: 81881
Final BP sample: 13848

================ FINAL DATASET =================
Final shape: (13848, 31)

Hypertension prevalence:
hypertension
0    10971
1     2877
Name: count, dtype: int64

Sex distribution:
sex
2.0    7656
1.0    6192
Name: count, dtype: int64

Education:
education_level
2.0    4580
0.0    3522
1.0    3493
3.0    2240
Name: count, dtype: int64

Residence:
residence_type
2.0    9042
1.0    4806
Name: count, dtype: int64

Wealth:
wealth_index
5.0    3127
4.0    2903
2.0    2683
3.0    2651
1.0    2484
Name: count, dtype: int64

Diabetes:
diabetes
0    12794
1     1054
Name: count, dtype: int64

BMI missing:
116

Glucose missing:
322

Dataset successfully exported!


ValueError: All arrays must be of the same length

In [ ]:
# ============================================================
# BDHS 2022 HYPERTENSION DATASET
# PR FILE ONLY
# FINAL CLEAN PUBLICATION-QUALITY PIPELINE
# WITH COMPLETE MISSING VALUE HANDLING
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_spss(
    "/content/BDPR81FL.SAV",
    convert_categoricals=False
)

print("=" * 60)
print("INITIAL DATA")
print("=" * 60)
print("Initial shape:", df.shape)

# ============================================================
# 2. ELIGIBILITY CRITERIA
# ============================================================

# De facto residents
df = df[df["HV103"] == 1].copy()

print("\nAfter de facto resident filter:", len(df))

# Adults ≥18 years
df = df[df["HV105"] >= 18].copy()

print("After adult filter:", len(df))

# ============================================================
# 3. BLOOD PRESSURE FUNCTION
# ============================================================

def get_bp_average(row, s_cols, d_cols):

    systolic = []
    diastolic = []

    for c in s_cols:

        val = row.get(c, np.nan)

        if pd.notna(val) and val < 900:
            systolic.append(val)
        else:
            systolic.append(np.nan)

    for c in d_cols:

        val = row.get(c, np.nan)

        if pd.notna(val) and val < 900:
            diastolic.append(val)
        else:
            diastolic.append(np.nan)

    # DHS recommendation:
    # average 2nd + 3rd reading

    if not np.isnan(systolic[1]) and not np.isnan(systolic[2]):

        sbp = (systolic[1] + systolic[2]) / 2
        dbp = (diastolic[1] + diastolic[2]) / 2

    elif not np.isnan(systolic[1]):

        sbp = systolic[1]
        dbp = diastolic[1]

    elif not np.isnan(systolic[0]):

        sbp = systolic[0]
        dbp = diastolic[0]

    else:

        sbp = np.nan
        dbp = np.nan

    return pd.Series([sbp, dbp])

# ============================================================
# 4. CREATE BP VARIABLES
# ============================================================

print("\n" + "=" * 60)
print("BLOOD PRESSURE PROCESSING")
print("=" * 60)

df["sbp_final"] = np.nan
df["dbp_final"] = np.nan
df["is_on_meds"] = 0

# ---------------- MEN ----------------

men = df["HV104"] == 1

bp_men = df[men].apply(
    lambda r: get_bp_average(
        r,
        ["MBP9", "MBP13", "MBP22"],
        ["MBP10", "MBP14", "MBP23"]
    ),
    axis=1
)

df.loc[men, ["sbp_final", "dbp_final"]] = bp_men.values

df.loc[
    men & (df["MBP19"] == 1),
    "is_on_meds"
] = 1

print("Men BP participants:",
      df.loc[men, "sbp_final"].notna().sum())

# ---------------- WOMEN ----------------

women = df["HV104"] == 2

bp_women = df[women].apply(
    lambda r: get_bp_average(
        r,
        ["WBP9", "WBP13", "WBP22"],
        ["WBP10", "WBP14", "WBP23"]
    ),
    axis=1
)

df.loc[women, ["sbp_final", "dbp_final"]] = bp_women.values

df.loc[
    women & (df["WBP19"] == 1),
    "is_on_meds"
] = 1

print("Women BP participants:",
      df.loc[women, "sbp_final"].notna().sum())

# ============================================================
# 5. KEEP VALID BP PARTICIPANTS
# ============================================================

before_bp = len(df)

df = df.dropna(
    subset=["sbp_final", "dbp_final"]
).copy()

after_bp = len(df)

print("\nAfter valid BP filter:", after_bp)
print("Removed:", before_bp - after_bp)

# ============================================================
# 6. HYPERTENSION OUTCOME
# ============================================================

df["hypertension"] = np.where(
    (df["sbp_final"] >= 140) |
    (df["dbp_final"] >= 90) |
    (df["is_on_meds"] == 1),
    1,
    0
)

print("\nHypertension prevalence:")
print(df["hypertension"].value_counts())

# ============================================================
# 7. BMI
# ============================================================

print("\n" + "=" * 60)
print("BMI PROCESSING")
print("=" * 60)

df["bmi_raw"] = np.where(
    df["HV104"] == 1,
    df["HB40"],
    df["HA40"]
)

df["BMI"] = np.where(
    df["bmi_raw"] < 9000,
    df["bmi_raw"] / 100,
    np.nan
)

print("BMI missing before fill:",
      df["BMI"].isna().sum())

# # fill BMI using median
bmi_median = df["BMI"].median()

df["BMI"] = df["BMI"].fillna(bmi_median)

print("BMI missing after fill:",
      df["BMI"].isna().sum())

# BMI categories

df["BMI_category"] = pd.cut(
    df["BMI"],
    bins=[0, 18.5, 25, 30, 100],
    labels=[
        "Underweight",
        "Normal",
        "Overweight",
        "Obese"
    ]
)

# ============================================================
# 8. GLUCOSE / DIABETES
# ============================================================

print("\n" + "=" * 60)
print("GLUCOSE PROCESSING")
print("=" * 60)

df["glucose"] = df["SB367"].fillna(df["SB267"])

# invalid values
df.loc[
    df["glucose"] >= 900,
    "glucose"
] = np.nan

print("Glucose missing before fill:",
      df["glucose"].isna().sum())

# # fill glucose using median
glucose_median = df["glucose"].median()

df["glucose"] = df["glucose"].fillna(glucose_median)

print("Glucose missing after fill:",
       df["glucose"].isna().sum())

# diabetes
df["diabetes"] = np.where(
    df["glucose"] >= 126,
    1,
    0
)

# prediabetes
df["prediabetes"] = np.where(
    (df["glucose"] >= 100) &
    (df["glucose"] < 126),
    1,
    0
)

# ============================================================
# 9. AGE GROUPS
# ============================================================

# ============================================================
# AGE GROUPS
# ============================================================

df["age_group"] = pd.cut(
    df["HV105"],
    bins=[18, 29, 39, 49, 59, 69, 120],
    labels=[
        "18-29",
        "30-39",
        "40-49",
        "50-59",
        "60-69",
        "70+"
    ],
    include_lowest=True
)

print("Age group missing:",
      df["age_group"].isna().sum())

# ============================================================
# 10. HOUSEHOLD VARIABLES
# ============================================================

df["crowding_index"] = (
    df["HV009"] / df["HV216"]
)

# fill infinite values
df["crowding_index"] = df["crowding_index"].replace(
    [np.inf, -np.inf],
    np.nan
)

crowding_median = df["crowding_index"].median()
df["crowding_index"] = df["crowding_index"].fillna(
     crowding_median
 )

# ============================================================
# 11. WATER
# ============================================================

improved_water_codes = [
    11, 12, 13, 14, 21, 31, 41,51,61
]

df["improved_water"] = np.where(
    df["HV201"].isin(improved_water_codes),
    1,
    0
)

# ============================================================
# 12. TOILET
# ============================================================

improved_toilet_codes = [
    11, 12, 13, 14, 21, 22,23
]

df["improved_toilet"] = np.where(
    df["HV205"].isin(improved_toilet_codes),
    1,
    0
)

# # ============================================================
# # 13. CLEAN FUEL
# # ============================================================

# clean_fuel_codes = [
#     1, 2, 3, 4, 5
# ]

# df["clean_cooking_fuel"] = np.where(
#     df["HV226"].isin(clean_fuel_codes),
#     1,
#     0
# )

# ============================================================
# 14. ELECTRICITY
# ============================================================

df["electricity"] = df["HV206"]

# ============================================================
# 15. MOBILE PHONE
# ============================================================

if "HV243A" in df.columns:

    df["mobile_phone"] = df["HV243A"]

else:

    df["mobile_phone"] = 0

# # fill missing
df["mobile_phone"] = df["mobile_phone"].fillna(0)

# ============================================================
# 16. INTERNET
# ============================================================

if "HV243B" in df.columns:

    df["internet_use"] = df["HV243B"]

else:

    df["internet_use"] = 0

# # fill missing
df["internet_use"] = df["internet_use"].fillna(0)

# ============================================================
# 17. HOUSING MATERIALS
# ============================================================

df["floor_material"] = df["HV213"]
df["wall_material"] = df["HV214"]
df["roof_material"] = df["HV215"]

# ============================================================
# 18. EDUCATION CLEANING
# ============================================================

print("\n" + "=" * 60)
print("MISSING VALUE HANDLING")
print("=" * 60)

print("Education missing before:",
      df["HV106"].isna().sum())

# remove don't know
df["HV106"] = df["HV106"].replace(8, np.nan)

# fill using mode
edu_mode = df["HV106"].mode()[0]

df["HV106"] = df["HV106"].fillna(edu_mode)

print("Education missing after:",
      df["HV106"].isna().sum())

# ============================================================
# 19. MARITAL STATUS
# ============================================================

marital_mode = df["HV115"].mode()[0]

df["HV115"] = df["HV115"].fillna(marital_mode)

# ============================================================
# 20. FINAL VARIABLE SELECTION
# ============================================================

final_variables = {

    # outcome
    "hypertension": "hypertension",

    # blood pressure
    "sbp_final": "sbp_final",
    "dbp_final": "dbp_final",

    # demographics
    "HV105": "age",
    "age_group": "age_group",
    "HV104": "sex",
    "HV106": "education_level",
    "HV115": "marital_status",
    "HV025": "residence_type",
    "HV024": "division",
    "HV270": "wealth_index",

    # anthropometric
    "BMI": "BMI",
    "BMI_category": "BMI_category",

    # metabolic
    "glucose": "glucose",
    "diabetes": "diabetes",
    "prediabetes": "prediabetes",

    # household
    "HV009": "household_size",
    "HV216": "sleeping_rooms",
    "crowding_index": "crowding_index",
    "electricity": "electricity",
    "improved_water": "improved_water",
    "improved_toilet": "improved_toilet",
    # "clean_cooking_fuel": "clean_cooking_fuel",

    # housing
    "floor_material": "floor_material",
    "wall_material": "wall_material",
    "roof_material": "roof_material",

    # technology
    "mobile_phone": "mobile_phone",
    # "internet_use": "internet_use",

    # survey design
    "HV005": "sample_weight",
    "HV021": "cluster_id",
    "HV023": "strata_id"
}

# ============================================================
# 21. CREATE FINAL DATASET
# ============================================================

df_final = df[
    list(final_variables.keys())
].rename(columns=final_variables)

# ============================================================
# 22. SURVEY WEIGHT
# ============================================================

df_final["sample_weight"] = (
    df_final["sample_weight"] / 1000000
)

# ============================================================
# 23. REMOVE DUPLICATES
# ============================================================

before_dup = len(df_final)

df_final = df_final.drop_duplicates()

after_dup = len(df_final)

print("\nDuplicates removed:",
      before_dup - after_dup)

# ============================================================
# 24. FINAL MISSING VALUE CHECK
# ============================================================

print("\n" + "=" * 60)
print("FINAL MISSING VALUES")
print("=" * 60)

missing_summary = df_final.isna().sum()

print(missing_summary)

# ============================================================
# 25. REMOVE REMAINING MISSING
# ============================================================

before_na = len(df_final)

df_final = df_final.dropna()

after_na = len(df_final)

print("\nRows removed due to remaining NA:",
      before_na - after_na)

print("Final analytic sample:",
      after_na)

# ============================================================
# 26. EXPORT FINAL DATASET
# ============================================================

df_final.to_csv(
    "KZsBDHS_2022_Hypertension_PR_FINAL_CLEAN.csv",
    index=False
)

# ============================================================
# 27. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)

print("Final shape:", df_final.shape)

print("\nHypertension prevalence:")
print(df_final["hypertension"].value_counts())

print("\nSex distribution:")
print(df_final["sex"].value_counts())

print("\nEducation distribution:")
print(df_final["education_level"].value_counts())

print("\nResidence distribution:")
print(df_final["residence_type"].value_counts())

print("\nWealth distribution:")
print(df_final["wealth_index"].value_counts())

print("\nDiabetes distribution:")
print(df_final["diabetes"].value_counts())

print("\nBMI category:")
print(df_final["BMI_category"].value_counts())

print("\nFinal missing values:")
print(df_final.isna().sum().sum())

print("\nDataset successfully exported!")

# ============================================================
# 28. VARIABLE DESCRIPTION TABLE
# ============================================================

variable_names = list(df_final.columns)

variable_descriptions = [

    "Hypertension outcome",
    "Final systolic blood pressure",
    "Final diastolic blood pressure",

    "Age in years",
    "Age category",
    "Sex",
    "Education level",
    "Marital status",
    "Urban/rural residence",
    "Administrative division",
    "Wealth index",

    "Body mass index",
    "BMI category",

    "Blood glucose",
    "Diabetes status",
    "Prediabetes status",

    "Household size",
    "Sleeping rooms",
    "Crowding index",
    "Electricity availability",
    "Improved drinking water",
    "Improved sanitation",
    # "Clean cooking fuel",

    "Floor material",
    "Wall material",
    "Roof material",

    "Mobile phone ownership",
    # "Internet use",

    "Survey sample weight",
    "Primary sampling unit",
    "Sampling strata"
]

print("\nVariables count:", len(variable_names))
print("Descriptions count:", len(variable_descriptions))

variable_description = pd.DataFrame({
    "Variable": variable_names,
    "Description": variable_descriptions
})

variable_description.to_csv(
    "BDHS_2022_Variable_Descriptions.csv",
    index=False
)

print("\nVariable description file exported!")

INITIAL DATA
Initial shape: (132463, 549)

After de facto resident filter: 126597
After adult filter: 81881

BLOOD PRESSURE PROCESSING
Men BP participants: 6192
Women BP participants: 7656

After valid BP filter: 13848
Removed: 68033

Hypertension prevalence:
hypertension
0    10971
1     2877
Name: count, dtype: int64

BMI PROCESSING
BMI missing before fill: 116
BMI missing after fill: 0

GLUCOSE PROCESSING
Glucose missing before fill: 322
Glucose missing after fill: 0
Age group missing: 0

MISSING VALUE HANDLING
Education missing before: 0
Education missing after: 0

Duplicates removed: 0

FINAL MISSING VALUES
hypertension       0
sbp_final          0
dbp_final          0
age                0
age_group          0
sex                0
education_level    0
marital_status     0
residence_type     0
division           0
wealth_index       0
BMI                0
BMI_category       0
glucose            0
diabetes           0
prediabetes        0
household_size     0
sleeping_rooms     0
cr

In [ ]:
df=pd.read_csv("/content/KZBDHS_2022_Hypertension_PR_FINAL_CLEAN.csv")
df.columns

Index(['hypertension', 'sbp_final', 'dbp_final', 'age', 'age_group', 'sex',
       'education_level', 'marital_status', 'residence_type', 'division',
       'wealth_index', 'BMI', 'BMI_category', 'glucose', 'diabetes',
       'prediabetes', 'household_size', 'sleeping_rooms', 'crowding_index',
       'electricity', 'improved_water', 'improved_toilet', 'floor_material',
       'wall_material', 'roof_material', 'mobile_phone', 'internet_use',
       'sample_weight', 'cluster_id', 'strata_id'],
      dtype='object')

In [ ]:
df.isnull().sum()

,0
hypertension,0
sbp_final,0
dbp_final,0
age,0
age_group,0
sex,0
education_level,0
marital_status,0
residence_type,0
division,0


In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("/content/BDHS_Hypertension_Final_13848.csv")

# 1. Handle Missing Values (BMI & Glucose)
# Using median to maintain the full 13,848 rows
df['BMI'] = df['BMI'].fillna(df['BMI'].median())
df['glucose'] = df['glucose'].fillna(df['glucose'].median())

# 2. Refine Education
# Map 8.0 (Don't Know) to 0.0 (No Education)
df['education_level'] = df['education_level'].replace(8.0, 0.0)

# 3. Simplify Household Features
# Create Overcrowding Index for better SHAP interpretability
df['crowding_index'] = df['household_size'] / df['sleeping_rooms']
df['crowding_index'] = df['crowding_index'].replace([np.inf, -np.inf], 0)

# 4. Define Study Features (Categorical & Continuous)
# sex: 1=Male, 2=Female
# residence_type: 1=Urban, 2=Rural
# marital_status: 1=Married, 0=Others
# water_source/toilet_type: Grouped into Improved (1) vs Unimproved (0)

improved_water = [11, 12, 13, 14, 21, 31, 51, 71]
df['water_source_imp'] = df['water_source'].apply(lambda x: 1 if x in improved_water else 0)

improved_toilet = [11, 12, 13, 14, 15, 21, 22]
df['toilet_type_imp'] = df['toilet_type'].apply(lambda x: 1 if x in improved_toilet else 0)

# 5. Final Feature Matrix (X) and Target (y)
features = [
    'age', 'sex', 'education_level', 'wealth_index', 'residence_type',
    'marital_status', 'crowding_index', 'BMI', 'glucose',
    'water_source_imp', 'toilet_type_imp', 'division'
]

# One-hot encoding for Division (8 regions of Bangladesh)
X = pd.get_dummies(df[features], columns=['division'], prefix='div', drop_first=True)
y = df['hypertension']
weights = df['sample_weight']

print(f"Final Model Matrix: {X.shape[0]} rows and {X.shape[1]} features.")

Final Model Matrix: 13848 rows and 18 features.


In [ ]:
df.columns

Index(['age', 'sex', 'education_level', 'marital_status', 'residence_type',
       'division', 'wealth_index', 'household_size', 'head_sex',
       'has_electricity', 'water_source', 'toilet_type', 'sleeping_rooms',
       'sample_weight', 'cluster_id', 'strata_id', 'hypertension', 'sbp_final',
       'dbp_final', 'BMI', 'glucose', 'smoker', 'crowding_index',
       'water_source_imp', 'toilet_type_imp'],
      dtype='object')

In [ ]:
df['crowding_index'].value_counts()

,count
crowding_index,
2.000000,2741
3.000000,1420
2.500000,1388
1.500000,1240
1.000000,1127
...,...
0.285714,4
0.250000,4
0.300000,4


In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------

df = pd.read_spss(
    "/content/BDPR81FL.SAV",
    convert_categoricals=False
)

# --------------------------------------------------
# 1. DHS FILTERS
# --------------------------------------------------
# HV103 = de facto resident
# HV105 = age
# --------------------------------------------------

df = df[
    (df["HV103"] == 1) &
    (df["HV105"] >= 18)
].copy()

# --------------------------------------------------
# 2. BP AVERAGING FUNCTION
# --------------------------------------------------

def calculate_bdhs_average(
    row,
    s1, d1,
    s2, d2,
    s3, d3
):

    # -------------------------------
    # SYSTOLIC
    # -------------------------------

    sbp1 = row[s1]
    sbp2 = row[s2]
    sbp3 = row[s3]

    if sbp2 < 900 and sbp3 < 900:
        sbp_avg = (sbp2 + sbp3) / 2

    elif sbp2 < 900:
        sbp_avg = sbp2

    elif sbp1 < 900:
        sbp_avg = sbp1

    else:
        sbp_avg = np.nan

    # -------------------------------
    # DIASTOLIC
    # -------------------------------

    dbp1 = row[d1]
    dbp2 = row[d2]
    dbp3 = row[d3]

    if dbp2 < 900 and dbp3 < 900:
        dbp_avg = (dbp2 + dbp3) / 2

    elif dbp2 < 900:
        dbp_avg = dbp2

    elif dbp1 < 900:
        dbp_avg = dbp1

    else:
        dbp_avg = np.nan

    return sbp_avg, dbp_avg

# --------------------------------------------------
# 3. CREATE VARIABLES
# --------------------------------------------------

df["sbp_avg"] = np.nan
df["dbp_avg"] = np.nan
df["is_on_meds"] = 0

# --------------------------------------------------
# MEN
# --------------------------------------------------

m_mask = df["HV104"] == 1

m_readings = df[m_mask].apply(
    lambda r: calculate_bdhs_average(

        r,

        # 1st reading
        "MBP12", "MBP13",

        # 2nd reading
        "MBP22", "MBP23",

        # 3rd reading
        "MBP22", "MBP23"

    ),
    axis=1,
    result_type="expand"
)

if not m_readings.empty:

    df.loc[m_mask, ["sbp_avg", "dbp_avg"]] = m_readings.values

    # medication
    df.loc[
        m_mask & (df["MBP19"] == 1),
        "is_on_meds"
    ] = 1

# --------------------------------------------------
# WOMEN
# --------------------------------------------------

w_mask = df["HV104"] == 2

w_readings = df[w_mask].apply(
    lambda r: calculate_bdhs_average(

        r,

        # 1st reading
        "WBP12", "WBP13",

        # 2nd reading
        "WBP22", "WBP23",

        # 3rd reading
        "WBP22", "WBP23"

    ),
    axis=1,
    result_type="expand"
)

if not w_readings.empty:

    df.loc[w_mask, ["sbp_avg", "dbp_avg"]] = w_readings.values

    df.loc[
        w_mask & (df["WBP19"] == 1),
        "is_on_meds"
    ] = 1

# --------------------------------------------------
# 4. REMOVE INVALID BP
# --------------------------------------------------

df = df.dropna(
    subset=["sbp_avg", "dbp_avg"]
)

# --------------------------------------------------
# 5. HYPERTENSION
# --------------------------------------------------

df["hypertension"] = np.where(

    (
        (df["sbp_avg"] >= 140) |
        (df["dbp_avg"] >= 90) |
        (df["is_on_meds"] == 1)
    ),

    1,
    0
)

# --------------------------------------------------
# 6. SAMPLE WEIGHT
# --------------------------------------------------

df["sample_weight"] = (
    df["HV005"] / 1000000
)

# --------------------------------------------------
# 7. RESULTS
# --------------------------------------------------

print("Final valid sample size:", len(df))

print("\nGender counts:")
print(df["HV104"].value_counts())

print("\nHypertension prevalence by gender:")
print(
    df.groupby("HV104")["hypertension"]
    .mean() * 100
)

Final valid sample size: 13556

Gender counts:
HV104
2.0    7529
1.0    6027
Name: count, dtype: int64

Hypertension prevalence by gender:
HV104
1.0    19.678115
2.0    25.355293
Name: hypertension, dtype: float64


In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------
df = pd.read_spss("/content/BDPR81FL.SAV", convert_categoricals=False)

# --------------------------------------------------
# 1. DHS FILTERS (Population Inclusion/Exclusion)
# --------------------------------------------------
# Inclusion: De facto residents (HV103=1) and Adults (HV105>=18)
df = df[(df["HV103"] == 1) & (df["HV105"] >= 18)].copy()

# --------------------------------------------------
# 2. CORRECTED BP AVERAGING FUNCTION
# --------------------------------------------------
def calculate_bdhs_average(row, s1, d1, s2, d2, s3, d3):
    # Clean raw readings: treat 900+ (Refused/Missing) as NaN
    s = [row[s1] if row[s1] < 900 else np.nan for s1 in [s1, s2, s3]]
    d = [row[d1] if row[d1] < 900 else np.nan for d1 in [d1, d2, d3]]

    # Hierarchical Logic (Section 14.2 of BDHS Report)
    # 1. Average of 2nd and 3rd readings
    if not np.isnan(s[1]) and not np.isnan(s[2]):
        sbp_avg, dbp_avg = (s[1] + s[2]) / 2, (d[1] + d[2]) / 2
    # 2. Fallback to 2nd reading if 3rd is missing
    elif not np.isnan(s[1]):
        sbp_avg, dbp_avg = s[1], d[1]
    # 3. Fallback to 1st reading if 2nd/3rd are missing
    elif not np.isnan(s[0]):
        sbp_avg, dbp_avg = s[0], d[0]
    # 4. Exclusion: No valid measurements
    else:
        sbp_avg, dbp_avg = np.nan, np.nan

    return sbp_avg, dbp_avg

# --------------------------------------------------
# 3. APPLY LOGIC BY GENDER (Corrected Column Mapping)
# --------------------------------------------------
df["sbp_avg"] = np.nan
df["dbp_avg"] = np.nan
df["is_on_meds"] = 0

# --- MEN ---
m_mask = df["HV104"] == 1
if m_mask.any():
    m_readings = df[m_mask].apply(
        lambda r: calculate_bdhs_average(
            r,
            "MBP9",  "MBP10", # 1st Reading
            "MBP13", "MBP14", # 2nd Reading
            "MBP22", "MBP23"  # 3rd Reading
        ), axis=1, result_type="expand"
    )
    df.loc[m_mask, ["sbp_avg", "dbp_avg"]] = m_readings.values
    df.loc[m_mask & (df["MBP19"] == 1), "is_on_meds"] = 1

# --- WOMEN ---
w_mask = df["HV104"] == 2
if w_mask.any():
    w_readings = df[w_mask].apply(
        lambda r: calculate_bdhs_average(
            r,
            "WBP9",  "WBP10", # 1st Reading
            "WBP13", "WBP14", # 2nd Reading
            "WBP22", "WBP23"  # 3rd Reading
        ), axis=1, result_type="expand"
    )
    df.loc[w_mask, ["sbp_avg", "dbp_avg"]] = w_readings.values
    df.loc[w_mask & (df["WBP19"] == 1), "is_on_meds"] = 1

# --------------------------------------------------
# 4. MEASUREMENT-LEVEL EXCLUSION
# --------------------------------------------------
# Exclusion: Remove respondents with zero valid readings
df = df.dropna(subset=["sbp_avg", "dbp_avg"])

# --------------------------------------------------
# 5. OUTCOME & WEIGHTING
# --------------------------------------------------
df["hypertension"] = np.where(
    (df["sbp_avg"] >= 140) | (df["dbp_avg"] >= 90) | (df["is_on_meds"] == 1), 1, 0
)

# Sample Weight Scaling
df["sample_weight"] = df["HV005"] / 1000000

# --------------------------------------------------
# 6. ACCURATE WEIGHTED PREVALENCE
# --------------------------------------------------
def get_weighted_mean(group):
    return (group["hypertension"] * group["sample_weight"]).sum() / group["sample_weight"].sum()

print(f"Final valid sample size: {len(df)}")
print("\nWeighted Hypertension Prevalence (%) by Gender (1=Male, 2=Female):")
print(df.groupby("HV104").apply(get_weighted_mean) * 100)

Final valid sample size: 13848

Weighted Hypertension Prevalence (%) by Gender (1=Male, 2=Female):
HV104
1.0    16.972828
2.0    23.187128
dtype: float64


/tmp/ipykernel_619/1552878497.py:98: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df.groupby("HV104").apply(get_weighted_mean) * 100)


In [ ]:
df = pd.read_csv("/content/BDHS_Final_Hypertension_Data.csv")
df.head()

,hypertension,sbp_final,dbp_final,is_on_meds,sample_weight,HV021,HV023,HV024,HV025,HV104,HV105,HV106,HV270,SB367,SB267
0,1,119.0,79.0,1,0.155639,1.0,1.0,1.0,1.0,2.0,75.0,2.0,5.0,NaN,114.0
1,1,139.0,95.0,1,0.155639,1.0,1.0,1.0,1.0,1.0,55.0,3.0,5.0,74.0,NaN
2,1,129.0,97.0,1,0.155639,1.0,1.0,1.0,1.0,2.0,46.0,3.0,5.0,NaN,139.0
3,1,138.5,90.0,0,0.155639,1.0,1.0,1.0,1.0,1.0,51.0,0.0,4.0,115.0,NaN
4,0,108.5,84.0,0,0.155639,1.0,1.0,1.0,1.0,2.0,35.0,0.0,4.0,NaN,93.0


In [ ]:
df['hypertension'].value_counts()

,count
hypertension,
0,10971
1,2877


In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------
df = pd.read_spss("/content/BDPR81FL.SAV", convert_categoricals=False)

# --------------------------------------------------
# 1. APPLY FILTERS (BDHS STANDARDS)
# --------------------------------------------------
# De facto residents (HV103=1) and Age 18+ (HV105>=18)
df = df[(df["HV103"] == 1) & (df["HV105"] >= 18)].copy()

# --------------------------------------------------
# 2. DEFINE BP AVERAGING LOGIC
# --------------------------------------------------
def calculate_bdhs_average(row, s1, d1, s2, d2, s3, d3):
    """Calculates average SBP/DBP according to BDHS hierarchical logic."""
    # Systolic Logic
    sbp1, sbp2, sbp3 = row[s1], row[s2], row[s3]
    if sbp2 < 900 and sbp3 < 900:
        sbp_avg = (sbp2 + sbp3) / 2
    elif sbp2 < 900:
        sbp_avg = sbp2
    elif sbp1 < 900:
        sbp_avg = sbp1
    else:
        sbp_avg = np.nan

    # Diastolic Logic
    dbp1, dbp2, dbp3 = row[d1], row[d2], row[d3]
    if dbp2 < 900 and dbp3 < 900:
        dbp_avg = (dbp2 + dbp3) / 2
    elif dbp2 < 900:
        dbp_avg = dbp2
    elif dbp1 < 900:
        dbp_avg = dbp1
    else:
        dbp_avg = np.nan

    return sbp_avg, dbp_avg

# --------------------------------------------------
# 3. PROCESS MEN AND WOMEN SEPARATELY
# --------------------------------------------------
df["sbp_avg"] = np.nan
df["dbp_avg"] = np.nan
df["is_on_meds"] = 0

# Men (HV104 == 1)
m_mask = df["HV104"] == 1
m_readings = df[m_mask].apply(
    lambda r: calculate_bdhs_average(r, "MBP9", "MBP10", "MBP13", "MBP14", "MBP22", "MBP23"),
    axis=1, result_type='expand'
)
if not m_readings.empty:
    df.loc[m_mask, ["sbp_avg", "dbp_avg"]] = m_readings.values
    # MBP19: 1 = Yes, 0 = No (Exclude 8=DK and 9=Missing)
    df.loc[m_mask & (df["MBP19"] == 1), "is_on_meds"] = 1

# Women (HV104 == 2)
w_mask = df["HV104"] == 2
w_readings = df[w_mask].apply(
    lambda r: calculate_bdhs_average(r, "WBP9", "WBP10", "WBP13", "WBP14", "WBP22", "WBP23"),
    axis=1, result_type='expand'
)
if not w_readings.empty:
    df.loc[w_mask, ["sbp_avg", "dbp_avg"]] = w_readings.values
    # WBP19: 1 = Yes
    df.loc[w_mask & (df["WBP19"] == 1), "is_on_meds"] = 1

# --------------------------------------------------
# 4. FINAL HYPERTENSION DEFINITION
# --------------------------------------------------
# Drop those with no valid measurements at all
df = df.dropna(subset=["sbp_avg", "dbp_avg"])

df["hypertension"] = np.where(
    (df["sbp_avg"] >= 140) |
    (df["dbp_avg"] >= 90) |
    (df["is_on_meds"] == 1),
    1, 0
)

# --------------------------------------------------
# 5. SAMPLE WEIGHT & OUTPUT
# --------------------------------------------------
df["sample_weight"] = df["HV005"] / 1000000

print(f"Final valid sample size: {len(df)}")
print("\nUnweighted prevalence by gender:")
print(df.groupby("HV104")["hypertension"].mean() * 100)

Final valid sample size: 13848

Unweighted prevalence by gender:
HV104
1.0    17.118863
2.0    23.733020
Name: hypertension, dtype: float64


In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. LOAD & INITIAL FILTERING
# --------------------------------------------------
df = pd.read_spss("/content/BDPR81FL.SAV", convert_categoricals=False)

# Filters: De Facto Residents (HV103=1) and Adults (HV105 >= 18)
df = df[(df["HV103"] == 1) & (df["HV105"] >= 18)].copy()

# --------------------------------------------------
# 2. DEFINE HIERARCHICAL AVERAGING FUNCTION
# --------------------------------------------------
def get_bdhs_avg(row, s1, d1, s2, d2, s3, d3):
    """
    Implements the BDHS report logic:
    1. Average of 2nd & 3rd readings.
    2. If 3rd is missing, use 2nd.
    3. If 2nd/3rd are missing, use 1st.
    Valid readings are < 900.
    """
    # Systolic
    sbp1, sbp2, sbp3 = row[s1], row[s2], row[s3]
    if sbp2 < 900 and sbp3 < 900:
        s_avg = (sbp2 + sbp3) / 2
    elif sbp2 < 900:
        s_avg = sbp2
    elif sbp1 < 900:
        s_avg = sbp1
    else:
        s_avg = np.nan

    # Diastolic
    dbp1, dbp2, dbp3 = row[d1], row[d2], row[d3]
    if dbp2 < 900 and dbp3 < 900:
        d_avg = (dbp2 + dbp3) / 2
    elif dbp2 < 900:
        d_avg = dbp2
    elif dbp1 < 900:
        d_avg = dbp1
    else:
        d_avg = np.nan

    return s_avg, d_avg

# --------------------------------------------------
# 3. APPLY LOGIC BY GENDER (Using Variables.docx names)
# --------------------------------------------------
df["sbp_final"] = np.nan
df["dbp_final"] = np.nan
df["on_medication"] = 0

# Process Men
m_mask = df["HV104"] == 1
if m_mask.any():
    m_data = df[m_mask].apply(
        lambda r: get_bdhs_avg(r, "MBP9", "MBP10", "MBP13", "MBP14", "MBP22", "MBP23"),
        axis=1, result_type='expand'
    )
    df.loc[m_mask, ["sbp_final", "dbp_final"]] = m_data.values
    # MBP19: 1 = Yes, 0 = No (Ignore 8/9 special codes)
    df.loc[m_mask & (df["MBP19"] == 1), "on_medication"] = 1

# Process Women
w_mask = df["HV104"] == 2
if w_mask.any():
    w_data = df[w_mask].apply(
        lambda r: get_bdhs_avg(r, "WBP9", "WBP10", "WBP13", "WBP14", "WBP22", "WBP23"),
        axis=1, result_type='expand'
    )
    df.loc[w_mask, ["sbp_final", "dbp_final"]] = w_data.values
    # WBP19: 1 = Yes
    df.loc[w_mask & (df["WBP19"] == 1), "on_medication"] = 1

# --------------------------------------------------
# 4. DEFINE HYPERTENSION STATUS
# --------------------------------------------------
# Drop cases with no valid BP measurements
df = df.dropna(subset=["sbp_final", "dbp_final"])

# Hypertension: SBP >= 140 OR DBP >= 90 OR taking medication
df["hypertension"] = np.where(
    (df["sbp_final"] >= 140) |
    (df["dbp_final"] >= 90) |
    (df["on_medication"] == 1),
    1, 0
)

# --------------------------------------------------
# 5. VALIDATION & WEIGHTING
# --------------------------------------------------
df["sample_weight"] = df["HV005"] / 1000000

print(f"Total Valid Sample: {len(df)}")
print("\nUnweighted Prevalence by Gender (1=Male, 2=Female):")
print(df.groupby("HV104")["hypertension"].mean() * 100)

Total Valid Sample: 13848

Unweighted Prevalence by Gender (1=Male, 2=Female):
HV104
1.0    17.118863
2.0    23.733020
Name: hypertension, dtype: float64


In [ ]:
import pandas as pd
import numpy as np

# -------------------------------------------------------------------------
# 1. LOAD DATA & INITIAL SELECTION
# -------------------------------------------------------------------------
# Load the dataset
file_path = "/content/BDPR81FL.SAV"
df = pd.read_spss(file_path, convert_categoricals=False)

# Select all relevant variables for the study
# Includes Administrative, Demographic, Clinical, and Socio-economic factors
cols_to_keep = [
    # Survey Admin & Weights
    "HV001", "HV002", "HV005", "HV021", "HV022", "HV024", "HV025",
    # Eligibility
    "HV103", "HV104", "HV105", "HV106", "HV270",
    # Clinical - Men
    "MBP9", "MBP10", "MBP13", "MBP14", "MBP22", "MBP23", "MBP19", "MBP16",
    # Clinical - Women
    "WBP9", "WBP10", "WBP13", "WBP14", "WBP22", "WBP23", "WBP19", "WBP16",
    # Comorbidities (Diabetes)
    "SB367", "SB267"
]

# Keep only available columns to avoid errors
existing_cols = [c for c in cols_to_keep if c in df.columns]
df = df[existing_cols].copy()

# -------------------------------------------------------------------------
# 2. POPULATION FILTERS
# -------------------------------------------------------------------------
# Filter for De Facto residents (HV103 == 1) and Adults (HV105 >= 18)
df = df[(df["HV103"] == 1) & (df["HV105"] >= 18)].copy()

# -------------------------------------------------------------------------
# 3. BLOOD PRESSURE CALCULATION LOGIC
# -------------------------------------------------------------------------
def get_bdhs_averages(row, s1, d1, s2, d2, s3, d3):
    """
    Applies BDHS hierarchy:
    1. Average of 2nd and 3rd readings.
    2. If 3rd is missing, use 2nd.
    3. If 2nd and 3rd are missing, use 1st.
    DHS codes 900+ (994, 996, 999, etc.) are treated as missing.
    """
    # Extract readings
    sys = [row[s1], row[s2], row[s3]]
    dia = [row[d1], row[d2], row[d3]]

    # Valid readings are < 900
    s1_v, s2_v, s3_v = [x if x < 900 else np.nan for x in sys]
    d1_v, d2_v, d3_v = [x if x < 900 else np.nan for x in dia]

    # Calculate Systolic Average
    if not np.isnan(s2_v) and not np.isnan(s3_v):
        sbp_avg = (s2_v + s3_v) / 2
    elif not np.isnan(s2_v):
        sbp_avg = s2_v
    else:
        sbp_avg = s1_v # May still be NaN if s1 is missing

    # Calculate Diastolic Average
    if not np.isnan(d2_v) and not np.isnan(d3_v):
        dbp_avg = (d2_v + d3_v) / 2
    elif not np.isnan(d2_v):
        dbp_avg = d2_v
    else:
        dbp_avg = d1_v

    return sbp_avg, dbp_avg

# -------------------------------------------------------------------------
# 4. DATA PROCESSING BY GENDER
# -------------------------------------------------------------------------
df["sbp_final"] = np.nan
df["dbp_final"] = np.nan
df["is_on_meds"] = 0

# Logic for Men (HV104 == 1)
m_mask = df["HV104"] == 1
if m_mask.any():
    m_res = df[m_mask].apply(
        lambda r: get_bdhs_averages(r, "MBP9", "MBP10", "MBP13", "MBP14", "MBP22", "MBP23"),
        axis=1, result_type='expand'
    )
    df.loc[m_mask, ["sbp_final", "dbp_final"]] = m_res.values
    # MBP19: 1 = Yes
    df.loc[m_mask & (df["MBP19"] == 1), "is_on_meds"] = 1

# Logic for Women (HV104 == 2)
w_mask = df["HV104"] == 2
if w_mask.any():
    w_res = df[w_mask].apply(
        lambda r: get_bdhs_averages(r, "WBP9", "WBP10", "WBP13", "WBP14", "WBP22", "WBP23"),
        axis=1, result_type='expand'
    )
    df.loc[w_mask, ["sbp_final", "dbp_final"]] = w_res.values
    # WBP19: 1 = Yes
    df.loc[w_mask & (df["WBP19"] == 1), "is_on_meds"] = 1

# -------------------------------------------------------------------------
# 5. DEFINE OUTCOME VARIABLES
# -------------------------------------------------------------------------
# Clean dataset: Remove those with no valid measurement
df = df.dropna(subset=["sbp_final", "dbp_final"])

# Hypertension: SBP >= 140 OR DBP >= 90 OR Taking Medication
df["hypertension"] = np.where(
    (df["sbp_final"] >= 140) |
    (df["dbp_final"] >= 90) |
    (df["is_on_meds"] == 1),
    1, 0
)

# Sample Weight (DHS standard)
df["sample_weight"] = df["HV005"] / 1000000

# -------------------------------------------------------------------------
# 6. RESULTS SUMMARY
# -------------------------------------------------------------------------
print("-" * 30)
print(f"Study Sample Size: {len(df)}")
print("-" * 30)

# Unweighted prevalence for quick check
prevalence = df.groupby("HV104")["hypertension"].mean() * 100
print("\nUnweighted Hypertension Prevalence:")
print(f"Men (1): {prevalence.get(1, 0):.2f}%")
print(f"Women (2): {prevalence.get(2, 0):.2f}%")

# Weighted prevalence
weighted_prev = (df.groupby("HV104").apply(lambda x: (x["hypertension"] * x["sample_weight"]).sum() / x["sample_weight"].sum())) * 100
print("\nWeighted Hypertension Prevalence:")
print(f"Men (1): {weighted_prev.get(1, 0):.2f}%")
print(f"Women (2): {weighted_prev.get(2, 0):.2f}%")

------------------------------
Study Sample Size: 13848
------------------------------

Unweighted Hypertension Prevalence:
Men (1): 17.12%
Women (2): 23.73%

Weighted Hypertension Prevalence:
Men (1): 16.97%
Women (2): 23.19%


/tmp/ipykernel_20386/3059829353.py:133: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_prev = (df.groupby("HV104").apply(lambda x: (x["hypertension"] * x["sample_weight"]).sum() / x["sample_weight"].sum())) * 100


In [ ]:
import pandas as pd
import numpy as np

# -------------------------------------------------------------------------
# 1. LOAD DATA & INITIAL SELECTION
# -------------------------------------------------------------------------
# Load the dataset
file_path = "/content/BDPR81FL.SAV"
df = pd.read_spss(file_path, convert_categoricals=False)

# Select all relevant variables for the study
cols_to_keep = [
    # Survey Admin & Weights
    "HV001", "HV002", "HV005", "HV021", "HV022", "HV024", "HV025",
    # Eligibility & Demographics
    "HV103", "HV104", "HV105", "HV106", "HV270",
    # Clinical - Men
    "MBP9", "MBP10", "MBP13", "MBP14", "MBP22", "MBP23", "MBP19", "MBP16",
    # Clinical - Women
    "WBP9", "WBP10", "WBP13", "WBP14", "WBP22", "WBP23", "WBP19", "WBP16",
    # Comorbidities (Diabetes)
    "SB367", "SB267"
]

# Keep only available columns to avoid errors
existing_cols = [c for c in cols_to_keep if c in df.columns]
df = df[existing_cols].copy()

# -------------------------------------------------------------------------
# 2. POPULATION FILTERS
# -------------------------------------------------------------------------
# Filter for De Facto residents (HV103 == 1) and Adults (HV105 >= 18)
df = df[(df["HV103"] == 1) & (df["HV105"] >= 18)].copy()

# -------------------------------------------------------------------------
# 3. BLOOD PRESSURE CALCULATION LOGIC
# -------------------------------------------------------------------------
def get_bdhs_averages(row, s1, d1, s2, d2, s3, d3):
    """
    Applies BDHS hierarchy:
    1. Average of 2nd and 3rd readings.
    2. If 3rd is missing, use 2nd.
    3. If 2nd and 3rd are missing, use 1st.
    """
    # Extract readings and treat codes 900+ as NaN
    s1_v = row[s1] if row[s1] < 900 else np.nan
    s2_v = row[s2] if row[s2] < 900 else np.nan
    s3_v = row[s3] if row[s3] < 900 else np.nan

    d1_v = row[d1] if row[d1] < 900 else np.nan
    d2_v = row[d2] if row[d2] < 900 else np.nan
    d3_v = row[d3] if row[d3] < 900 else np.nan

    # Systolic Average
    if not np.isnan(s2_v) and not np.isnan(s3_v):
        sbp_avg = (s2_v + s3_v) / 2
    elif not np.isnan(s2_v):
        sbp_avg = s2_v
    else:
        sbp_avg = s1_v

    # Diastolic Average
    if not np.isnan(d2_v) and not np.isnan(d3_v):
        dbp_avg = (d2_v + d3_v) / 2
    elif not np.isnan(d2_v):
        dbp_avg = d2_v
    else:
        dbp_avg = d1_v

    return sbp_avg, dbp_avg

# -------------------------------------------------------------------------
# 4. DATA PROCESSING BY GENDER
# -------------------------------------------------------------------------
df["sbp_final"] = np.nan
df["dbp_final"] = np.nan
df["is_on_meds"] = 0

# Logic for Men (HV104 == 1)
m_mask = df["HV104"] == 1
if m_mask.any():
    m_res = df[m_mask].apply(
        lambda r: get_bdhs_averages(r, "MBP9", "MBP10", "MBP13", "MBP14", "MBP22", "MBP23"),
        axis=1, result_type='expand'
    )
    df.loc[m_mask, ["sbp_final", "dbp_final"]] = m_res.values
    df.loc[m_mask & (df["MBP19"] == 1), "is_on_meds"] = 1

# Logic for Women (HV104 == 2)
w_mask = df["HV104"] == 2
if w_mask.any():
    w_res = df[w_mask].apply(
        lambda r: get_bdhs_averages(r, "WBP9", "WBP10", "WBP13", "WBP14", "WBP22", "WBP23"),
        axis=1, result_type='expand'
    )
    df.loc[w_mask, ["sbp_final", "dbp_final"]] = w_res.values
    df.loc[w_mask & (df["WBP19"] == 1), "is_on_meds"] = 1

# -------------------------------------------------------------------------
# 5. DEFINE OUTCOME VARIABLES
# -------------------------------------------------------------------------
# Remove those without valid BP readings
df = df.dropna(subset=["sbp_final", "dbp_final"])

# Hypertension Outcome
df["hypertension"] = np.where(
    (df["sbp_final"] >= 140) | (df["dbp_final"] >= 90) | (df["is_on_meds"] == 1),
    1, 0
)

# Sample Weight
df["sample_weight"] = df["HV005"] / 1000000

# -------------------------------------------------------------------------
# 6. SAVE AND EXPORT
# -------------------------------------------------------------------------
# Define file name
output_filename = "bdhs_hypertension_final.csv"

# Save to CSV
df.to_csv(output_filename, index=False)

print(f"File saved successfully: {output_filename}")
print(f"Total Rows Exported: {len(df)}")

# Quick verification of prevalence
weighted_prev = (df.groupby("HV104").apply(
    lambda x: (x["hypertension"] * x["sample_weight"]).sum() / x["sample_weight"].sum()
)) * 100
print("\nWeighted Hypertension Prevalence by Gender:")
print(weighted_prev)

File saved successfully: bdhs_hypertension_final.csv
Total Rows Exported: 13848

Weighted Hypertension Prevalence by Gender:
HV104
1.0    16.972828
2.0    23.187128
dtype: float64


/tmp/ipykernel_20386/2882089161.py:127: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_prev = (df.groupby("HV104").apply(


In [ ]:
df = pd.read_csv("/content/bdhs_hypertension_final.csv")
df.head()



,HV001,HV002,HV005,HV021,HV022,HV024,HV025,HV103,HV104,HV105,...,WBP23,WBP19,WBP16,SB367,SB267,sbp_final,dbp_final,is_on_meds,hypertension,sample_weight
0,1.0,1.0,155639.0,1.0,1.0,1.0,1.0,1.0,2.0,75.0,...,76.0,1.0,1.0,NaN,114.0,119.0,79.0,1,1,0.155639
1,1.0,1.0,155639.0,1.0,1.0,1.0,1.0,1.0,1.0,55.0,...,NaN,NaN,NaN,74.0,NaN,139.0,95.0,1,1,0.155639
2,1.0,1.0,155639.0,1.0,1.0,1.0,1.0,1.0,2.0,46.0,...,96.0,1.0,1.0,NaN,139.0,129.0,97.0,1,1,0.155639
3,1.0,18.0,155639.0,1.0,1.0,1.0,1.0,1.0,1.0,51.0,...,NaN,NaN,NaN,115.0,NaN,138.5,90.0,0,1,0.155639
4,1.0,18.0,155639.0,1.0,1.0,1.0,1.0,1.0,2.0,35.0,...,82.0,NaN,0.0,NaN,93.0,108.5,84.0,0,0,0.155639


In [ ]:
df['hypertension'].value_counts()

,count
hypertension,
0,10971
1,2877


In [ ]:
df.columns

Index(['HV001', 'HV002', 'HV005', 'HV021', 'HV022', 'HV024', 'HV025', 'HV103',
       'HV104', 'HV105', 'HV106', 'HV270', 'MBP9', 'MBP10', 'MBP13', 'MBP14',
       'MBP22', 'MBP23', 'MBP19', 'MBP16', 'WBP9', 'WBP10', 'WBP13', 'WBP14',
       'WBP22', 'WBP23', 'WBP19', 'WBP16', 'SB367', 'SB267', 'sbp_final',
       'dbp_final', 'is_on_meds', 'hypertension', 'sample_weight'],
      dtype='object')

In [ ]:
final_cols = [
    'hypertension', 'sbp_final', 'dbp_final', 'sample_weight',
    'HV021', 'HV022', 'HV024', 'HV025', 'HV104', 'HV105',
    'HV106', 'HV270', 'is_on_meds', 'SB367', 'SB267'
]
df_clean = df[final_cols]

In [ ]:
# List of columns to check
final_cols = [
    'hypertension', 'sbp_final', 'dbp_final', 'sample_weight',
    'HV021', 'HV022', 'HV024', 'HV025', 'HV104', 'HV105',
    'HV106', 'HV270', 'is_on_meds', 'SB367', 'SB267'
]

# Loop through and print value counts for categorical/discrete variables
# Note: We skip continuous variables like 'sbp_final' or 'sample_weight'
# for readability, as they have too many unique values.

categorical_cols = [
    'hypertension', 'HV024', 'HV025', 'HV104',
    'HV106', 'HV270', 'is_on_meds'
]

print("--- VALUE COUNTS FOR CATEGORICAL VARIABLES ---")
for col in categorical_cols:
    if col in df_clean.columns:
        print(f"\nValue Counts for {col}:")
        print(df_clean[col].value_counts().sort_index())
        print("-" * 30)

# For continuous variables, it is better to look at descriptive statistics
print("\n--- DESCRIPTIVE STATS FOR CONTINUOUS VARIABLES ---")
continuous_cols = ['sbp_final', 'dbp_final', 'HV105', 'SB367']
print(df_clean[continuous_cols].describe())

--- VALUE COUNTS FOR CATEGORICAL VARIABLES ---

Value Counts for hypertension:
hypertension
0    10971
1     2877
Name: count, dtype: int64
------------------------------

Value Counts for HV024:
HV024
1.0    1470
2.0    1938
3.0    1883
4.0    1787
5.0    1577
6.0    1797
7.0    1716
8.0    1680
Name: count, dtype: int64
------------------------------

Value Counts for HV025:
HV025
1.0    4806
2.0    9042
Name: count, dtype: int64
------------------------------

Value Counts for HV104:
HV104
1.0    6192
2.0    7656
Name: count, dtype: int64
------------------------------

Value Counts for HV106:
HV106
0.0    3522
1.0    3493
2.0    4580
3.0    2240
8.0      13
Name: count, dtype: int64
------------------------------

Value Counts for HV270:
HV270
1.0    2484
2.0    2683
3.0    2651
4.0    2903
5.0    3127
Name: count, dtype: int64
------------------------------

Value Counts for is_on_meds:
is_on_meds
0    12505
1     1343
Name: count, dtype: int64
------------------------------

--- 

In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------

df = pd.read_spss(
    "/content/BDPR81FL.SAV",
    convert_categoricals=False
)

print("Initial:", df.shape)

# --------------------------------------------------
# 1. DE FACTO RESIDENTS
# HV103 = 1
# --------------------------------------------------

df = df[df["HV103"] == 1].copy()

print("After de facto:", df.shape)

# --------------------------------------------------
# 2. AGE 18+
# --------------------------------------------------

df = df[df["HV105"] >= 18].copy()

print("After age:", df.shape)

# --------------------------------------------------
# 3. VALID BP MEASUREMENT
# DHS FINAL BP VARIABLES
# --------------------------------------------------

male_valid = (
    (df["HV104"] == 1) &
    (df["MBP24"] < 900) &
    (df["MBP25"] < 900)
)

female_valid = (
    (df["HV104"] == 2) &
    (df["WBP24"] < 900) &
    (df["WBP25"] < 900)
)

df = df[male_valid | female_valid].copy()

print("After BP filter:", df.shape)

# --------------------------------------------------
# 4. CREATE FINAL BP VARIABLES
# --------------------------------------------------

df["sbp_final"] = np.nan
df["dbp_final"] = np.nan
df["bp_meds"] = np.nan

# MEN
male = df["HV104"] == 1

df.loc[male, "sbp_final"] = df.loc[male, "MBP24"]
df.loc[male, "dbp_final"] = df.loc[male, "MBP25"]
df.loc[male, "bp_meds"] = df.loc[male, "MBP19"]

# WOMEN
female = df["HV104"] == 2

df.loc[female, "sbp_final"] = df.loc[female, "WBP24"]
df.loc[female, "dbp_final"] = df.loc[female, "WBP25"]
df.loc[female, "bp_meds"] = df.loc[female, "WBP19"]

# --------------------------------------------------
# 5. CLEAN SPECIAL MISSING
# --------------------------------------------------

df.loc[df["bp_meds"] >= 8, "bp_meds"] = np.nan

# --------------------------------------------------
# 6. HYPERTENSION
# --------------------------------------------------

df["hypertension"] = np.where(
    (
        (df["sbp_final"] >= 140) |
        (df["dbp_final"] >= 90) |
        (df["bp_meds"] == 1)
    ),
    1,
    0
)

# --------------------------------------------------
# 7. SAMPLE WEIGHT
# --------------------------------------------------

df["sample_weight"] = df["HV005"] / 1000000

# --------------------------------------------------
# 8. CHECK SAMPLE
# --------------------------------------------------

print("\nGender counts:")
print(df["HV104"].value_counts())

print("\nHypertension prevalence:")
print(df["hypertension"].value_counts())

print("\nCross-tab:")
print(
    pd.crosstab(
        df["HV104"],
        df["hypertension"],
        margins=True
    )
)

Initial: (132463, 549)
After de facto: (126597, 549)
After age: (81881, 549)
After BP filter: (13848, 549)

Gender counts:
HV104
2.0    7656
1.0    6192
Name: count, dtype: int64

Hypertension prevalence:
hypertension
0    10971
1     2877
Name: count, dtype: int64

Cross-tab:
hypertension      0     1    All
HV104                           
1.0            5132  1060   6192
2.0            5839  1817   7656
All           10971  2877  13848


In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------
df = pd.read_spss("/content/BDPR81FL.SAV", convert_categoricals=False)

# 1. DE FACTO RESIDENTS (HV103 = 1)
df = df[df["HV103"] == 1].copy()

# 2. AGE 18+ (HV105 >= 18)
df = df[df["HV105"] >= 18].copy()

# --------------------------------------------------
# 3. IDENTIFY VALID MEASUREMENTS & MEDICATION
# --------------------------------------------------
# We check for valid BP readings (values < 900 represent valid mmHg)
# We also identify the medication variable (MBP19/WBP19)

# Male logic
male_mask = (df["HV104"] == 1)
# Female logic
female_mask = (df["HV104"] == 2)

# Create clean BP and Meds columns
df["sbp_final"] = np.nan
df["dbp_final"] = np.nan
df["is_on_meds"] = 0  # Default to 0 (No)

# MEN: Map MBP24 (SBP), MBP25 (DBP), and MBP19 (Meds)
df.loc[male_mask, "sbp_final"] = df.loc[male_mask, "MBP24"]
df.loc[male_mask, "dbp_final"] = df.loc[male_mask, "MBP25"]
# Ensure medication codes (1=Yes, 0=No). Treat 8 (Don't know) or 9 (Missing) as No/NaN
df.loc[male_mask & (df["MBP19"] == 1), "is_on_meds"] = 1

# WOMEN: Map WBP24 (SBP), WBP25 (DBP), and WBP19 (Meds)
df.loc[female_mask, "sbp_final"] = df.loc[female_mask, "WBP24"]
df.loc[female_mask, "dbp_final"] = df.loc[female_mask, "WBP25"]
df.loc[female_mask & (df["WBP19"] == 1), "is_on_meds"] = 1

# --------------------------------------------------
# 4. FILTER FOR VALID BP SAMPLES
# --------------------------------------------------
# Exclude respondents who don't have a valid final BP reading
df = df[(df["sbp_final"] < 900) & (df["dbp_final"] < 900)].copy()

# --------------------------------------------------
# 5. DEFINE HYPERTENSION
# --------------------------------------------------
# Definition: SBP >= 140 OR DBP >= 90 OR taking medication
df["hypertension"] = np.where(
    (df["sbp_final"] >= 140) |
    (df["dbp_final"] >= 90) |
    (df["is_on_meds"] == 1),
    1,
    0
)

# --------------------------------------------------
# 6. WEIGHTING AND ANALYSIS
# --------------------------------------------------
df["sample_weight"] = df["HV005"] / 1000000

print(f"Final Sample Size: {len(df)}")
print("\nHypertension Prevalence (Unweighted):")
print(df["hypertension"].value_counts(normalize=True) * 100)

Final Sample Size: 13848

Hypertension Prevalence (Unweighted):
hypertension
0    79.224437
1    20.775563
Name: proportion, dtype: float64


In [ ]:
# ============================================================
# BDHS 2022 HYPERTENSION DATASET
# EXACT DHS TABLE MATCHING VERSION
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD PR FILE
# ============================================================

df = pd.read_spss(
    "/content/BDPR81FL.SAV",
    convert_categoricals=False
)

print("Initial:", df.shape)

# ============================================================
# 2. KEEP DE FACTO RESIDENTS
# ============================================================
# HV103
# 1 = slept in household last night
# ============================================================

df = df[df["HV103"] == 1].copy()

print("After de facto:", df.shape)

# ============================================================
# 3. KEEP ADULTS AGE 18+
# ============================================================
# HV105 = age
# ============================================================

df = df[df["HV105"] >= 18].copy()

print("After age:", df.shape)

# ============================================================
# 4. KEEP ONLY SUCCESSFUL BP MEASUREMENTS
# ============================================================
# MEN:
# MBP5
# 1 = successful BP measurement
#
# WOMEN:
# WBP5
# 1 = successful BP measurement
#
# THIS IS THE DHS OFFICIAL FILTER
# ============================================================

male_valid = (
    (df["HV104"] == 1) &
    (df["MBP5"] == 1)
)

female_valid = (
    (df["HV104"] == 2) &
    (df["WBP5"] == 1)
)

df = df[male_valid | female_valid].copy()

print("After BP eligibility:", df.shape)

# ============================================================
# 5. CREATE FINAL BP VARIABLES
# ============================================================
# MEN:
# MBP24 = final systolic
# MBP25 = final diastolic
#
# WOMEN:
# WBP24 = final systolic
# WBP25 = final diastolic
# ============================================================

df["sbp_final"] = np.nan
df["dbp_final"] = np.nan
df["bp_meds"] = np.nan

# ------------------------------------------------------------
# MEN
# ------------------------------------------------------------

male = df["HV104"] == 1

df.loc[male, "sbp_final"] = df.loc[male, "MBP24"]
df.loc[male, "dbp_final"] = df.loc[male, "MBP25"]
df.loc[male, "bp_meds"] = df.loc[male, "MBP19"]

# ------------------------------------------------------------
# WOMEN
# ------------------------------------------------------------

female = df["HV104"] == 2

df.loc[female, "sbp_final"] = df.loc[female, "WBP24"]
df.loc[female, "dbp_final"] = df.loc[female, "WBP25"]
df.loc[female, "bp_meds"] = df.loc[female, "WBP19"]

# ============================================================
# 6. CLEAN SPECIAL DHS MISSING CODES
# ============================================================
# DHS uses:
# 999 / 9998 / 9999 etc
# ============================================================

df.loc[df["sbp_final"] >= 900, "sbp_final"] = np.nan
df.loc[df["dbp_final"] >= 900, "dbp_final"] = np.nan

# Medication:
# 8 = DK
# 9 = Missing

df.loc[df["bp_meds"] >= 8, "bp_meds"] = np.nan

# ============================================================
# 7. CREATE HYPERTENSION VARIABLE
# ============================================================
# DHS / WHO definition:
#
# HTN = 1 if:
# SBP >= 140
# OR DBP >= 90
# OR taking medication
# ============================================================

df["hypertension"] = np.where(
    (
        (df["sbp_final"] >= 140) |
        (df["dbp_final"] >= 90) |
        (df["bp_meds"] == 1)
    ),
    1,
    0
)

# ============================================================
# 8. CREATE BMI VARIABLE
# ============================================================
# MEN:
# HB40 = BMI x100
#
# WOMEN:
# HA40 = BMI x100
# ============================================================

df["BMI"] = np.nan

# MEN
df.loc[
    male & (df["HB40"] < 9000),
    "BMI"
] = df.loc[
    male & (df["HB40"] < 9000),
    "HB40"
] / 100

# WOMEN
df.loc[
    female & (df["HA40"] < 9000),
    "BMI"
] = df.loc[
    female & (df["HA40"] < 9000),
    "HA40"
] / 100

# ============================================================
# 9. BMI CATEGORIES
# ============================================================

df["BMI_category"] = np.nan

# Underweight
df.loc[df["BMI"] < 18.5, "BMI_category"] = 1

# Normal
df.loc[
    (df["BMI"] >= 18.5) &
    (df["BMI"] < 25),
    "BMI_category"
] = 2

# Overweight
df.loc[
    (df["BMI"] >= 25) &
    (df["BMI"] < 30),
    "BMI_category"
] = 3

# Obese
df.loc[df["BMI"] >= 30, "BMI_category"] = 4

# ============================================================
# 10. CREATE SURVEY WEIGHT
# ============================================================

df["sample_weight"] = df["HV005"] / 1000000

# ============================================================
# 11. KEEP FINAL VARIABLES
# ============================================================

final_cols = [

    # IDENTIFIERS
    "HV001",
    "HV002",
    "HV003",

    # WEIGHT
    "sample_weight",

    # DEMOGRAPHICS
    "HV104",
    "HV105",
    "HV024",
    "HV025",
    "HV106",
    "HV270",

    # BMI
    "BMI",
    "BMI_category",

    # BP VARIABLES
    "sbp_final",
    "dbp_final",
    "bp_meds",

    # OUTCOME
    "hypertension"
]

df_final = df[final_cols].copy()

# ============================================================
# 12. RENAME VARIABLES
# ============================================================

df_final = df_final.rename(columns={

    "HV001": "cluster",
    "HV002": "household",
    "HV003": "line_number",

    "HV104": "gender",
    "HV105": "age",

    "HV024": "division",
    "HV025": "residence",

    "HV106": "education",
    "HV270": "wealth_index"

})

# ============================================================
# 13. OUTPUT CHECKS
# ============================================================

print("\n==============================")
print("FINAL SAMPLE")
print("==============================")

print("\nFinal shape:")
print(df_final.shape)

print("\nGender counts:")
print(df_final["gender"].value_counts())

print("\nHypertension counts:")
print(df_final["hypertension"].value_counts())

print("\nCross-tab:")
print(
    pd.crosstab(
        df_final["gender"],
        df_final["hypertension"],
        margins=True
    )
)

# ============================================================
# 14. WEIGHTED PREVALENCE
# ============================================================

# weighted_prev = (
#     df_final
#     .groupby("gender")
#     .apply(
#         lambda x:
#         np.average(
#             x["hypertension"],
#             weights=x["sample_weight"]
#         ) * 100
#     )
# )

# print("\nWeighted hypertension prevalence:")
# print(weighted_prev)

# # ============================================================
# # 15. SAVE FINAL DATASET
# # ============================================================

# df_final.to_csv(
#     r"E:\BDHS_2022_Hypertension_Final.csv",
#     index=False
# )

# print("\nDataset saved successfully.")

Initial: (132463, 549)
After de facto: (126597, 549)
After age: (81881, 549)
After BP eligibility: (1157, 549)

FINAL SAMPLE

Final shape:
(1157, 16)

Gender counts:
gender
1.0    616
2.0    541
Name: count, dtype: int64

Hypertension counts:
hypertension
0    872
1    285
Name: count, dtype: int64

Cross-tab:
hypertension    0    1   All
gender                      
1.0           508  108   616
2.0           364  177   541
All           872  285  1157
